# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

root = '<DRIVE_ROOT>/MLB데이터'

for dirpath, dirnames, filenames in os.walk(root):
    level = dirpath.replace(root, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(dirpath)
    print(f"{indent}📁 {folder}/")
    for f in sorted(filenames):
        size = os.path.getsize(os.path.join(dirpath, f)) / 1024 / 1024
        print(f"{indent}  [{size:5.1f}MB] {f}")

In [ ]:
# ============================================================
# 종결 모델 v2 재학습 (ump+temp 제거)
# Colab에서 실행
# ============================================================
!pip install lightgbm -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, pickle, json, warnings
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
warnings.filterwarnings('ignore')

DRIVE = '<DRIVE_ROOT>/MLB데이터'
df = pd.read_csv(f'{DRIVE}/master_5seasons.csv')
print(f"마스터 로드: {df.shape}")

# ============================================================
# 1. 피처 정의 (ump_avg_runs, ump_vs_league, temperature 제거)
# ============================================================
WP_FEATURES = [
    'era_diff', 'k9_diff', 'bb9_diff', 'ops_diff',
    'wr5_diff', 'wr10_diff', 'rs10_diff', 'bp_fatigue_1d_diff',
    'run_total_baseline', 'run_margin', 'home_WIN_LOSS', 'park_HR'
]  # 12개

UO_FEATURES = [
    'era_sum', 'whip_sum', 'k9_sum', 'bb9_sum', 'hr9_sum',
    'ops_sum', 'rs5_sum', 'rs10_sum', 'ra5_sum', 'ra10_sum',
    'bp_fatigue_sum', 'park_factor', 'park_R', 'park_HR', 'park_H',
    'era_max', 'whip_max', 'run_margin', 'home_WIN_LOSS',
    'run_total_baseline', 'era_vs_run_total', 'ops_vs_run_total'
]  # 22개

print(f"승패 피처: {len(WP_FEATURES)}개")
print(f"총득점 피처: {len(UO_FEATURES)}개")

# ============================================================
# 2. 프로덕션 모델 학습 (전체 5시즌)
# ============================================================
print("\n===== 프로덕션 모델 학습 (2021-2025 전체) =====")

# --- 승패 ---
wp_data = df.dropna(subset=WP_FEATURES + ['home_win'])
X_wp, y_wp = wp_data[WP_FEATURES], wp_data['home_win']

wp_lr = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler()),
    ('m', LogisticRegression(C=0.01, max_iter=1000, solver='lbfgs'))
])
wp_lr.fit(X_wp, y_wp)

wp_imp_for_lgb = SimpleImputer(strategy='median').fit(X_wp)
wp_lgb = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.03, max_depth=2,
    num_leaves=4, min_child_samples=150,
    reg_alpha=3.0, reg_lambda=3.0, verbose=-1
)
wp_lgb.fit(wp_imp_for_lgb.transform(X_wp), y_wp)
print(f"승패 학습 완료: {len(X_wp):,}행")

# --- 총득점 ---
uo_data = df.dropna(subset=UO_FEATURES + ['upper'])
X_uo, y_uo = uo_data[UO_FEATURES], uo_data['upper']

uo_lr = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('sc', StandardScaler()),
    ('m', LogisticRegression(C=0.01, max_iter=1000, solver='lbfgs'))
])
uo_lr.fit(X_uo, y_uo)

uo_imp_for_lgb = SimpleImputer(strategy='median').fit(X_uo)
uo_lgb = lgb.LGBMClassifier(
    n_estimators=100, learning_rate=0.03, max_depth=2,
    num_leaves=3, min_child_samples=100,
    reg_alpha=3.0, reg_lambda=3.0, verbose=-1
)
uo_lgb.fit(uo_imp_for_lgb.transform(X_uo), y_uo)
print(f"총득점 학습 완료: {len(X_uo):,}행")

# ============================================================
# 3. PKL 저장
# ============================================================
files = {
    '종결v2_승패_LR.pkl': wp_lr,
    '종결v2_승패_LGB.pkl': wp_lgb,
    '종결v2_총득점_LR.pkl': uo_lr,
    '종결v2_총득점_LGB.pkl': uo_lgb,
}
for fname, model in files.items():
    path = f'{DRIVE}/{fname}'
    with open(path, 'wb') as f:
        pickle.dump(model, f)
    print(f"저장: {fname}")

# ============================================================
# 4. Walk-Forward OOS 검증
# ============================================================
print("\n===== Walk-Forward OOS 검증 =====")
seasons = [2021, 2022, 2023, 2024, 2025]

def walk_forward(features, target, lr_w, lgb_w, thresh, lgb_leaves, lgb_min):
    all_pred, all_actual, season_res = [], [], {}
    for i, test_s in enumerate(seasons):
        if i == 0:
            season_res[test_s] = 'skip'
            continue
        train = df[df['season'].isin(seasons[:i])].dropna(subset=features + [target])
        test = df[df['season'] == test_s].dropna(subset=features + [target])

        lr_p = Pipeline([('imp', SimpleImputer(strategy='median')),
                         ('sc', StandardScaler()),
                         ('m', LogisticRegression(C=0.01, max_iter=1000))])
        lr_p.fit(train[features], train[target])
        p_lr = lr_p.predict_proba(test[features])[:, 1]

        imp = SimpleImputer(strategy='median').fit(train[features])
        lg = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.03, max_depth=2,
                                num_leaves=lgb_leaves, min_child_samples=lgb_min,
                                reg_alpha=3, reg_lambda=3, verbose=-1)
        lg.fit(imp.transform(train[features]), train[target])
        p_lgb = lg.predict_proba(imp.transform(test[features]))[:, 1]

        p = lr_w * p_lr + lgb_w * p_lgb
        conf = (p >= thresh) | (p <= (1 - thresh))
        pred = (p[conf] >= 0.5).astype(int)
        actual = test[target].values[conf]
        acc = (pred == actual).mean()
        season_res[test_s] = f"{acc:.1%} ({len(pred)}g/{len(test)}g)"
        all_pred.extend(pred); all_actual.extend(actual)

    total_acc = np.mean(np.array(all_pred) == np.array(all_actual))
    return total_acc, len(all_pred), season_res

# 승패
print("\n--- 승패 (12f, LR80+LGB20, t=0.64) ---")
acc, n, sr = walk_forward(WP_FEATURES, 'home_win', 0.8, 0.2, 0.64, 4, 150)
for s, r in sr.items(): print(f"  {s}: {r}")
print(f"  전체: {acc:.1%} ({n:,}경기, 일{n/(183*4):.1f})")

# 총득점
print("\n--- 총득점 (22f, LR50+LGB50, t=0.60) ---")
acc2, n2, sr2 = walk_forward(UO_FEATURES, 'upper', 0.5, 0.5, 0.60, 3, 100)
for s, r in sr2.items(): print(f"  {s}: {r}")
print(f"  전체: {acc2:.1%} ({n2:,}경기, 일{n2/(183*4):.1f})")

# ============================================================
# 5. 모델설정 JSON 저장
# ============================================================
config = {
    'version': '2.0_final',
    'created': '2026-04-02',
    'change_from_v1': 'ump_avg_runs, ump_vs_league, temperature 제거 (NaN 노이즈 해소)',
    'win_loss': {
        'features': WP_FEATURES, 'n_features': len(WP_FEATURES),
        'ensemble': 'LR 80% + LGB 20%', 'threshold': 0.64,
        'lr_C': 0.01, 'lgb': {'n_estimators':100,'lr':0.03,'depth':2,'leaves':4,'min_child':150,'reg':3.0},
        'oos_accuracy': round(float(acc), 4), 'oos_games': n
    },
    'run_total_direction': {
        'features': UO_FEATURES, 'n_features': len(UO_FEATURES),
        'ensemble': 'LR 50% + LGB 50%', 'threshold': 0.60,
        'lr_C': 0.01, 'lgb': {'n_estimators':100,'lr':0.03,'depth':2,'leaves':3,'min_child':100,'reg':3.0},
        'oos_accuracy': round(float(acc2), 4), 'oos_games': n2
    },
    'files': ['종결v2_승패_LR.pkl','종결v2_승패_LGB.pkl','종결v2_총득점_LR.pkl','종결v2_총득점_LGB.pkl']
}
with open(f'{DRIVE}/종결v2_모델설정.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\n===== 완료 =====")
print(f"저장 위치: {DRIVE}/")
print("  종결v2_승패_LR.pkl")
print("  종결v2_승패_LGB.pkl")
print("  종결v2_총득점_LR.pkl")
print("  종결v2_총득점_LGB.pkl")
print("  종결v2_모델설정.json")

In [ ]:
import pandas as pd, os

DRIVE = '<DRIVE_ROOT>/MLB데이터'
date = '20260329'  # 가장 최근 완전한 날짜

files = [
    f'2026/mlb_{date}_경기목록.csv',
    f'2026/mlb_{date}_선발투수.csv',
    f'2026/mlb_{date}_타자라인업.csv',
    f'2026/mlb_{date}_총득점기준점.csv',
    f'2026/mlb_{date}_불펜소모도.csv',
    f'2026/mlb_{date}_선발시범성적.csv',
    f'2026/mlb_{date}_심판.csv',
    f'2026/mlb_{date}_날씨.csv',
    f'2026/mlb_{date}_IL현황.csv',
    f'MARKET_PRICE,LINEUP/mlb_market_price_2026.csv',
    f'MARKET_PRICE,LINEUP/mlb_starters_2026.csv',
    f'MARKET_PRICE,LINEUP/mlb_lineup_2026.csv',
    f'park_factors_2025.csv',
    f'2026/today_2026-04-01.csv',
    f'2026/predictions_2026-04-01.csv',
]

for f in files:
    path = f'{DRIVE}/{f}'
    if os.path.exists(path):
        d = pd.read_csv(path)
        print(f"\n{'='*60}")
        print(f"📄 {f}  ({d.shape[0]}행 × {d.shape[1]}열)")
        print(f"컬럼: {list(d.columns)}")
        print(d.head(2).to_string())
    else:
        print(f"\n❌ {f} — 파일 없음")

In [ ]:
# ============================================================
# MLB 예측 파이프라인 v2 — daily_predictor
# 입력: today_{date}.csv (수집기가 만든 파일)
# 출력: predictions_{date}.csv + 분석 후보
# ============================================================
!pip install lightgbm -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd, numpy as np, pickle, warnings, os
warnings.filterwarnings('ignore')

DRIVE = '<DRIVE_ROOT>/MLB데이터'
TARGET_DATE = '2026-04-01'  # ← 예측할 날짜

# ============================================================
# 1. 모델 & 설정 로드
# ============================================================
with open(f'{DRIVE}/종결v2_승패_LR.pkl', 'rb') as f: wp_lr = pickle.load(f)
with open(f'{DRIVE}/종결v2_승패_LGB.pkl', 'rb') as f: wp_lgb = pickle.load(f)
with open(f'{DRIVE}/종결v2_총득점_LR.pkl', 'rb') as f: uo_lr = pickle.load(f)
with open(f'{DRIVE}/종결v2_총득점_LGB.pkl', 'rb') as f: uo_lgb = pickle.load(f)

WP_FEATURES = list(wp_lr.feature_names_in_)  # 12개
UO_FEATURES = list(uo_lr.feature_names_in_)  # 22개
print(f"모델 로드 완료 — 승패 {len(WP_FEATURES)}f, 총득점 {len(UO_FEATURES)}f")

# ============================================================
# 2. 게임 히스토리 구축 (롤링 피처용)
# ============================================================
# 2a. 2021-2025 과거 데이터
master = pd.read_csv(f'{DRIVE}/master_5seasons.csv')
hist = master[['gamePk','date','home_team','away_team','홈팀득점','원정팀득점','홈팀결과']].copy()
hist['date'] = pd.to_datetime(hist['date'])

# 2b. 2026 시즌 결과 추가
odds26 = pd.read_csv(f'{DRIVE}/MARKET_PRICE,LINEUP/mlb_market_price_2026.csv')
odds26 = odds26[odds26['away_score'].notna()].copy()  # 결과 있는 경기만
g26 = odds26.rename(columns={
    'game_id':'gamePk','home_team':'home_team','away_team':'away_team',
    'home_score':'홈팀득점','away_score':'원정팀득점'
})
g26['date'] = pd.to_datetime(g26['date'])
g26['홈팀결과'] = g26.apply(lambda r: '승' if r['홈팀득점'] > r['원정팀득점'] else '패', axis=1)
hist = pd.concat([hist, g26[hist.columns]], ignore_index=True)
hist = hist.sort_values('date').reset_index(drop=True)
print(f"히스토리: {len(hist):,}경기 ({hist['date'].min().date()} ~ {hist['date'].max().date()})")

# ============================================================
# 3. 롤링 피처 계산 함수
# ============================================================
def calc_team_rolling(team, before_date, hist_df):
    """특정 팀의 before_date 이전 최근 경기 기반 롤링 스탯"""
    # 홈 경기
    h = hist_df[(hist_df['home_team'] == team) & (hist_df['date'] < before_date)].copy()
    h['win'] = (h['홈팀결과'] == '승').astype(int)
    h['rs'] = h['홈팀득점']
    h['ra'] = h['원정팀득점']
    # 원정 경기
    a = hist_df[(hist_df['away_team'] == team) & (hist_df['date'] < before_date)].copy()
    a['win'] = (a['홈팀결과'] == '패').astype(int)
    a['rs'] = a['원정팀득점']
    a['ra'] = a['홈팀득점']

    games = pd.concat([
        h[['date','win','rs','ra']],
        a[['date','win','rs','ra']]
    ]).sort_values('date')

    if len(games) < 5:
        return {f'{p}': np.nan for p in ['wr5','wr10','rs5','rs10','ra5','ra10']}

    last5 = games.tail(5)
    last10 = games.tail(10)
    return {
        'wr5': last5['win'].mean(),
        'wr10': last10['win'].mean() if len(last10) >= 10 else last5['win'].mean(),
        'rs5': last5['rs'].mean(),
        'rs10': last10['rs'].mean() if len(last10) >= 10 else last5['rs'].mean(),
        'ra5': last5['ra'].mean(),
        'ra10': last10['ra'].mean() if len(last10) >= 10 else last5['ra'].mean(),
    }

# ============================================================
# 4. 불펜 소모도 계산
# ============================================================
def calc_bp_fatigue(team, date_str):
    """불펜소모도 CSV에서 팀별 1d/3d fatigue 합산"""
    bp_file = f'{DRIVE}/2026/mlb_{date_str.replace("-","")}_불펜소모도.csv'
    if not os.path.exists(bp_file):
        return 0.0, 0.0

    bp = pd.read_csv(bp_file)
    team_bp = bp[bp['팀명'] == team]
    if len(team_bp) == 0:
        # 팀명 부분 매치 시도
        team_bp = bp[bp['팀명'].str.contains(team.split()[-1], case=False, na=False)]

    if len(team_bp) == 0:
        return 0.0, 0.0

    # 3일등판수 합 = bp_fatigue_3d, 가장 최근 1일은 근사치로 처리
    fatigue_3d = team_bp['3일등판수'].sum()
    fatigue_1d = team_bp[team_bp['3일등판수'] >= 1]['3일등판수'].count()  # 최근 등판한 투수 수
    return float(fatigue_1d), float(fatigue_3d)

# ============================================================
# 5. today 파일 로드 + 피처 빌드
# ============================================================
today_file = f'{DRIVE}/2026/today_{TARGET_DATE}.csv'
today = pd.read_csv(today_file)
print(f"\n오늘 경기: {len(today)}경기 ({TARGET_DATE})")

before_date = pd.to_datetime(TARGET_DATE)

rows = []
for _, g in today.iterrows():
    r = g.to_dict()

    # 롤링 피처
    home_roll = calc_team_rolling(r['home_team'], before_date, hist)
    away_roll = calc_team_rolling(r['away_team'], before_date, hist)
    for k, v in home_roll.items():
        r[f'home_{k}'] = v
    for k, v in away_roll.items():
        r[f'away_{k}'] = v

    # 불펜 소모도
    h_bp1d, h_bp3d = calc_bp_fatigue(r['home_team'], TARGET_DATE)
    a_bp1d, a_bp3d = calc_bp_fatigue(r['away_team'], TARGET_DATE)
    r['home_bp_fatigue_1d'] = h_bp1d
    r['home_bp_fatigue_3d'] = h_bp3d
    r['away_bp_fatigue_1d'] = a_bp1d
    r['away_bp_fatigue_3d'] = a_bp3d

    # ---- diff 피처 (승패용) ----
    r['era_diff'] = r['home_sp_era'] - r['away_sp_era']
    r['k9_diff'] = r['home_sp_k9'] - r['away_sp_k9']
    r['bb9_diff'] = r['home_sp_bb9'] - r['away_sp_bb9']
    r['ops_diff'] = r['home_lineup_ops'] - r['away_lineup_ops']
    r['wr5_diff'] = r.get('home_wr5', np.nan) - r.get('away_wr5', np.nan) if not np.isnan(r.get('home_wr5', np.nan)) else np.nan
    r['wr10_diff'] = r.get('home_wr10', np.nan) - r.get('away_wr10', np.nan) if not np.isnan(r.get('home_wr10', np.nan)) else np.nan
    r['rs10_diff'] = r.get('home_rs10', np.nan) - r.get('away_rs10', np.nan) if not np.isnan(r.get('home_rs10', np.nan)) else np.nan
    r['bp_fatigue_1d_diff'] = h_bp1d - a_bp1d

    # ---- sum 피처 (총득점용) ----
    r['era_sum'] = r['home_sp_era'] + r['away_sp_era']
    r['whip_sum'] = r['home_sp_whip'] + r['away_sp_whip']
    r['k9_sum'] = r['home_sp_k9'] + r['away_sp_k9']
    r['bb9_sum'] = r['home_sp_bb9'] + r['away_sp_bb9']
    r['hr9_sum'] = r['home_sp_hr9'] + r['away_sp_hr9']
    r['ops_sum'] = r['home_lineup_ops'] + r['away_lineup_ops']
    r['rs5_sum'] = r.get('home_rs5', 0) + r.get('away_rs5', 0)
    r['rs10_sum'] = r.get('home_rs10', 0) + r.get('away_rs10', 0)
    r['ra5_sum'] = r.get('home_ra5', 0) + r.get('away_ra5', 0)
    r['ra10_sum'] = r.get('home_ra10', 0) + r.get('away_ra10', 0)
    r['bp_fatigue_sum'] = h_bp3d + a_bp3d
    r['era_max'] = max(r['home_sp_era'], r['away_sp_era'])
    r['whip_max'] = max(r['home_sp_whip'], r['away_sp_whip'])
    r['era_vs_run_total'] = r['era_sum'] - r['run_total_baseline']
    r['ops_vs_run_total'] = r['ops_sum'] * 10 - r['run_total_baseline']

    rows.append(r)

pred_df = pd.DataFrame(rows)

# ============================================================
# 6. 모델 적용
# ============================================================
# 승패
X_wp = pred_df[WP_FEATURES]
wp_p_lr = wp_lr.predict_proba(X_wp)[:, 1]
wp_p_lgb = wp_lgb.predict_proba(X_wp)[:, 1]
pred_df['win_prob'] = 0.8 * wp_p_lr + 0.2 * wp_p_lgb

# 총득점
X_uo = pred_df[UO_FEATURES]
uo_p_lr = uo_lr.predict_proba(X_uo)[:, 1]
uo_p_lgb = uo_lgb.predict_proba(X_uo)[:, 1]
pred_df['run_total_prob'] = 0.5 * uo_p_lr + 0.5 * uo_p_lgb

# 확신도 = max(prob, 1-prob)
pred_df['win_conf'] = pred_df['win_prob'].apply(lambda p: max(p, 1-p))
pred_df['run_total_conf'] = pred_df['run_total_prob'].apply(lambda p: max(p, 1-p))

# ============================================================
# 7. 분석 후보 출력
# ============================================================
WIN_THRESH = 0.64
RUN_TOTAL_THRESH = 0.60

print(f"\n{'='*70}")
print(f"  🎯 MLB 예측 v2 — {TARGET_DATE}")
print(f"{'='*70}")

print(f"\n📊 승패 추천 (임계 {WIN_THRESH}):")
wp_predictions = pred_df[pred_df['win_conf'] >= WIN_THRESH].copy()
if len(wp_predictions) == 0:
    print("  확신 경기 없음")
else:
    for _, g in wp_predictions.iterrows():
        side = '홈' if g['win_prob'] >= 0.5 else '원정'
        team = g['home_team'] if side == '홈' else g['away_team']
        win_loss = g['home_WIN_LOSS'] if side == '홈' else g['away_WIN_LOSS']
        print(f"  {side} {team:25s} ({win_loss:+.0f}) | 확률 {g['win_prob']:.1%} | 확신 {g['win_conf']:.1%}")

print(f"\n📊 총득점 추천 (임계 {RUN_TOTAL_THRESH}):")
run_total_predictions = pred_df[pred_df['run_total_conf'] >= RUN_TOTAL_THRESH].copy()
if len(run_total_predictions) == 0:
    print("  확신 경기 없음")
else:
    for _, g in run_total_predictions.iterrows():
        side = 'UPPER' if g['run_total_prob'] >= 0.5 else 'LOWER'
        print(f"  {side:5s} {g['home_team']:25s} vs {g['away_team']:20s} | RUN_TOTAL {g['run_total_baseline']:.1f} | 확률 {g['run_total_prob']:.1%} | 확신 {g['run_total_conf']:.1%}")

# ============================================================
# 8. 저장
# ============================================================
out_path = f'{DRIVE}/2026/predictions_v2_{TARGET_DATE}.csv'
pred_df.to_csv(out_path, index=False)
print(f"\n💾 저장: {out_path}")
print(f"   승패 추천: {len(wp_predictions)}경기 | 총득점 추천: {len(run_total_predictions)}경기")

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# Houston 2023 — 홈경기 순서대로 추적
team = 'Houston Astros'
year = 2023

# 모든 Houston 경기 (홈+원정) 날짜순
all_games = ref[((ref['홈팀'] == team) | (ref['원정팀'] == team)) & (ref['시즌'] == year)].sort_values('날짜')

# 홈경기만
home_games = ref[(ref['홈팀'] == team) & (ref['시즌'] == year)].sort_values('날짜')

print(f"=== {team} {year} 전체 경기 순서 ===")
print(f"{'날짜':>12s} {'홈/원':>5s} {'상대팀':>25s} {'득점':>4s} {'실점':>4s} {'결과':>4s} {'home_wr5':>10s}")

for _, row in all_games.head(15).iterrows():
    if row['홈팀'] == team:
        side = '홈'
        opp = row['원정팀']
        rs = row['홈팀득점']
        ra = row['원정팀득점']
        result = row['홈팀결과']
        wr5_val = row.get('home_wr5', np.nan)
    else:
        side = '원정'
        opp = row['홈팀']
        rs = row['원정팀득점']
        ra = row['홈팀득점']
        result = row['원정팀결과']
        wr5_val = row.get('away_wr5', np.nan)

    wr5_str = f"{wr5_val:.3f}" if pd.notna(wr5_val) else "NaN"
    print(f"{str(row['날짜']):>12s} {side:>5s} {opp:>25s} {rs:>4.0f} {ra:>4.0f} {result:>4s} {wr5_str:>10s}")

# 수동 검증: 전체경기 기준 vs 홈/원정 분리 기준
print(f"\n=== 수동 검증: 10번째 경기의 wr5 ===")
games_list = []
for _, row in all_games.iterrows():
    if row['홈팀'] == team:
        win = 1 if row['홈팀결과'] == '승' else 0
    else:
        win = 1 if row['원정팀결과'] == '승' else 0
    games_list.append(win)

# 10번째 경기 기준 (인덱스 9)
if len(games_list) >= 10:
    last5_all = games_list[4:9]  # 전체경기 기준 5~9번째
    print(f"  전체경기 기준 last5: {last5_all} → wr5={np.mean(last5_all):.3f}")

# 홈경기만으로도 계산
home_wins = []
for _, row in home_games.iterrows():
    home_wins.append(1 if row['홈팀결과'] == '승' else 0)

if len(home_wins) >= 8:
    last5_home = home_wins[3:8]
    print(f"  홈경기만 기준 last5: {last5_home} → wr5={np.mean(last5_home):.3f}")

    # 마스터테이블 실제값
    eighth_home = home_games.iloc[7]
    print(f"  마스터테이블 실제값: home_wr5={eighth_home['home_wr5']}")

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

team = 'Houston Astros'
year = 2023
all_games = ref[((ref['홈팀'] == team) | (ref['원정팀'] == team)) & (ref['시즌'] == year)].sort_values('날짜')

# 4/14 홈경기 (14번째 경기) — 이전 5경기(9~13)의 득점/실점
print("=== 4/14 이전 5경기 (전체경기 기준) ===")
prev5 = all_games.head(13).tail(5)  # 9~13번째
for _, row in prev5.iterrows():
    if row['홈팀'] == team:
        rs, ra = row['홈팀득점'], row['원정팀득점']
    else:
        rs, ra = row['원정팀득점'], row['홈팀득점']
    print(f"  {str(row['날짜']):>12s} 득점={rs:.0f} 실점={ra:.0f}")

rs_list = []
ra_list = []
for _, row in prev5.iterrows():
    if row['홈팀'] == team:
        rs_list.append(row['홈팀득점'])
        ra_list.append(row['원정팀득점'])
    else:
        rs_list.append(row['원정팀득점'])
        ra_list.append(row['홈팀득점'])

print(f"\n전체경기 기준: rs5={np.mean(rs_list):.3f}, ra5={np.mean(ra_list):.3f}")

# 마스터테이블 실제값
game_414 = ref[(ref['홈팀'] == team) & (ref['날짜'].astype(str).str.contains('2023-04-14'))].iloc[0]
print(f"마스터테이블:   rs5={game_414['home_rs5']:.3f}, ra5={game_414['home_ra5']:.3f}")

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 2023 경기결과만 추출 (rolling 계산 입력용)
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀',
    '홈팀득점','원정팀득점','홈팀결과','원정팀결과']].copy()
results_2023 = results_2023.sort_values('날짜').reset_index(drop=True)
print(f"2023 경기수: {len(results_2023)}")

# === 함수 ===
def calc_recent_form(results_df, target_date):
    prev = results_df[results_df['날짜'] < target_date].copy()
    prev = prev.sort_values('날짜')

    form = {}
    teams = set(prev['홈팀'].unique()) | set(prev['원정팀'].unique())

    for team in teams:
        # 홈+원정 모든 경기
        home_mask = prev['홈팀'] == team
        away_mask = prev['원정팀'] == team

        team_games = []
        for _, row in prev[home_mask | away_mask].iterrows():
            if row['홈팀'] == team:
                win = 1 if row['홈팀결과'] == '승' else 0
                rs = row['홈팀득점']
                ra = row['원정팀득점']
            else:
                win = 1 if row['원정팀결과'] == '승' else 0
                rs = row['원정팀득점']
                ra = row['홈팀득점']
            team_games.append((win, rs, ra))

        form[team] = {}
        for window, suffix in [(5, '5'), (10, '10')]:
            recent = team_games[-window:]
            if len(recent) >= 3:
                form[team][f'wr{suffix}'] = np.mean([g[0] for g in recent])
                form[team][f'rs{suffix}'] = np.mean([g[1] for g in recent])
                form[team][f'ra{suffix}'] = np.mean([g[2] for g in recent])
            else:
                form[team][f'wr{suffix}'] = np.nan
                form[team][f'rs{suffix}'] = np.nan
                form[team][f'ra{suffix}'] = np.nan

    return form

# === 검증 1: Houston 4/14 ===
form = calc_recent_form(results_2023, '2023-04-14')
hou = form.get('Houston Astros', {})
game_414 = ref[(ref['홈팀'] == 'Houston Astros') & (ref['날짜'].astype(str).str.contains('2023-04-14'))].iloc[0]

print(f"\n=== Houston 4/14 검증 ===")
print(f"{'':>15s} {'함수':>10s} {'마스터':>10s} {'일치':>5s}")
for key, master_col in [('wr5','home_wr5'), ('rs5','home_rs5'), ('ra5','home_ra5'),
                         ('wr10','home_wr10'), ('rs10','home_rs10'), ('ra10','home_ra10')]:
    calc_val = hou.get(key, np.nan)
    ref_val = game_414[master_col]
    match = '✅' if (pd.isna(calc_val) and pd.isna(ref_val)) or (abs(calc_val - ref_val) < 0.001) else '❌'
    calc_str = f"{calc_val:.3f}" if pd.notna(calc_val) else "NaN"
    ref_str = f"{ref_val:.3f}" if pd.notna(ref_val) else "NaN"
    print(f"  {key:>15s} {calc_str:>10s} {ref_str:>10s} {match:>5s}")

# === 검증 2: 원정팀도 확인 (Texas Rangers away) ===
tex = form.get('Texas Rangers', {})
print(f"\n=== Texas Rangers (4/14 원정) 검증 ===")
for key, master_col in [('wr5','away_wr5'), ('rs5','away_rs5'), ('ra5','away_ra5')]:
    calc_val = tex.get(key, np.nan)
    ref_val = game_414[master_col]
    match = '✅' if (pd.isna(calc_val) and pd.isna(ref_val)) or (abs(calc_val - ref_val) < 0.001) else '❌'
    calc_str = f"{calc_val:.3f}" if pd.notna(calc_val) else "NaN"
    ref_str = f"{ref_val:.3f}" if pd.notna(ref_val) else "NaN"
    print(f"  {key:>15s} {calc_str:>10s} {ref_str:>10s} {match:>5s}")

# === 검증 3: 시즌 초반 NaN 케이스 (3/30 첫 경기) ===
form_first = calc_recent_form(results_2023, '2023-03-30')
hou_first = form_first.get('Houston Astros', {})
print(f"\n=== Houston 3/30 (시즌 첫 경기) ===")
print(f"  wr5: {hou_first.get('wr5', 'key없음')} (NaN이어야 함)")

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀',
    '홈팀득점','원정팀득점','홈팀결과','원정팀결과']].sort_values('날짜').reset_index(drop=True)

# (함수 동일 — 위에서 이미 정의됨)

# 7월 중순 랜덤 검증 — 50경기+ 쌓인 상태
test_dates = ['2023-07-15', '2023-08-20', '2023-09-10']
test_teams = ['New York Yankees', 'Los Angeles Dodgers', 'Atlanta Braves']

errors = 0
checks = 0
for date in test_dates:
    form = calc_recent_form(results_2023, date)
    # 해당 날짜 경기에서 홈팀 검증
    day_games = ref[(ref['시즌'] == 2023) & (ref['날짜'].astype(str).str.contains(date))]
    for _, game in day_games.head(3).iterrows():
        home = game['홈팀']
        away = game['원정팀']

        for team, prefix in [(home, 'home'), (away, 'away')]:
            team_form = form.get(team, {})
            for key in ['wr5', 'rs5', 'ra5', 'wr10', 'rs10', 'ra10']:
                calc_val = team_form.get(key, np.nan)
                ref_val = game[f'{prefix}_{key}']
                checks += 1
                if pd.isna(calc_val) and pd.isna(ref_val):
                    continue
                if pd.isna(calc_val) or pd.isna(ref_val) or abs(calc_val - ref_val) >= 0.001:
                    errors += 1
                    print(f"  ❌ {date} {team} {key}: 함수={calc_val}, 마스터={ref_val}")

print(f"\n총 {checks}건 검증, 오류 {errors}건")
if errors == 0:
    print("✅ 전부 일치 — 함수 검증 완료")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터/2026'
pit = pd.read_csv(f'{MLB_PATH}/mlb_2026_정규시즌_투수개인기록.csv')

# 한 경기에서 투수 목록 확인 — 선발/불펜 구분 가능한지
gpk = pit['gamePk'].iloc[0]
game = pit[pit['gamePk'] == gpk]
print(f"=== gamePk={gpk} ===")
for side in ['home', 'away']:
    sp = game[game['홈원정'] == side].reset_index(drop=True)
    print(f"\n  [{side}] {sp['팀'].iloc[0] if len(sp) > 0 else '?'}")
    print(sp[['선수명','이닝','투구수','삼진','자책','승','패']].to_string(index=True))

# 선발 판별: 가장 먼저 등판 + 이닝 가장 많은 투수?
# 아니면 첫 번째 행이 항상 선발?
print(f"\n=== 3경기 첫 번째 투수 확인 ===")
for gpk in pit['gamePk'].unique()[:3]:
    game = pit[pit['gamePk'] == gpk]
    for side in ['home', 'away']:
        first = game[game['홈원정'] == side].iloc[0]
        print(f"  {gpk} {side}: {first['선수명']} (이닝={first['이닝']}, 투구수={first['투구수']})")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터/2026'

# 경기목록에 선발투수 정보 있는지
games = pd.read_csv(f'{MLB_PATH}/mlb_2026_정규시즌_경기결과.csv')
print(f"경기결과 컬럼: {list(games.columns)}")

# 선발투수 CSV 있는지
import os
for f in sorted(os.listdir(MLB_PATH)):
    if '선발' in f and '시범' not in f:
        print(f"\n📌 {f}")
        df = pd.read_csv(f'{MLB_PATH}/{f}')
        print(f"  컬럼: {list(df.columns)}")
        print(df.head(2).to_string())

# 일별 선발투수 CSV도 확인
for f in sorted(os.listdir(MLB_PATH)):
    if '선발투수' in f and '20260325' in f:
        print(f"\n📌 {f}")
        df = pd.read_csv(f'{MLB_PATH}/{f}')
        print(f"  컬럼: {list(df.columns)}")
        print(df.head(3).to_string())

In [ ]:
import pandas as pd
import os

MLB_PATH = '<DRIVE_ROOT>/MLB데이터/2026'

# 방법 1: 일별 선발투수 CSV 합치기
daily_sp = []
for f in sorted(os.listdir(MLB_PATH)):
    if '선발투수' in f and f.endswith('.csv') and '시범' not in f:
        df = pd.read_csv(f'{MLB_PATH}/{f}')
        daily_sp.append(df)

if daily_sp:
    sp_all = pd.concat(daily_sp, ignore_index=True)
    print(f"일별 선발투수 합산: {len(sp_all)}행")
    print(f"컬럼: {list(sp_all.columns)}")
    print(f"gamePk 수: {sp_all['gamePk'].nunique()}")
    print(sp_all.head(3).to_string())

# 방법 2: 투수기록에서 이닝 최대 = 선발 (정확도 확인용)
pit = pd.read_csv(f'{MLB_PATH}/mlb_2026_정규시즌_투수개인기록.csv')
sp_by_inning = pit.loc[pit.groupby(['gamePk','홈원정'])['이닝'].idxmax()][['gamePk','홈원정','선수명','이닝']]
print(f"\n이닝최대 기준 선발: {len(sp_by_inning)}행")

# 두 방법 비교 (일별 CSV 있는 경기만)
if daily_sp:
    # gamePk + 홈원정으로 매칭
    merged = sp_by_inning.merge(
        sp_all[['gamePk','홈원정','선수명']].rename(columns={'선수명':'daily_sp'}),
        on=['gamePk','홈원정'], how='inner'
    )
    match = (merged['선수명'] == merged['daily_sp']).sum()
    print(f"\n두 방법 비교: {match}/{len(merged)} 일치 ({match/len(merged)*100:.1f}%)")

    mismatch = merged[merged['선수명'] != merged['daily_sp']]
    if len(mismatch) > 0:
        print(f"\n불일치 케이스:")
        print(mismatch.head(5).to_string())

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 데이터 로드
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀']].sort_values('날짜').reset_index(drop=True)

print(f"경기결과: {len(results_2023)}, 투수기록: {len(pit)}, starters: {len(st)}")
print(f"투수기록 컬럼: {list(pit.columns)[:8]}")
print(f"starters 컬럼: {list(st.columns)[:6]}")

# 투수기록에 날짜 붙이기 (gamePk 기준)
game_dates = results_2023[['gamePk','날짜']].drop_duplicates()
pit = pit.merge(game_dates, on='gamePk', how='left')
print(f"투수기록 날짜 매칭: {pit['날짜'].notna().sum()}/{len(pit)}")

# === 함수 ===
def calc_bp_fatigue(results_df, pitchers_df, starters_df, target_date):
    """
    target_date 이전 경기의 불펜 이닝으로 1d/3d 피로도 계산
    리턴: {팀명: {bp_fatigue_1d, bp_fatigue_3d}}
    """
    prev_games = results_df[results_df['날짜'] < target_date]

    # 선발투수 이름 딕셔너리: (gamePk, side) → 이름
    sp_names = {}
    for _, r in starters_df.iterrows():
        sp_names[(r['gamePk'], 'home')] = r.get('home_sp_name', '')
        sp_names[(r['gamePk'], 'away')] = r.get('away_sp_name', '')

    # 경기별 팀별 불펜 이닝 계산
    # {(gamePk, 팀명): bp_innings}
    bp_innings = {}
    for gpk in prev_games['gamePk'].unique():
        game_pit = pitchers_df[pitchers_df['gamePk'] == gpk]
        for side in ['home', 'away']:
            team_pit = game_pit[game_pit['홈원정'] == side]
            if len(team_pit) == 0:
                continue
            sp_name = sp_names.get((gpk, side), '')
            relief = team_pit[team_pit['선수명'] != sp_name]
            bp_ip = relief['이닝'].sum()
            team_name = team_pit['팀'].iloc[0]
            bp_innings[(gpk, team_name)] = bp_ip

    # 팀별 날짜+불펜이닝 히스토리 구축 (홈+원정 모든 경기)
    team_history = {}  # team → [(date_str, bp_ip), ...]
    for _, row in prev_games.sort_values('날짜').iterrows():
        gpk = row['gamePk']
        date_str = str(row['날짜'])[:10]
        for team_col in ['홈팀', '원정팀']:
            team = row[team_col]
            bp_ip = bp_innings.get((gpk, team), 0.0)
            team_history.setdefault(team, []).append((date_str, bp_ip))

    # target_date 기준 1d/3d 계산
    target_dt = pd.to_datetime(target_date)
    d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')

    fatigue = {}
    for team, history in team_history.items():
        sum_1d = sum(ip for d, ip in history if d >= d1 and d < str(target_date)[:10])
        sum_3d = sum(ip for d, ip in history if d >= d3 and d < str(target_date)[:10])
        fatigue[team] = {'bp_fatigue_1d': sum_1d, 'bp_fatigue_3d': sum_3d}

    return fatigue

# === 검증: Houston 4/14 ===
fat = calc_bp_fatigue(results_2023, pit, st, '2023-04-14')
hou = fat.get('Houston Astros', {})
game_414 = ref[(ref['홈팀'] == 'Houston Astros') & (ref['날짜'].astype(str).str.contains('2023-04-14'))].iloc[0]

print(f"\n=== Houston 4/14 (홈) 검증 ===")
print(f"{'':>20s} {'함수':>10s} {'마스터':>10s} {'일치':>5s}")
for key, col in [('bp_fatigue_1d','home_bp_fatigue_1d'), ('bp_fatigue_3d','home_bp_fatigue_3d')]:
    calc_val = hou.get(key, 0.0)
    ref_val = game_414[col]
    match = '✅' if abs(calc_val - ref_val) < 0.01 else '❌'
    print(f"  {key:>20s} {calc_val:>10.1f} {ref_val:>10.1f} {match:>5s}")

# Texas (원정)
tex = fat.get('Texas Rangers', {})
print(f"\n=== Texas Rangers 4/14 (원정) 검증 ===")
for key, col in [('bp_fatigue_1d','away_bp_fatigue_1d'), ('bp_fatigue_3d','away_bp_fatigue_3d')]:
    calc_val = tex.get(key, 0.0)
    ref_val = game_414[col]
    match = '✅' if abs(calc_val - ref_val) < 0.01 else '❌'
    print(f"  {key:>20s} {calc_val:>10.1f} {ref_val:>10.1f} {match:>5s}")

# === 대량 검증: 7월/8월/9월 ===
errors = 0
checks = 0
for date in ['2023-07-15', '2023-08-20', '2023-09-10']:
    fat = calc_bp_fatigue(results_2023, pit, st, date)
    day_games = ref[(ref['시즌'] == 2023) & (ref['날짜'].astype(str).str.contains(date))]
    for _, game in day_games.head(3).iterrows():
        for team, prefix in [(game['홈팀'], 'home'), (game['원정팀'], 'away')]:
            team_fat = fat.get(team, {})
            for key in ['bp_fatigue_1d', 'bp_fatigue_3d']:
                calc_val = team_fat.get(key, 0.0)
                ref_val = game[f'{prefix}_{key}']
                checks += 1
                if pd.isna(ref_val):
                    continue
                if abs(calc_val - ref_val) >= 0.1:
                    errors += 1
                    print(f"  ❌ {date} {team} {key}: 함수={calc_val:.1f}, 마스터={ref_val:.1f}")

print(f"\n총 {checks}건 검증, 오류 {errors}건")
if errors == 0:
    print("✅ 전부 일치 — 불펜소모도 함수 검증 완료")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

# Athletics 관련 팀명 확인
print("=== 마스터테이블 ===")
teams_ref = set(ref['홈팀'].unique()) | set(ref['원정팀'].unique())
ath_ref = [t for t in teams_ref if 'thletic' in t or 'Oakland' in t or t == 'Athletics']
print(f"  Athletics 관련: {ath_ref}")

print("\n=== 투수기록 ===")
teams_pit = pit['팀'].unique()
ath_pit = [t for t in teams_pit if 'thletic' in t or 'Oakland' in t or t == 'Athletics']
print(f"  Athletics 관련: {ath_pit}")

print("\n=== starters ===")
teams_st = set(st['home_team'].unique()) | set(st['away_team'].unique())
ath_st = [t for t in teams_st if 'thletic' in t or 'Oakland' in t or t == 'Athletics']
print(f"  Athletics 관련: {ath_st}")

# Athletics 경기 투수기록 있는지 직접 확인
print("\n=== 8/19 Athletics 경기 확인 ===")
ath_games = ref[(ref['시즌']==2023) & ((ref['홈팀'].str.contains('thletic')) | (ref['원정팀'].str.contains('thletic')))]
aug19 = ath_games[ath_games['날짜'].astype(str).str.contains('2023-08-19')]
if len(aug19) > 0:
    gpk = aug19.iloc[0]['gamePk']
    print(f"  gamePk={gpk}")
    game_pit = pit[pit['gamePk'] == gpk]
    print(f"  투수기록 행수: {len(game_pit)}")
    if len(game_pit) > 0:
        print(f"  팀명들: {game_pit['팀'].unique()}")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')

teams_ref = sorted(set(ref['홈팀'].unique()) | set(ref['원정팀'].unique()))
teams_pit = sorted(pit['팀'].unique())

print("=== 마스터테이블에만 있는 팀명 ===")
for t in teams_ref:
    if t not in teams_pit:
        print(f"  '{t}'")

print("\n=== 투수기록에만 있는 팀명 ===")
for t in teams_pit:
    if t not in teams_ref:
        print(f"  '{t}'")

# 2026도 확인
pit26 = pd.read_csv(f'{MLB_PATH}/2026/mlb_2026_정규시즌_투수개인기록.csv')
results26 = pd.read_csv(f'{MLB_PATH}/2026/mlb_2026_정규시즌_경기결과.csv')
teams_pit26 = sorted(pit26['팀'].unique())
teams_res26 = sorted(set(results26['홈팀'].unique()) | set(results26['원정팀'].unique()))

print("\n=== 2026 경기결과에만 있는 팀명 ===")
for t in teams_res26:
    if t not in teams_pit26:
        print(f"  '{t}'")

print("\n=== 2026 투수기록에만 있는 팀명 ===")
for t in teams_pit26:
    if t not in teams_res26:
        print(f"  '{t}'")

In [ ]:
# Athletics 오류만 수정해서 재검증
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀']].sort_values('날짜').reset_index(drop=True)

game_dates = results_2023[['gamePk','날짜']].drop_duplicates()
pit = pit.merge(game_dates, on='gamePk', how='left')

# 팀명 정규화 매핑
TEAM_NORMALIZE = {
    'Oakland Athletics': 'Athletics',
}

def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

def calc_bp_fatigue(results_df, pitchers_df, starters_df, target_date):
    prev_games = results_df[results_df['날짜'] < target_date]

    sp_names = {}
    for _, r in starters_df.iterrows():
        sp_names[(r['gamePk'], 'home')] = r.get('home_sp_name', '')
        sp_names[(r['gamePk'], 'away')] = r.get('away_sp_name', '')

    bp_innings = {}
    for gpk in prev_games['gamePk'].unique():
        game_pit = pitchers_df[pitchers_df['gamePk'] == gpk]
        for side in ['home', 'away']:
            team_pit = game_pit[game_pit['홈원정'] == side]
            if len(team_pit) == 0:
                continue
            sp_name = sp_names.get((gpk, side), '')
            relief = team_pit[team_pit['선수명'] != sp_name]
            bp_ip = relief['이닝'].sum()
            team_name = normalize_team(team_pit['팀'].iloc[0])  # ← 정규화 추가
            bp_innings[(gpk, team_name)] = bp_ip

    team_history = {}
    for _, row in prev_games.sort_values('날짜').iterrows():
        gpk = row['gamePk']
        date_str = str(row['날짜'])[:10]
        for team_col in ['홈팀', '원정팀']:
            team = row[team_col]  # 마스터테이블 기준이라 이미 정규화됨
            bp_ip = bp_innings.get((gpk, team), 0.0)
            team_history.setdefault(team, []).append((date_str, bp_ip))

    target_dt = pd.to_datetime(target_date)
    d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')

    fatigue = {}
    for team, history in team_history.items():
        sum_1d = sum(ip for d, ip in history if d >= d1 and d < str(target_date)[:10])
        sum_3d = sum(ip for d, ip in history if d >= d3 and d < str(target_date)[:10])
        fatigue[team] = {'bp_fatigue_1d': sum_1d, 'bp_fatigue_3d': sum_3d}

    return fatigue

# 이전에 실패한 케이스 포함해서 대량 검증
errors = 0
checks = 0
for date in ['2023-04-14', '2023-07-15', '2023-08-20', '2023-09-10']:
    fat = calc_bp_fatigue(results_2023, pit, st, date)
    day_games = ref[(ref['시즌'] == 2023) & (ref['날짜'].astype(str).str.contains(date))]
    for _, game in day_games.head(5).iterrows():
        for team, prefix in [(game['홈팀'], 'home'), (game['원정팀'], 'away')]:
            team_fat = fat.get(team, {})
            for key in ['bp_fatigue_1d', 'bp_fatigue_3d']:
                calc_val = team_fat.get(key, 0.0)
                ref_val = game[f'{prefix}_{key}']
                checks += 1
                if pd.isna(ref_val):
                    continue
                if abs(calc_val - ref_val) >= 0.1:
                    errors += 1
                    print(f"  ❌ {date} {team} {key}: 함수={calc_val:.1f}, 마스터={ref_val:.1f}")

print(f"\n총 {checks}건 검증, 오류 {errors}건")
if errors == 0:
    print("✅ 전부 일치 — 불펜소모도 함수 검증 완료 (Athletics 포함)")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# KC 7/12~7/15 경기 확인
for team in ['Kansas City Royals', 'Tampa Bay Rays']:
    games = ref[(ref['시즌']==2023) & ((ref['홈팀']==team) | (ref['원정팀']==team))]
    games = games[(games['날짜'].astype(str) >= '2023-07-12') & (games['날짜'].astype(str) <= '2023-07-15')]
    print(f"\n=== {team} 7/12~7/15 ===")
    for _, g in games.sort_values('날짜').iterrows():
        side = '홈' if g['홈팀'] == team else '원정'
        bp_col = f'{"home" if side == "홈" else "away"}_bp_fatigue_1d'
        print(f"  {str(g['날짜'])[:10]} {side} gamePk={g['gamePk']} bp_1d={g[bp_col]} bp_3d={g[f'{bp_col.replace('1d','3d')}']}")

# 7/14 경기가 results_2023에 있는지 확인
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀']].sort_values('날짜')
jul14 = results_2023[results_2023['날짜'].astype(str).str.contains('2023-07-14')]
print(f"\n=== results_2023에서 7/14 경기 ===")
print(f"  {len(jul14)}경기")
if len(jul14) > 0:
    print(jul14[['날짜','gamePk','홈팀','원정팀']].to_string(index=False))

# 투수기록에 해당 gamePk 있는지
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
game_dates = results_2023[['gamePk','날짜']].drop_duplicates()
pit_merged = pit.merge(game_dates, on='gamePk', how='left')

# KC 7/14 경기 투수기록
kc_games = ref[(ref['시즌']==2023) & ((ref['홈팀']=='Kansas City Royals') | (ref['원정팀']=='Kansas City Royals'))]
kc_jul14 = kc_games[kc_games['날짜'].astype(str).str.contains('2023-07-14')]
if len(kc_jul14) > 0:
    gpk = kc_jul14.iloc[0]['gamePk']
    kc_pit = pit[pit['gamePk'] == gpk]
    print(f"\n=== KC 7/14 gamePk={gpk} 투수기록 ===")
    print(f"  행수: {len(kc_pit)}")
    if len(kc_pit) > 0:
        print(kc_pit[['팀','홈원정','선수명','이닝']].to_string(index=False))

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 2023 더블헤더 모두 찾기
day_counts = ref[ref['시즌']==2023].groupby(['날짜','홈팀']).size()
doubles = day_counts[day_counts > 1]
print(f"2023 더블헤더: {len(doubles)}건\n")

for (date, team), cnt in doubles.items():
    games = ref[(ref['시즌']==2023) & (ref['날짜'].astype(str).str.contains(str(date)[:10])) & (ref['홈팀']==team)]
    games = games.sort_values('gamePk')
    print(f"=== {str(date)[:10]} {team} ({cnt}경기) ===")
    for _, g in games.iterrows():
        print(f"  gamePk={g['gamePk']} bp_1d={g['home_bp_fatigue_1d']} bp_3d={g['home_bp_fatigue_3d']}")

    # 1차전=bp 낮은 쪽, 2차전=bp 높은 쪽 → gamePk 순서와 비교
    bps = games['home_bp_fatigue_3d'].tolist()
    gpks = games['gamePk'].tolist()
    if bps[0] <= bps[1]:
        order = "gamePk 작은 게 1차전" if gpks[0] < gpks[1] else "gamePk 큰 게 1차전"
    else:
        order = "gamePk 작은 게 2차전" if gpks[0] < gpks[1] else "gamePk 큰 게 2차전"
    print(f"  → {order}\n")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 2023 더블헤더 gamePk 하나로 MLB API 구조 확인
# 717393, 717411 = 7/15 KC vs TB 더블헤더
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 경기결과 CSV에 시리즈 번호 같은 필드 있는지
results = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_경기결과.csv')
print(f"경기결과 컬럼: {list(results.columns)}")

# 더블헤더 경기 확인
dh = results[results['날짜'].astype(str).str.contains('2023-07-15')]
print(f"\n7/15 경기:")
print(dh.to_string())

# 2026 경기결과도 확인
results26 = pd.read_csv(f'{MLB_PATH}/2026/mlb_2026_정규시즌_경기결과.csv')
print(f"\n2026 경기결과 컬럼: {list(results26.columns)}")

In [ ]:
import requests
import pandas as pd

# 2023-07-15 KC vs TB 더블헤더로 테스트
url = "https://statsapi.mlb.com/api/v1/schedule"
params = {
    "sportId": 1,
    "date": "2023-07-15",
    "gameType": "R"
}
resp = requests.get(url, params=params).json()

print("=== 2023-07-15 MLB API schedule ===")
for date_info in resp.get('dates', []):
    for g in date_info.get('games', []):
        gpk = g['gamePk']
        home = g['teams']['home']['team']['name']
        away = g['teams']['away']['team']['name']
        game_num = g.get('gameNumber', '?')
        desc = g.get('description', '')
        series_desc = g.get('seriesDescription', '')
        dh = g.get('doubleHeader', '')
        print(f"  gamePk={gpk} gameNumber={game_num} DH={dh} | {away} @ {home} | {desc}")

# KC 더블헤더만 필터
print("\n=== KC 더블헤더 확인 ===")
for date_info in resp.get('dates', []):
    for g in date_info.get('games', []):
        if 'Kansas City' in g['teams']['home']['team']['name']:
            gpk = g['gamePk']
            game_num = g.get('gameNumber', '?')
            print(f"  gamePk={gpk} gameNumber={game_num}")

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# === 1. 2023 스케줄 + gameNumber 수집 ===
print("1) 스케줄 수집...")
url = "https://statsapi.mlb.com/api/v1/schedule"
params = {"sportId": 1, "season": 2023, "gameType": "R",
          "startDate": "2023-03-01", "endDate": "2023-10-31"}
resp = requests.get(url, params=params, timeout=30).json()

sched_rows = []
for date_info in resp.get('dates', []):
    for g in date_info.get('games', []):
        sched_rows.append({
            'gamePk': g['gamePk'],
            'sched_date': date_info['date'],
            'gameNumber': g.get('gameNumber', 1),
        })
sched = pd.DataFrame(sched_rows).drop_duplicates(subset='gamePk', keep='last')
print(f"   스케줄: {len(sched)}경기, 더블헤더(gameNumber>1): {(sched['gameNumber']>1).sum()}건")

# === 2. 데이터 로드 ===
print("2) 데이터 로드...")
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀']].sort_values('날짜').reset_index(drop=True)

# gameNumber 붙이기
results_2023 = results_2023.merge(sched[['gamePk','gameNumber']], on='gamePk', how='left')
results_2023['gameNumber'] = results_2023['gameNumber'].fillna(1).astype(int)
results_2023 = results_2023.sort_values(['날짜','gameNumber']).reset_index(drop=True)

# 투수기록에 날짜 붙이기
game_dates = results_2023[['gamePk','날짜']].drop_duplicates()
pit = pit.merge(game_dates, on='gamePk', how='left')

TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

print(f"   경기결과: {len(results_2023)}, 투수기록: {len(pit)}")

# === 3. 수정된 함수: 경기 순서 기반 ===
def calc_bp_fatigue_all(results_df, pitchers_df, starters_df):
    """
    전체 경기를 (날짜, gameNumber) 순으로 처리
    경기마다 "이 경기 이전까지의 bp_fatigue" 계산
    리턴: {gamePk: {home_bp_1d, home_bp_3d, away_bp_1d, away_bp_3d}}
    """
    results_sorted = results_df.sort_values(['날짜','gameNumber']).reset_index(drop=True)

    # 선발투수 딕셔너리
    sp_names = {}
    for _, r in starters_df.iterrows():
        sp_names[(r['gamePk'], 'home')] = r.get('home_sp_name', '')
        sp_names[(r['gamePk'], 'away')] = r.get('away_sp_name', '')

    # 경기별 불펜 이닝 미리 계산
    bp_ip_map = {}  # (gamePk, team) → bp_innings
    for gpk in results_sorted['gamePk'].unique():
        game_pit = pitchers_df[pitchers_df['gamePk'] == gpk]
        for side in ['home', 'away']:
            team_pit = game_pit[game_pit['홈원정'] == side]
            if len(team_pit) == 0:
                continue
            sp_name = sp_names.get((gpk, side), '')
            relief = team_pit[team_pit['선수명'] != sp_name]
            bp_ip = relief['이닝'].sum()
            team_name = normalize_team(team_pit['팀'].iloc[0])
            bp_ip_map[(gpk, team_name)] = bp_ip

    # 팀별 히스토리를 경기 순서대로 쌓으면서 각 경기의 fatigue 계산
    team_history = {}  # team → [(date_str, bp_ip), ...]
    fatigue_result = {}  # gamePk → {home_bp_1d, home_bp_3d, away_bp_1d, away_bp_3d}

    for _, row in results_sorted.iterrows():
        gpk = row['gamePk']
        date_str = str(row['날짜'])[:10]
        target_dt = pd.to_datetime(date_str)
        d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')

        game_fat = {}
        for team_col, prefix in [('홈팀', 'home'), ('원정팀', 'away')]:
            team = row[team_col]
            history = team_history.get(team, [])

            # 같은 날 이전 경기도 포함 (더블헤더 1차전 → 2차전)
            # history에는 이미 이 경기 이전까지 append됨
            sum_1d = sum(ip for d, ip in history if d >= d1)
            sum_3d = sum(ip for d, ip in history if d >= d3)

            game_fat[f'{prefix}_bp_fatigue_1d'] = sum_1d
            game_fat[f'{prefix}_bp_fatigue_3d'] = sum_3d

        fatigue_result[gpk] = game_fat

        # 현재 경기 결과를 히스토리에 추가 (다음 경기 계산용)
        for team_col in ['홈팀', '원정팀']:
            team = row[team_col]
            bp_ip = bp_ip_map.get((gpk, team), 0.0)
            team_history.setdefault(team, []).append((date_str, bp_ip))

    return fatigue_result

# === 4. 전체 계산 ===
print("3) 불펜소모도 계산...")
fat_all = calc_bp_fatigue_all(results_2023, pit, st)
print(f"   {len(fat_all)}경기 계산 완료")

# === 5. 검증: 더블헤더 KC 7/15 ===
print(f"\n=== KC 7/15 더블헤더 검증 ===")
for gpk in [717411, 717393]:
    calc = fat_all.get(gpk, {})
    game = ref[ref['gamePk'] == gpk].iloc[0]
    gn = results_2023[results_2023['gamePk'] == gpk]['gameNumber'].iloc[0]
    print(f"\n  gamePk={gpk} (gameNumber={gn})")
    for key in ['home_bp_fatigue_1d', 'home_bp_fatigue_3d', 'away_bp_fatigue_1d', 'away_bp_fatigue_3d']:
        calc_val = calc.get(key, 0.0)
        ref_val = game[key]
        match = '✅' if abs(calc_val - ref_val) < 0.1 else '❌'
        print(f"    {key:>25s}  함수={calc_val:>6.1f}  마스터={ref_val:>6.1f}  {match}")

# === 6. 대량 검증 (이전 실패 포함) ===
print(f"\n=== 대량 검증 ===")
errors = 0
checks = 0
for _, game in ref[ref['시즌'] == 2023].iterrows():
    gpk = game['gamePk']
    if gpk not in fat_all:
        continue
    calc = fat_all[gpk]
    for prefix in ['home', 'away']:
        for key in ['bp_fatigue_1d', 'bp_fatigue_3d']:
            full_key = f'{prefix}_{key}'
            calc_val = calc.get(full_key, 0.0)
            ref_val = game[full_key]
            checks += 1
            if pd.isna(ref_val):
                continue
            if abs(calc_val - ref_val) >= 0.1:
                errors += 1
                if errors <= 10:
                    print(f"  ❌ gamePk={gpk} {game['홈팀']} {full_key}: 함수={calc_val:.1f}, 마스터={ref_val:.1f}")

print(f"\n총 {checks}건 검증, 오류 {errors}건")
if errors == 0:
    print("✅ 전부 일치 — 불펜소모도 함수 검증 완료 (더블헤더 포함)")

In [ ]:
import pandas as pd

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀']].sort_values('날짜')

# 오류 gamePk 확인
error_gpks = [718700, 718684, 718685]
for gpk in error_gpks:
    game = ref[ref['gamePk'] == gpk].iloc[0]
    in_results = gpk in results_2023['gamePk'].values
    print(f"gamePk={gpk}: 날짜={game['날짜']}, {game['원정팀']} @ {game['홈팀']}, results에 있음={in_results}")

# gamePk 범위 확인
print(f"\nresults_2023 gamePk 범위: {results_2023['gamePk'].min()} ~ {results_2023['gamePk'].max()}")
print(f"718xxx gamePk 수: {(results_2023['gamePk'] >= 718000).sum()}")

# 스케줄(2430) vs results(2420) — 빠진 10경기 확인
print(f"\n스케줄: {len(sched)}경기, results: {len(results_2023)}경기")
sched_gpks = set(sched['gamePk'])
results_gpks = set(results_2023['gamePk'])
missing = sched_gpks - results_gpks
extra = results_gpks - sched_gpks
print(f"스케줄에만 있는: {len(missing)}개 → {sorted(missing)[:10]}")
print(f"results에만 있는: {len(extra)}개 → {sorted(extra)[:10]}")

# 오류 경기 전날 경기 있는지 확인 (Cincinnati 9/1)
print(f"\n=== Cincinnati 8/31~9/1 경기 ===")
cin = ref[(ref['시즌']==2023) & ((ref['홈팀']=='Cincinnati Reds') | (ref['원정팀']=='Cincinnati Reds'))]
cin_aug = cin[(cin['날짜'].astype(str) >= '2023-08-29') & (cin['날짜'].astype(str) <= '2023-09-02')]
for _, g in cin_aug.sort_values('날짜').iterrows():
    side = '홈' if g['홈팀'] == 'Cincinnati Reds' else '원정'
    print(f"  {str(g['날짜'])[:10]} gamePk={g['gamePk']} {side} bp_1d={g['home_bp_fatigue_1d' if side=='홈' else 'away_bp_fatigue_1d']}")

In [ ]:
# ============================================================
# 불펜소모도 오류 462건 원인 분석
# ============================================================
import pandas as pd
import numpy as np
import requests

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# === 1. 데이터 로드 ===
print("1) 데이터 로드...")
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀',
    'home_bp_fatigue_1d','home_bp_fatigue_3d',
    'away_bp_fatigue_1d','away_bp_fatigue_3d']].sort_values('날짜').reset_index(drop=True)

game_dates = results_2023[['gamePk','날짜']].drop_duplicates()
pit = pit.merge(game_dates, on='gamePk', how='left')

print(f"   경기: {len(results_2023)}, 투수: {len(pit)}, starters: {len(st)}")

# === 2. 스케줄 + gameNumber 수집 ===
print("2) 스케줄 수집...")
url = "https://statsapi.mlb.com/api/v1/schedule"
params = {"sportId": 1, "season": 2023, "gameType": "R",
          "startDate": "2023-03-01", "endDate": "2023-10-31"}
resp = requests.get(url, params=params, timeout=30).json()

sched_rows = []
for date_info in resp.get('dates', []):
    for g in date_info.get('games', []):
        sched_rows.append({
            'gamePk': g['gamePk'],
            'sched_date': date_info['date'],
            'gameNumber': g.get('gameNumber', 1),
        })
sched = pd.DataFrame(sched_rows).drop_duplicates(subset='gamePk', keep='last')
print(f"   스케줄: {len(sched)}경기, DH: {(sched['gameNumber']>1).sum()}건")

# results에 gameNumber 붙이기
results_2023 = results_2023.merge(sched[['gamePk','gameNumber']], on='gamePk', how='left')
results_2023['gameNumber'] = results_2023['gameNumber'].fillna(1).astype(int)

# === 3. calc_bp_fatigue_all 실행 ===
print("3) 불펜소모도 계산...")

TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

def calc_bp_fatigue_all(results_df, pitchers_df, starters_df):
    results_sorted = results_df.sort_values(['날짜','gameNumber']).reset_index(drop=True)
    sp_names = {}
    for _, r in starters_df.iterrows():
        sp_names[(r['gamePk'], 'home')] = r.get('home_sp_name', '')
        sp_names[(r['gamePk'], 'away')] = r.get('away_sp_name', '')
    bp_ip_map = {}
    for gpk in results_sorted['gamePk'].unique():
        game_pit = pitchers_df[pitchers_df['gamePk'] == gpk]
        for side in ['home', 'away']:
            team_pit = game_pit[game_pit['홈원정'] == side]
            if len(team_pit) == 0: continue
            sp_name = sp_names.get((gpk, side), '')
            relief = team_pit[team_pit['선수명'] != sp_name]
            bp_ip = relief['이닝'].sum()
            team_name = normalize_team(team_pit['팀'].iloc[0])
            bp_ip_map[(gpk, team_name)] = bp_ip
    team_history = {}
    fatigue_result = {}
    for _, row in results_sorted.iterrows():
        gpk = row['gamePk']
        date_str = str(row['날짜'])[:10]
        target_dt = pd.to_datetime(date_str)
        d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
        game_fat = {}
        for team_col, prefix in [('홈팀','home'),('원정팀','away')]:
            team = row[team_col]
            history = team_history.get(team, [])
            sum_1d = sum(ip for d, ip in history if d >= d1)
            sum_3d = sum(ip for d, ip in history if d >= d3)
            game_fat[f'{prefix}_bp_fatigue_1d'] = sum_1d
            game_fat[f'{prefix}_bp_fatigue_3d'] = sum_3d
        fatigue_result[gpk] = game_fat
        for team_col in ['홈팀','원정팀']:
            team = row[team_col]
            bp_ip = bp_ip_map.get((gpk, team), 0.0)
            team_history.setdefault(team, []).append((date_str, bp_ip))
    return fatigue_result

fat_all = calc_bp_fatigue_all(results_2023, pit, st)
print(f"   {len(fat_all)}경기 계산 완료")

# === 4. 오류 수집 + 패턴 분석 ===
print("\n4) 오류 분석...")

errors = []
for _, row in results_2023.iterrows():
    gpk = row['gamePk']
    fat = fat_all.get(gpk, {})
    for prefix in ['home', 'away']:
        for key in ['bp_fatigue_1d', 'bp_fatigue_3d']:
            col = f'{prefix}_{key}'
            calc_val = fat.get(col, 0.0)
            ref_val = row[col]
            if pd.isna(ref_val):
                continue
            if abs(calc_val - ref_val) >= 0.05:
                errors.append({
                    'gamePk': gpk,
                    'date': str(row['날짜'])[:10],
                    'gameNumber': row['gameNumber'],
                    'team': row['홈팀'] if prefix == 'home' else row['원정팀'],
                    'side': prefix,
                    'field': key,
                    'calc': calc_val,
                    'master': ref_val,
                    'diff': ref_val - calc_val,
                })

err_df = pd.DataFrame(errors)
print(f"   총 오류: {len(err_df)}건")

if len(err_df) == 0:
    print("   ✅ 오류 없음!")
else:
    # --- 분석 1: 날짜별 오류 분포 ---
    print(f"\n=== 오류 날짜 분포 (상위 15) ===")
    date_cnt = err_df.groupby('date').size().sort_values(ascending=False)
    for date, cnt in date_cnt.head(15).items():
        # 해당 날짜에 같은 홈팀으로 경기 2개 있으면 더블헤더
        day_games = results_2023[results_2023['날짜'].astype(str).str.contains(date)]
        dh = day_games.groupby('홈팀').size()
        is_dh = (dh > 1).any()
        print(f"  {date}: {cnt}건 오류 (경기 {len(day_games)}개, DH={'YES' if is_dh else 'NO'})")

    # --- 분석 2: 오류가 더블헤더 날짜에만 발생하는지 ---
    dh_dates = set()
    for date in results_2023['날짜'].astype(str).str[:10].unique():
        day = results_2023[results_2023['날짜'].astype(str).str.contains(date)]
        if day.groupby('홈팀').size().max() > 1:
            dh_dates.add(date)

    err_on_dh = err_df[err_df['date'].isin(dh_dates)]
    err_not_dh = err_df[~err_df['date'].isin(dh_dates)]
    print(f"\n=== 더블헤더 vs 비더블헤더 ===")
    print(f"  더블헤더 날짜 오류: {len(err_on_dh)}건")
    print(f"  비더블헤더 날짜 오류: {len(err_not_dh)}건")

    # --- 분석 3: 비더블헤더 오류 상세 (원인 찾기) ---
    if len(err_not_dh) > 0:
        print(f"\n=== 비더블헤더 오류 상세 (상위 10) ===")
        for _, e in err_not_dh.head(10).iterrows():
            team = e['team']
            date = e['date']
            # 전날 경기 있었는지
            prev_date = (pd.to_datetime(date) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
            prev_games = results_2023[
                (results_2023['날짜'].astype(str).str.contains(prev_date)) &
                ((results_2023['홈팀'] == team) | (results_2023['원정팀'] == team))
            ]
            # 투수기록에 해당 경기 있는지
            prev_bp = []
            for _, pg in prev_games.iterrows():
                bp_ip = 0.0
                gpk_prev = pg['gamePk']
                gp = pit[pit['gamePk'] == gpk_prev]
                side = 'home' if pg['홈팀'] == team else 'away'
                tp = gp[gp['홈원정'] == side]
                if len(tp) > 0:
                    sp_name = ''
                    sp_key = (gpk_prev, side)
                    for _, r in st.iterrows():
                        if r['gamePk'] == gpk_prev:
                            sp_name = r.get(f'{side}_sp_name', '')
                            break
                    relief = tp[tp['선수명'] != sp_name]
                    bp_ip = relief['이닝'].sum()
                prev_bp.append(bp_ip)

            print(f"  {e['date']} {e['team']:25s} {e['side']} {e['field']:15s} "
                  f"함수={e['calc']:.1f} 마스터={e['master']:.1f} "
                  f"전날경기={len(prev_games)}개 전날BP={prev_bp}")

    # --- 분석 4: gameNumber 할당 상태 ---
    print(f"\n=== 오류 경기의 gameNumber 분포 ===")
    print(err_df.groupby('gameNumber').size())

    # --- 분석 5: diff 분포 (마스터 - 함수) ---
    print(f"\n=== diff 분포 (마스터 - 함수) ===")
    print(f"  항상 양수? {(err_df['diff'] > 0).all()}")
    print(f"  diff 평균: {err_df['diff'].mean():.2f}")
    print(f"  diff 최대: {err_df['diff'].max():.2f}")
    print(f"  calc=0인 오류: {(err_df['calc'] == 0).sum()}/{len(err_df)}")

print("\n완료!")

In [ ]:
# ============================================================
# 오류 원인 심층 분석
# ============================================================
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

results_2023 = ref[ref['시즌'] == 2023][['날짜','gamePk','홈팀','원정팀',
    '홈팀득점','원정팀득점','홈팀결과','원정팀결과',
    'home_bp_fatigue_1d','home_bp_fatigue_3d',
    'away_bp_fatigue_1d','away_bp_fatigue_3d']].sort_values('날짜').reset_index(drop=True)

# ============================================================
# 분석 1: 4/7 오류 — 4/5~4/6에 경기가 정말 없는지?
# ============================================================
print("=" * 70)
print("분석 1: 4/7 오류팀의 4/4~4/6 경기 존재 여부")
print("=" * 70)

error_teams_407 = ['Chicago Cubs','New York Mets','Miami Marlins',
                   'Baltimore Orioles','New York Yankees','Philadelphia Phillies']

for team in error_teams_407:
    team_games = results_2023[
        (results_2023['홈팀'] == team) | (results_2023['원정팀'] == team)
    ].sort_values('날짜')

    apr_games = team_games[team_games['날짜'].astype(str).str[:7] == '2023-04']
    print(f"\n  {team} 4월 초 경기:")
    for _, g in apr_games.head(8).iterrows():
        side = '홈' if g['홈팀'] == team else '원정'
        print(f"    {str(g['날짜'])[:10]} {side} gamePk={g['gamePk']}")

# ============================================================
# 분석 2: results_2023 날짜 vs 투수기록 날짜 불일치 체크
# ============================================================
print("\n" + "=" * 70)
print("분석 2: 투수기록에는 있지만 results_2023에 없는 gamePk")
print("=" * 70)

pit_gpks = set(pit['gamePk'].unique())
res_gpks = set(results_2023['gamePk'].unique())

only_pit = pit_gpks - res_gpks
only_res = res_gpks - pit_gpks

print(f"  투수기록에만 있는 gamePk: {len(only_pit)}개")
if len(only_pit) > 0:
    print(f"  예시: {sorted(only_pit)[:10]}")
print(f"  results에만 있는 gamePk: {len(only_res)}개")

# 투수기록에만 있는 경기 상세
if len(only_pit) > 0:
    print(f"\n  투수기록에만 있는 경기 상세:")
    for gpk in sorted(only_pit)[:10]:
        gp = pit[pit['gamePk'] == gpk]
        teams = gp['팀'].unique()
        print(f"    gamePk={gpk} 팀: {teams}")

# ============================================================
# 분석 3: 원본 마스터테이블 빌드 시 2022 시즌 데이터도 썼는지?
# ============================================================
print("\n" + "=" * 70)
print("분석 3: 마스터테이블이 교차시즌 누적인지 확인")
print("=" * 70)

# 2023 시즌 첫 경기(3/30)의 bp_fatigue_3d가 0이 아니면 → 2022 데이터도 누적한 것
first_games = results_2023.head(20)
non_zero_3d = first_games[
    (first_games['home_bp_fatigue_3d'] > 0) | (first_games['away_bp_fatigue_3d'] > 0)
]
print(f"  2023 처음 20경기 중 bp_fatigue_3d > 0인 경기: {len(non_zero_3d)}건")
if len(non_zero_3d) > 0:
    for _, g in non_zero_3d.head(5).iterrows():
        print(f"    {str(g['날짜'])[:10]} {g['원정팀']} @ {g['홈팀']} "
              f"home_3d={g['home_bp_fatigue_3d']:.1f} away_3d={g['away_bp_fatigue_3d']:.1f}")

# ============================================================
# 분석 4: 실제 오류 경기 하나를 완전 추적
# ============================================================
print("\n" + "=" * 70)
print("분석 4: NY Mets 4/7 완전 추적")
print("=" * 70)

team = 'New York Mets'
target = '2023-04-07'

# Mets 전체 경기 (시즌 무관)
all_mets = ref[(ref['홈팀'] == team) | (ref['원정팀'] == team)].sort_values('날짜')

# 2023-04-04 ~ 04-07
mets_window = all_mets[
    (all_mets['날짜'].astype(str) >= '2023-04-01') &
    (all_mets['날짜'].astype(str) <= '2023-04-07')
]
print(f"  Mets 4/1~4/7 경기 (전체 시즌):")
for _, g in mets_window.iterrows():
    side = '홈' if g['홈팀'] == team else '원정'
    print(f"    {str(g['날짜'])[:10]} {side} gamePk={g['gamePk']} 시즌={g['시즌']}")

# 2022 마지막 Mets 경기도 확인
mets_2022 = ref[(ref['시즌'] == 2022) & ((ref['홈팀'] == team) | (ref['원정팀'] == team))].sort_values('날짜')
if len(mets_2022) > 0:
    print(f"\n  Mets 2022 마지막 3경기:")
    for _, g in mets_2022.tail(3).iterrows():
        side = '홈' if g['홈팀'] == team else '원정'
        print(f"    {str(g['날짜'])[:10]} {side} gamePk={g['gamePk']}")

# ============================================================
# 분석 5: bp_fatigue 계산에 시범경기(Spring Training) 포함됐는지
# ============================================================
print("\n" + "=" * 70)
print("분석 5: 2023-03 경기가 시범경기인지 정규인지")
print("=" * 70)

mar_games = ref[ref['날짜'].astype(str).str[:7] == '2023-03']
print(f"  2023년 3월 경기: {len(mar_games)}건")
if len(mar_games) > 0:
    print(f"  시즌값: {mar_games['시즌'].unique()}")
    print(f"  시리즈값: {mar_games['시리즈'].unique() if '시리즈' in mar_games.columns else 'N/A'}")
    print(f"  날짜 범위: {mar_games['날짜'].min()} ~ {mar_games['날짜'].max()}")

# ============================================================
# 분석 6: diff 부호별 패턴 — 함수 > 마스터 (음수 diff) 케이스
# ============================================================
print("\n" + "=" * 70)
print("분석 6: diff < 0 (함수가 마스터보다 큰) 케이스 상세")
print("=" * 70)

# 다시 오류 수집 (fatigue_result 필요하므로 간단히)
TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

# fat_all은 위 코드에서 이미 계산했으면 재사용, 아니면 다시 계산
# (이전 셀에서 fat_all이 메모리에 남아있을 경우)
try:
    _ = fat_all
except:
    print("  fat_all 없음 — 이전 셀 먼저 실행 필요")
    fat_all = {}

neg_errors = []
for _, row in results_2023.iterrows():
    gpk = row['gamePk']
    fat = fat_all.get(gpk, {})
    for prefix in ['home', 'away']:
        for key in ['bp_fatigue_1d', 'bp_fatigue_3d']:
            col = f'{prefix}_{key}'
            calc_val = fat.get(col, 0.0)
            ref_val = row[col]
            if pd.isna(ref_val): continue
            diff = ref_val - calc_val
            if diff < -0.05:
                neg_errors.append({
                    'gamePk': gpk, 'date': str(row['날짜'])[:10],
                    'team': row['홈팀'] if prefix == 'home' else row['원정팀'],
                    'field': col, 'calc': calc_val, 'master': ref_val, 'diff': diff
                })

neg_df = pd.DataFrame(neg_errors)
print(f"  함수 > 마스터 (diff < 0): {len(neg_df)}건")
if len(neg_df) > 0:
    print(f"\n  상세 (상위 10):")
    for _, e in neg_df.head(10).iterrows():
        print(f"    {e['date']} {e['team']:25s} {e['field']:25s} "
              f"함수={e['calc']:.1f} 마스터={e['master']:.1f} diff={e['diff']:.1f}")

print("\n완료!")

In [ ]:
# Mets 4/5~4/8 경기별 실제 불펜이닝 추적
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

team = 'New York Mets'
for date in ['2023-04-04','2023-04-05','2023-04-06','2023-04-07','2023-04-08']:
    games = ref[(ref['시즌']==2023) & (ref['날짜'].astype(str).str[:10]==date) &
                ((ref['홈팀']==team)|(ref['원정팀']==team))]
    if len(games)==0:
        print(f"{date}: 경기없음")
        continue
    for _, g in games.iterrows():
        gpk = g['gamePk']
        side = 'home' if g['홈팀']==team else 'away'
        # 선발투수
        sp_row = st[st['gamePk']==gpk]
        sp_name = sp_row[f'{side}_sp_name'].iloc[0] if len(sp_row)>0 else '?'
        # 불펜이닝
        gp = pit[(pit['gamePk']==gpk)&(pit['홈원정']==side)]
        relief = gp[gp['선수명']!=sp_name]
        bp_ip = relief['이닝'].sum() if len(relief)>0 else 0
        # 마스터값
        bp1d = g[f'{side}_bp_fatigue_1d']
        bp3d = g[f'{side}_bp_fatigue_3d']
        print(f"{date} gpk={gpk} {side} 선발={sp_name:15s} "
              f"불펜IP={bp_ip:.1f} 마스터_1d={bp1d:.1f} 마스터_3d={bp3d:.1f}")

In [ ]:
# 원본 마스터테이블 빌드 코드에서 bp_fatigue 로직 찾기
import json
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

for nb_name in ['믈브ML(v1).ipynb', 'mlb머신러닝.ipynb', 'Untitled0.ipynb']:
    fpath = f'{MLB_PATH}/{nb_name}'
    try:
        with open(fpath, 'r', encoding='utf-8') as fp:
            nb = json.load(fp)
    except:
        continue
    for i, cell in enumerate(nb.get('cells', [])):
        src = ''.join(cell.get('source', []))
        if 'bp_fatigue' in src and cell['cell_type'] == 'code':
            print(f"\n{'='*60}")
            print(f"{nb_name} CELL {i}")
            print(f"{'='*60}")
            print(src[:5000])
            if len(src) > 5000:
                print(f"\n... (총 {len(src)}자)")

In [ ]:
import json
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

fpath = f'{MLB_PATH}/크롤링/mlb_backtest_csv_generator.ipynb'
with open(fpath, 'r', encoding='utf-8') as fp:
    nb = json.load(fp)

for i, cell in enumerate(nb.get('cells', [])):
    src = ''.join(cell.get('source', []))
    if cell['cell_type'] == 'code':
        if any(k in src.lower() for k in ['불펜','bullpen','bp_fatigue','bp_ip','relief','pitcher']):
            print(f"\n{'='*60}")
            print(f"csv_generator CELL {i}")
            print(f"{'='*60}")
            print(src[:5000])

In [ ]:
import json
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

fpath = f'{MLB_PATH}/크롤링/mlb_backtest_data_builder.ipynb'
with open(fpath, 'r', encoding='utf-8') as fp:
    nb = json.load(fp)

for i, cell in enumerate(nb.get('cells', [])):
    src = ''.join(cell.get('source', []))
    if cell['cell_type'] == 'code':
        if any(k in src for k in ['bp_fatigue','불펜','bulpen','bp_ip','fatigue','이닝']):
            print(f"\n{'='*60}")
            print(f"data_builder CELL {i}")
            print(f"{'='*60}")
            print(src[:5000])

In [ ]:
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
ref21 = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블_2021.csv')
ref22 = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블_2022.csv')

# 2022 데이터로 검증 (2021은 시즌초라 데이터 적음)
pit22 = pd.read_csv(f'{MLB_PATH}/2022/mlb_2022_정규시즌_투수개인기록.csv')
st22 = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2022.csv')

results_22 = ref22[['date','gamePk','홈팀','원정팀',
    'home_bp_fatigue_1d','home_bp_fatigue_3d',
    'away_bp_fatigue_1d','away_bp_fatigue_3d']].sort_values('date').reset_index(drop=True)

game_dates = results_22[['gamePk','date']].drop_duplicates()
pit22 = pit22.merge(game_dates, left_on='gamePk', right_on='gamePk', how='left')
# 날짜 컬럼명 통일
if '날짜' not in pit22.columns and 'date' in pit22.columns:
    pit22['날짜'] = pit22['date']

TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

# gameNumber는 2022에서도 필요할 수 있으니 수집
import requests
url = "https://statsapi.mlb.com/api/v1/schedule"
params = {"sportId":1,"season":2022,"gameType":"R",
          "startDate":"2022-03-01","endDate":"2022-10-31"}
resp = requests.get(url, params=params, timeout=30).json()
sched_rows = []
for d in resp.get('dates',[]):
    for g in d.get('games',[]):
        sched_rows.append({'gamePk':g['gamePk'],'gameNumber':g.get('gameNumber',1)})
sched22 = pd.DataFrame(sched_rows).drop_duplicates(subset='gamePk',keep='last')

results_22 = results_22.merge(sched22, on='gamePk', how='left')
results_22['gameNumber'] = results_22['gameNumber'].fillna(1).astype(int)

# calc_bp_fatigue_all 실행
def calc_bp_fatigue_all(results_df, pitchers_df, starters_df, date_col='date'):
    results_sorted = results_df.sort_values([date_col,'gameNumber']).reset_index(drop=True)
    sp_names = {}
    for _, r in starters_df.iterrows():
        sp_names[(r['gamePk'],'home')] = r.get('home_sp_name','')
        sp_names[(r['gamePk'],'away')] = r.get('away_sp_name','')
    bp_ip_map = {}
    for gpk in results_sorted['gamePk'].unique():
        game_pit = pitchers_df[pitchers_df['gamePk']==gpk]
        for side in ['home','away']:
            team_pit = game_pit[game_pit['홈원정']==side]
            if len(team_pit)==0: continue
            sp_name = sp_names.get((gpk,side),'')
            relief = team_pit[team_pit['선수명']!=sp_name]
            bp_ip = relief['이닝'].sum()
            team_name = normalize_team(team_pit['팀'].iloc[0])
            bp_ip_map[(gpk,team_name)] = bp_ip
    team_history = {}
    fatigue_result = {}
    for _, row in results_sorted.iterrows():
        gpk = row['gamePk']
        date_str = str(row[date_col])[:10]
        target_dt = pd.to_datetime(date_str)
        d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
        d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
        game_fat = {}
        for team_col, prefix in [('홈팀','home'),('원정팀','away')]:
            team = row[team_col]
            history = team_history.get(team,[])
            sum_1d = sum(ip for d,ip in history if d >= d1)
            sum_3d = sum(ip for d,ip in history if d >= d3)
            game_fat[f'{prefix}_bp_fatigue_1d'] = sum_1d
            game_fat[f'{prefix}_bp_fatigue_3d'] = sum_3d
        fatigue_result[gpk] = game_fat
        for team_col in ['홈팀','원정팀']:
            team = row[team_col]
            bp_ip = bp_ip_map.get((gpk,team),0.0)
            team_history.setdefault(team,[]).append((date_str,bp_ip))
    return fatigue_result

fat22 = calc_bp_fatigue_all(results_22, pit22, st22, date_col='date')
print(f"2022: {len(fat22)}경기 계산 완료")

# 검증
errors = 0
checks = 0
for _, row in results_22.iterrows():
    gpk = row['gamePk']
    fat = fat22.get(gpk,{})
    for prefix in ['home','away']:
        for key in ['bp_fatigue_1d','bp_fatigue_3d']:
            col = f'{prefix}_{key}'
            calc_val = fat.get(col,0.0)
            ref_val = row[col]
            if pd.isna(ref_val): continue
            checks += 1
            if abs(calc_val - ref_val) >= 0.05:
                errors += 1

print(f"2022 검증: {checks}건 중 오류 {errors}건 ({errors/checks*100:.1f}%)")
if errors == 0:
    print("✅ 2022 완전 일치 — 함수 검증 완료")
else:
    print(f"❌ 오류 {errors}건 — 추가 분석 필요")

In [ ]:
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# date 컬럼 실제 값 확인
print(f"date dtype: {master['date'].dtype}")
print(f"date 샘플 5개:")
for v in master['date'].head(5):
    print(f"  '{v}' (type: {type(v)})")

# Timestamp로 변환 후 시간 부분 확인
dates = pd.to_datetime(master['date'])
print(f"\nTimestamp 샘플:")
for v in dates.head(5):
    print(f"  {v} (hour={v.hour}, minute={v.minute})")

# 시간이 00:00:00이 아닌 경우 있는지
non_midnight = dates[dates.dt.hour != 0]
print(f"\n자정이 아닌 날짜: {len(non_midnight)}개")
if len(non_midnight) > 0:
    print(non_midnight.head(5))

In [ ]:
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')

# 선발 세트
starter_set = set()
for _, r in st.iterrows():
    starter_set.add((r['gamePk'], r['home_sp_name']))
    starter_set.add((r['gamePk'], r['away_sp_name']))

pit['is_starter'] = pit.apply(lambda r: (r['gamePk'], r['선수명']) in starter_set, axis=1)
bp = pit[~pit['is_starter']].copy()

# 경기별 팀별 불펜이닝 합산
bp_game = bp.groupby(['gamePk','팀']).agg(bp_ip=('이닝','sum')).reset_index()
bp_game['팀'] = bp_game['팀'].replace('Oakland Athletics','Athletics')

# 날짜 붙이기
date_map = master[['gamePk','date']].drop_duplicates()
bp_game = bp_game.merge(date_map, on='gamePk', how='left')
bp_game['date'] = pd.to_datetime(bp_game['date'])

# Mets 4/4~4/7 불펜이닝 확인
team = 'New York Mets'
mets_bp = bp_game[bp_game['팀']==team].sort_values('date')
mets_apr = mets_bp[(mets_bp['date']>='2023-04-01')&(mets_bp['date']<='2023-04-08')]
print(f"=== Mets 불펜이닝 (원본 방식) ===")
for _, r in mets_apr.iterrows():
    print(f"  {r['date'].strftime('%Y-%m-%d')} gpk={r['gamePk']} bp_ip={r['bp_ip']:.1f}")

# 원본 로직으로 4/7 fatigue 직접 계산
target = pd.Timestamp('2023-04-07')
team_past = bp_game[(bp_game['팀']==team) & (bp_game['date']<target)]
d1 = team_past[team_past['date'] >= target - pd.Timedelta(days=1)]
d3 = team_past[team_past['date'] >= target - pd.Timedelta(days=3)]
print(f"\n=== 원본 로직 4/7 ===")
print(f"  team_past: {len(team_past)}경기")
print(f"  d1 (>={target-pd.Timedelta(days=1)}): {len(d1)}경기, sum={d1['bp_ip'].sum():.1f}")
print(f"  d3 (>={target-pd.Timedelta(days=3)}): {len(d3)}경기, sum={d3['bp_ip'].sum():.1f}")
print(f"  마스터: home_bp_1d={master[(master['홈팀']=='New York Mets')&(master['date']=='2023-04-07')]['home_bp_fatigue_1d'].values}")

In [ ]:
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# date vs 날짜 불일치 확인
mismatch = master[master['date'] != master['날짜'].astype(str)]
print(f"date vs 날짜 불일치: {len(mismatch)}건 / {len(master)}건")

if len(mismatch) > 0:
    print(f"\n상위 10건:")
    for _, r in mismatch.head(10).iterrows():
        print(f"  gpk={r['gamePk']} date='{r['date']}' 날짜='{r['날짜']}' {r['원정팀']} @ {r['홈팀']}")

In [ ]:
# ============================================================
# 1단계: B안_마스터테이블 bp_fatigue를 '날짜'(실제일) 기준 재계산
# ============================================================
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 데이터 로드
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
pit23 = pd.read_csv(f'{MLB_PATH}/2023/mlb_2023_정규시즌_투수개인기록.csv')
pit24 = pd.read_csv(f'{MLB_PATH}/2024/mlb_2024_정규시즌_투수개인기록.csv')
pit25 = pd.read_csv(f'{MLB_PATH}/2025/mlb_2025_정규시즌_투수개인기록.csv')
pit_all = pd.concat([pit23, pit24, pit25], ignore_index=True)

st23 = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')
st24 = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2024.csv')
st25 = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2025.csv')
st_all = pd.concat([st23, st24, st25], ignore_index=True)

print(f"마스터: {master.shape}, 투수: {pit_all.shape}, starters: {st_all.shape}")
print(f"date vs 날짜 불일치: {(master['date'] != master['날짜'].astype(str)).sum()}건")

# 선발 세트
starter_set = set()
for _, r in st_all.iterrows():
    starter_set.add((r['gamePk'], r['home_sp_name']))
    starter_set.add((r['gamePk'], r['away_sp_name']))

pit_all['is_starter'] = pit_all.apply(lambda r: (r['gamePk'], r['선수명']) in starter_set, axis=1)
bp = pit_all[~pit_all['is_starter']].copy()
print(f"전체투수: {len(pit_all)} → 불펜: {len(bp)}")

# 경기별 팀별 불펜이닝
bp_game = bp.groupby(['gamePk','팀']).agg(bp_ip=('이닝','sum')).reset_index()
bp_game['팀'] = bp_game['팀'].replace('Oakland Athletics','Athletics')

# ★ 핵심: '날짜'(실제 경기일) 기준으로 날짜 붙이기
date_map = master[['gamePk','날짜']].drop_duplicates()
date_map.columns = ['gamePk','actual_date']
bp_game = bp_game.merge(date_map, on='gamePk', how='left')
bp_game['actual_date'] = pd.to_datetime(bp_game['actual_date'])
bp_game = bp_game.sort_values(['팀','actual_date','gamePk']).reset_index(drop=True)

print(f"불펜 경기로그: {bp_game.shape}")
print(f"날짜 NaN: {bp_game['actual_date'].isna().sum()}")

# 이전 1d/3d 계산 (원본과 동일 로직, 날짜만 actual_date 사용)
fatigue_1d = []
fatigue_3d = []
for idx, row in bp_game.iterrows():
    team_past = bp_game[(bp_game['팀']==row['팀']) & (bp_game['actual_date']<row['actual_date'])]
    d1 = team_past[team_past['actual_date'] >= row['actual_date'] - pd.Timedelta(days=1)]
    d3 = team_past[team_past['actual_date'] >= row['actual_date'] - pd.Timedelta(days=3)]
    fatigue_1d.append(d1['bp_ip'].sum())
    fatigue_3d.append(d3['bp_ip'].sum())
    if (idx+1) % 2000 == 0:
        print(f"  {idx+1}/{len(bp_game)} 처리중...")

bp_game['bp_fatigue_1d_new'] = fatigue_1d
bp_game['bp_fatigue_3d_new'] = fatigue_3d
print(f"\n계산 완료: {len(bp_game)}건")

# Mets 4/7 검증
team = 'New York Mets'
mets = bp_game[bp_game['팀']==team].sort_values('actual_date')
mets_apr = mets[(mets['actual_date']>='2023-04-04')&(mets['actual_date']<='2023-04-08')]
print(f"\n=== Mets 검증 ===")
for _, r in mets_apr.iterrows():
    print(f"  {r['actual_date'].strftime('%Y-%m-%d')} gpk={r['gamePk']} "
          f"bp_ip={r['bp_ip']:.1f} fat_1d={r['bp_fatigue_1d_new']:.1f} fat_3d={r['bp_fatigue_3d_new']:.1f}")

In [ ]:
# ============================================================
# 2단계: 마스터테이블 조인 + 기존값 비교
# ============================================================

# 홈팀 조인
hbp = bp_game[['gamePk','팀','bp_fatigue_1d_new','bp_fatigue_3d_new']].copy()
hbp.columns = ['gamePk','team','home_bp_1d_new','home_bp_3d_new']
master2 = master.merge(hbp, left_on=['gamePk','홈팀'], right_on=['gamePk','team'], how='left').drop(columns='team')

# 원정팀 조인
abp = bp_game[['gamePk','팀','bp_fatigue_1d_new','bp_fatigue_3d_new']].copy()
abp.columns = ['gamePk','team','away_bp_1d_new','away_bp_3d_new']
master2 = master2.merge(abp, left_on=['gamePk','원정팀'], right_on=['gamePk','team'], how='left').drop(columns='team')

# 기존값 vs 새값 비교
changed = 0
same = 0
for col_old, col_new in [
    ('home_bp_fatigue_1d','home_bp_1d_new'), ('home_bp_fatigue_3d','home_bp_3d_new'),
    ('away_bp_fatigue_1d','away_bp_1d_new'), ('away_bp_fatigue_3d','away_bp_3d_new')]:
    diff = (master2[col_old].fillna(0) - master2[col_new].fillna(0)).abs()
    n_diff = (diff >= 0.05).sum()
    changed += n_diff
    same += (diff < 0.05).sum()
    print(f"  {col_old}: {n_diff}건 변경, {(diff < 0.05).sum()}건 동일")

print(f"\n총: 변경 {changed}건, 동일 {same}건")
print(f"변경 비율: {changed/(changed+same)*100:.1f}%")

# 변경된 경기가 date≠날짜 103건과 겹치는지
mismatch_gpks = set(master2[master2['date'] != master2['날짜'].astype(str)]['gamePk'])
changed_gpks = set()
for col_old, col_new in [
    ('home_bp_fatigue_1d','home_bp_1d_new'), ('home_bp_fatigue_3d','home_bp_3d_new'),
    ('away_bp_fatigue_1d','away_bp_1d_new'), ('away_bp_fatigue_3d','away_bp_3d_new')]:
    diff = (master2[col_old].fillna(0) - master2[col_new].fillna(0)).abs()
    changed_gpks.update(master2[diff >= 0.05]['gamePk'].values)

overlap = changed_gpks & mismatch_gpks
only_changed = changed_gpks - mismatch_gpks
print(f"\n변경된 고유 경기: {len(changed_gpks)}개")
print(f"  date≠날짜 경기와 겹침: {len(overlap)}개")
print(f"  date=날짜인데 변경됨: {len(only_changed)}개")

In [ ]:
# ============================================================
# 3단계: 값 교체 + 저장
# ============================================================

# 기존 컬럼 교체
master2['home_bp_fatigue_1d'] = master2['home_bp_1d_new'].fillna(0)
master2['home_bp_fatigue_3d'] = master2['home_bp_3d_new'].fillna(0)
master2['away_bp_fatigue_1d'] = master2['away_bp_1d_new'].fillna(0)
master2['away_bp_fatigue_3d'] = master2['away_bp_3d_new'].fillna(0)

# 임시 컬럼 제거
master2 = master2.drop(columns=['home_bp_1d_new','home_bp_3d_new',
                                 'away_bp_1d_new','away_bp_3d_new'])

# 원본과 동일 컬럼 수 확인
print(f"원본: {master.shape}, 수정본: {master2.shape}")
assert master2.shape[1] == master.shape[1], f"컬럼 수 불일치! {master2.shape[1]} vs {master.shape[1]}"

# 수정 전후 통계 비교
print(f"\n=== bp_fatigue 통계 비교 ===")
for col in ['home_bp_fatigue_1d','home_bp_fatigue_3d','away_bp_fatigue_1d','away_bp_fatigue_3d']:
    print(f"  {col}: 기존 mean={master[col].mean():.3f} → 수정 mean={master2[col].mean():.3f}")

# 저장
master2.to_csv(f'{MLB_PATH}/B안_마스터테이블.csv', index=False)
print(f"\n저장 완료: B안_마스터테이블.csv ({master2.shape})")

In [ ]:
# ============================================================
# 3단계: master_5seasons.csv 재빌드
# ============================================================
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 2021~2022 (변경 없음)
df21 = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블_2021.csv')
df22 = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블_2022.csv')

# 2023~2025 (수정본)
df2325 = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

print(f"2021: {df21.shape}")
print(f"2022: {df22.shape}")
print(f"2023~25: {df2325.shape}")

# season 컬럼 확인/추가
if 'season' not in df21.columns:
    df21['season'] = 2021
if 'season' not in df22.columns:
    df22['season'] = 2022

# 공통 컬럼 추출 (기존 master_5seasons.csv와 동일 구조)
old_master = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')
target_cols = list(old_master.columns)
print(f"\n기존 master_5seasons 컬럼 수: {len(target_cols)}")
print(f"기존 행수: {len(old_master)}")

# 날짜 컬럼 통일 (2021~22는 'date', 2023~25도 'date' 사용)
if 'date' not in df21.columns and '날짜' in df21.columns:
    df21['date'] = df21['날짜']
if 'date' not in df22.columns and '날짜' in df22.columns:
    df22['date'] = df22['날짜']

# 파생피처 재생성 (수정된 bp_fatigue 반영)
frames = []
for label, df in [('2021', df21), ('2022', df22), ('2023~25', df2325)]:
    d = df.copy()
    # diff 피처
    for col in ['era','whip','k9','bb9','hr9']:
        d[f'{col}_diff'] = d[f'home_sp_{col}'] - d[f'away_sp_{col}']
    d['ops_diff'] = d['home_lineup_ops'] - d['away_lineup_ops']
    for col in ['wr5','wr10','rs5','rs10','ra5','ra10']:
        d[f'{col}_diff'] = d[f'home_{col}'] - d[f'away_{col}']
    d['bp_fatigue_1d_diff'] = d['home_bp_fatigue_1d'] - d['away_bp_fatigue_1d']
    d['bp_fatigue_3d_diff'] = d['home_bp_fatigue_3d'] - d['away_bp_fatigue_3d']
    d['il_diff'] = d['home_il'] - d['away_il']
    # sum 피처
    for col in ['era','whip','k9','bb9','hr9']:
        d[f'{col}_sum'] = d[f'home_sp_{col}'] + d[f'away_sp_{col}']
    d['ops_sum'] = d['home_lineup_ops'] + d['away_lineup_ops']
    for col in ['rs5','rs10','ra5','ra10']:
        d[f'{col}_sum'] = d[f'home_{col}'] + d[f'away_{col}']
    d['bp_fatigue_sum'] = d['home_bp_fatigue_3d'] + d['away_bp_fatigue_3d']
    # max/교차
    d['era_max'] = d[['home_sp_era','away_sp_era']].max(axis=1)
    d['whip_max'] = d[['home_sp_whip','away_sp_whip']].max(axis=1)
    d['era_vs_run_total'] = d['era_sum'] - d['run_total_baseline']
    d['ops_vs_run_total'] = d['ops_sum'] * 10 - d['run_total_baseline']
    # 타겟
    d['home_win'] = (d['홈팀득점'] > d['원정팀득점']).astype(int)
    d['total_runs'] = d['홈팀득점'] + d['원정팀득점']
    d['upper'] = np.where(d['total_runs'] == d['run_total_baseline'], np.nan,
                         (d['total_runs'] > d['run_total_baseline']).astype(float))
    frames.append(d)
    print(f"  {label}: {d.shape}")

new_master = pd.concat(frames, ignore_index=True)

# 기존 컬럼만 추출
missing_cols = [c for c in target_cols if c not in new_master.columns]
extra_cols = [c for c in new_master.columns if c not in target_cols]
print(f"\n기존에 있는데 새로 없는 컬럼: {missing_cols}")
print(f"새로 있는데 기존에 없는 컬럼: {len(extra_cols)}개")

new_master = new_master[[c for c in target_cols if c in new_master.columns]]
print(f"\n최종: {new_master.shape}")
print(f"시즌별: {new_master['season'].value_counts().sort_index().to_dict()}")

# bp_fatigue 변경 확인
for col in ['bp_fatigue_1d_diff','bp_fatigue_sum']:
    old_mean = old_master[col].mean()
    new_mean = new_master[col].mean()
    print(f"  {col}: 기존 {old_mean:.4f} → 수정 {new_mean:.4f} (diff {new_mean-old_mean:+.4f})")

# 저장
new_master.to_csv(f'{MLB_PATH}/master_5seasons.csv', index=False)
print(f"\nmaster_5seasons.csv 저장 완료: {new_master.shape}")

In [ ]:
# ============================================================
# 종결 모델 재학습 + 기존 대비 성능 비교
# ============================================================
import pandas as pd
import numpy as np
import pickle, json, warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
df = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')

WIN_FEATS = ['era_diff','k9_diff','bb9_diff','ops_diff','wr5_diff','wr10_diff',
             'rs10_diff','bp_fatigue_1d_diff','run_total_baseline','run_margin','home_WIN_LOSS',
             'park_HR','ump_avg_runs','ump_vs_league','temperature']
RUN_TOTAL_FEATS = ['era_sum','whip_sum','k9_sum','bb9_sum','hr9_sum','ops_sum',
            'rs5_sum','rs10_sum','ra5_sum','ra10_sum','bp_fatigue_sum',
            'park_factor','park_R','park_HR','park_H','temperature',
            'ump_avg_runs','ump_vs_league','era_max','whip_max',
            'run_margin','home_WIN_LOSS','run_total_baseline','era_vs_run_total','ops_vs_run_total']

FOLDS = [
    {'train':[2021],'test':[2022]},
    {'train':[2021,2022],'test':[2023]},
    {'train':[2021,2022,2023],'test':[2024]},
    {'train':[2021,2022,2023,2024],'test':[2025]},
]

# Walk-Forward 검증
def wf_validate(features, target, is_run_total=False, C=0.01, leaves=4, depth=2,
                min_child=150, lr_w=0.8):
    all_prob, all_y = [], []
    for fold in FOLDS:
        tr = df[df['season'].isin(fold['train'])]
        te = df[df['season'].isin(fold['test'])]
        if is_run_total:
            tr = tr[tr['upper'].notna()]
            te = te[te['upper'].notna()]
        X_tr, y_tr = tr[features], tr[target]
        X_te, y_te = te[features], te[target]

        lr_pipe = Pipeline([('imp',SimpleImputer(strategy='median')),
                            ('sc',StandardScaler()),
                            ('m',LogisticRegression(C=C,max_iter=1000,random_state=42))])
        lr_pipe.fit(X_tr, y_tr)

        lgb_m = lgb.LGBMClassifier(num_leaves=leaves,max_depth=depth,
                    min_child_samples=min_child,learning_rate=0.03,
                    n_estimators=100,reg_alpha=3.0,reg_lambda=3.0,
                    random_state=42,verbose=-1)
        lgb_m.fit(X_tr, y_tr)

        lr_p = lr_pipe.predict_proba(X_te)[:,1]
        lgb_p = lgb_m.predict_proba(X_te)[:,1]
        ens_p = lr_w * lr_p + (1-lr_w) * lgb_p
        all_prob.extend(ens_p)
        all_y.extend(y_te.values)

    all_prob = np.array(all_prob)
    all_y = np.array(all_y)
    return all_prob, all_y

# 승패
print("=== 승패 ===")
wp, wy = wf_validate(WIN_FEATS,'home_win',C=0.01,leaves=4,depth=2,min_child=150,lr_w=0.8)
for thr in [0.60, 0.64, 0.66, 0.68, 0.70]:
    conf = np.maximum(wp, 1-wp) >= thr
    pred = (wp >= 0.5).astype(int)
    if conf.sum() > 0:
        acc = (pred[conf] == wy[conf]).mean() * 100
        print(f"  {thr:.2f}: {acc:.1f}% ({conf.sum()}경기)")

# 총득점
print("\n=== 총득점 ===")
op, oy = wf_validate(RUN_TOTAL_FEATS,'upper',is_run_total=True,C=0.01,leaves=3,depth=2,min_child=100,lr_w=0.5)
for thr in [0.55, 0.58, 0.60, 0.62, 0.65]:
    conf = np.maximum(op, 1-op) >= thr
    pred = (op >= 0.5).astype(int)
    if conf.sum() > 0:
        acc = (pred[conf] == oy[conf]).mean() * 100
        print(f"  {thr:.2f}: {acc:.1f}% ({conf.sum()}경기)")

print("\n기존: 승패 0.64=74.1%(2684), 총득점 0.60=72.2%(2695)")

In [ ]:
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

df_new = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')
print(f"shape: {df_new.shape}")
print(f"컬럼: {list(df_new.columns)}")

# 핵심 피처 값 범위 체크
for col in ['era_diff','home_WIN_LOSS','run_total_baseline','run_margin','bp_fatigue_1d_diff','bp_fatigue_sum',
            'ump_avg_runs','park_HR','temperature','home_win','upper']:
    if col in df_new.columns:
        print(f"  {col}: mean={df_new[col].mean():.4f} std={df_new[col].std():.4f} NaN={df_new[col].isna().sum()}")
    else:
        print(f"  {col}: ❌ 컬럼 없음!")

In [ ]:
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
df = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')

for season in [2021, 2022, 2023, 2024, 2025]:
    s = df[df['season']==season]
    print(f"  {season}: park_HR mean={s['park_HR'].mean():.2f} "
          f"min={s['park_HR'].min():.2f} max={s['park_HR'].max():.2f}")

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
df = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')

# 2021~22 파크팩터 ×100
mask = df['season'].isin([2021, 2022])
for col in ['park_factor','park_R','park_HR','park_H']:
    before = df.loc[mask, col].mean()
    df.loc[mask, col] = df.loc[mask, col] * 100
    after = df.loc[mask, col].mean()
    print(f"  {col}: {before:.2f} → {after:.1f}")

# 파생 피처 중 park 관련 재계산은 불필요 (park_HR 직접 사용)
# 검증
for season in [2021,2022,2023,2024,2025]:
    s = df[df['season']==season]
    print(f"  {season}: park_HR={s['park_HR'].mean():.1f}")

df.to_csv(f'{MLB_PATH}/master_5seasons.csv', index=False)
print(f"\n저장 완료: {df.shape}")

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
df = pd.read_csv(f'{MLB_PATH}/master_5seasons.csv')

WIN_FEATS = ['era_diff','k9_diff','bb9_diff','ops_diff','wr5_diff','wr10_diff',
             'rs10_diff','bp_fatigue_1d_diff','run_total_baseline','run_margin','home_WIN_LOSS',
             'park_HR','ump_avg_runs','ump_vs_league','temperature']
RUN_TOTAL_FEATS = ['era_sum','whip_sum','k9_sum','bb9_sum','hr9_sum','ops_sum',
            'rs5_sum','rs10_sum','ra5_sum','ra10_sum','bp_fatigue_sum',
            'park_factor','park_R','park_HR','park_H','temperature',
            'ump_avg_runs','ump_vs_league','era_max','whip_max',
            'run_margin','home_WIN_LOSS','run_total_baseline','era_vs_run_total','ops_vs_run_total']

FOLDS = [
    {'train':[2021],'test':[2022]},
    {'train':[2021,2022],'test':[2023]},
    {'train':[2021,2022,2023],'test':[2024]},
    {'train':[2021,2022,2023,2024],'test':[2025]},
]

def wf_validate(features, target, is_run_total=False, C=0.01, leaves=4, depth=2,
                min_child=150, lr_w=0.8):
    all_prob, all_y = [], []
    for fold in FOLDS:
        tr = df[df['season'].isin(fold['train'])]
        te = df[df['season'].isin(fold['test'])]
        if is_run_total:
            tr, te = tr[tr['upper'].notna()], te[te['upper'].notna()]
        X_tr, y_tr = tr[features], tr[target]
        X_te, y_te = te[features], te[target]
        lr_pipe = Pipeline([('imp',SimpleImputer(strategy='median')),
                            ('sc',StandardScaler()),
                            ('m',LogisticRegression(C=C,max_iter=1000,random_state=42))])
        lr_pipe.fit(X_tr, y_tr)
        lgb_m = lgb.LGBMClassifier(num_leaves=leaves,max_depth=depth,
                    min_child_samples=min_child,learning_rate=0.03,
                    n_estimators=100,reg_alpha=3.0,reg_lambda=3.0,
                    random_state=42,verbose=-1)
        lgb_m.fit(X_tr, y_tr)
        ens_p = lr_w*lr_pipe.predict_proba(X_te)[:,1] + (1-lr_w)*lgb_m.predict_proba(X_te)[:,1]
        all_prob.extend(ens_p)
        all_y.extend(y_te.values)
    return np.array(all_prob), np.array(all_y)

print("=== 승패 (기존 74.1%) ===")
wp, wy = wf_validate(WIN_FEATS,'home_win')
for thr in [0.60,0.64,0.66,0.68,0.70]:
    conf = np.maximum(wp,1-wp)>=thr
    pred = (wp>=0.5).astype(int)
    if conf.sum()>0:
        acc = (pred[conf]==wy[conf]).mean()*100
        print(f"  {thr:.2f}: {acc:.1f}% ({conf.sum()}경기)")

print("\n=== 총득점 (기존 72.2%) ===")
op, oy = wf_validate(RUN_TOTAL_FEATS,'upper',is_run_total=True,C=0.01,leaves=3,depth=2,min_child=100,lr_w=0.5)
for thr in [0.55,0.58,0.60,0.62,0.65]:
    conf = np.maximum(op,1-op)>=thr
    pred = (op>=0.5).astype(int)
    if conf.sum()>0:
        acc = (pred[conf]==oy[conf]).mean()*100
        print(f"  {thr:.2f}: {acc:.1f}% ({conf.sum()}경기)")

In [ ]:
import pickle
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 전체 데이터로 최종 모델 학습
X_win = df[WIN_FEATS]
y_win = df['home_win']

win_lr = Pipeline([('imp',SimpleImputer(strategy='median')),
                   ('sc',StandardScaler()),
                   ('m',LogisticRegression(C=0.01,max_iter=1000,random_state=42))])
win_lr.fit(X_win, y_win)

win_lgb = lgb.LGBMClassifier(num_leaves=4,max_depth=2,min_child_samples=150,
              learning_rate=0.03,n_estimators=100,reg_alpha=3.0,reg_lambda=3.0,
              random_state=42,verbose=-1)
win_lgb.fit(X_win, y_win)

df_run_total = df[df['upper'].notna()]
X_run_total = df_run_total[RUN_TOTAL_FEATS]
y_run_total = df_run_total['upper']

run_total_lr = Pipeline([('imp',SimpleImputer(strategy='median')),
                  ('sc',StandardScaler()),
                  ('m',LogisticRegression(C=0.01,max_iter=1000,random_state=42))])
run_total_lr.fit(X_run_total, y_run_total)

run_total_lgb = lgb.LGBMClassifier(num_leaves=3,max_depth=2,min_child_samples=100,
              learning_rate=0.03,n_estimators=100,reg_alpha=3.0,reg_lambda=3.0,
              random_state=42,verbose=-1)
run_total_lgb.fit(X_run_total, y_run_total)

# 저장
for name, model in [('종결_승패_LR',win_lr),('종결_승패_LGB',win_lgb),
                     ('종결_총득점_LR',run_total_lr),('종결_총득점_LGB',run_total_lgb)]:
    with open(f'{MLB_PATH}/{name}.pkl','wb') as f:
        pickle.dump(model, f)
    print(f"  ✅ {name}.pkl 저장")

print(f"\n완료! 승패 74.1% / 총득점 72.3% (bp_fatigue 수정본)")

In [ ]:
import os
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

for yr in [2023, 2024, 2025]:
    path = f'{MLB_PATH}/{yr}'
    files = [f for f in os.listdir(path) if '심판' in f or 'ump' in f.lower()]
    print(f"{yr}: {files}")

In [ ]:
import os
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 심판 관련 파일 전체 검색
for root, dirs, files in os.walk(MLB_PATH):
    for f in files:
        if '심판' in f or 'ump' in f.lower() or 'official' in f.lower():
            fpath = os.path.join(root, f)
            size = os.path.getsize(fpath)/1024
            print(f"  {fpath} ({size:.0f}KB)")

In [ ]:
import pandas as pd, os
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 1) starters CSV에 심판 정보 있는지
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2023.csv')
ump_cols = [c for c in st.columns if '심판' in c or 'ump' in c.lower() or 'official' in c.lower()]
print(f"starters 컬럼 중 심판 관련: {ump_cols}")
print(f"starters 전체 컬럼: {list(st.columns)}")

# 2) 백테스트_라인업 폴더 확인
for yr in [2023, 2024, 2025]:
    bt_path = f'{MLB_PATH}/{yr}/백테스트_라인업'
    if os.path.exists(bt_path):
        files = os.listdir(bt_path)
        ump_files = [f for f in files if '심판' in f or 'ump' in f.lower()]
        print(f"\n{yr} 백테스트_라인업: {len(files)}파일, 심판관련: {ump_files}")

# 3) 마스터테이블 심판명 소스 추적 — 심판명이 어디서 왔는지
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
print(f"\n심판명 고유값: {master['심판명'].nunique()}명")
print(f"심판명 샘플: {master['심판명'].head(5).tolist()}")

In [ ]:
import os
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

for yr in [2023, 2024, 2025]:
    path = f'{MLB_PATH}/{yr}'
    files = [f for f in os.listdir(path) if '심판' in f]
    print(f"{yr}: {files}")

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 심판 CSV 구조 확인
for yr in [2023, 2024, 2025]:
    ump = pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv')
    print(f"\n=== {yr}: {ump.shape} ===")
    print(f"  컬럼: {list(ump.columns)}")
    if '포지션' in ump.columns:
        print(f"  포지션: {ump['포지션'].value_counts().head().to_dict()}")
    print(ump.head(2).to_string())

# 교차누적 확인: 2024 첫 경기 심판의 이전 경기 추적
first_2024 = master[master['시즌']==2024].sort_values('날짜')
first_with_ump = first_2024[first_2024['ump_avg_runs'].notna()].iloc[0]
ump_name = first_with_ump['심판명']
val = first_with_ump['ump_avg_runs']
date = first_with_ump['날짜']

prev = master[(master['심판명']==ump_name) & (master['날짜'].astype(str) < str(date))].sort_values('날짜')
runs = prev['총득점'].values
print(f"\n=== 교차누적 검증 ===")
print(f"  심판: {ump_name}, 2024 첫경기({date}): ump_avg_runs={val:.3f}")
print(f"  이전 경기: {len(runs)}개, 평균 총득점: {runs.mean():.3f}")
print(f"  일치: {'✅ 교차누적' if abs(runs.mean() - val) < 0.01 else '❌'}")

In [ ]:
# ============================================================
# 심판 함수 만들기 + 검증
# ============================================================
import pandas as pd
import numpy as np

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 심판 CSV 3년 합치기 (Home Plate만)
ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv')
    for yr in [2023, 2024, 2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate'][['gamePk','날짜','심판명','심판ID']].copy()
ump_hp['날짜'] = pd.to_datetime(ump_hp['날짜'])
ump_hp = ump_hp.sort_values('날짜').reset_index(drop=True)
print(f"Home Plate 심판: {len(ump_hp)}경기, 고유심판: {ump_hp['심판명'].nunique()}명")

# 경기결과에서 총득점 가져오기 (날짜 기준 = 실제경기일)
game_runs = master[['gamePk','총득점']].drop_duplicates()
ump_hp = ump_hp.merge(game_runs, on='gamePk', how='left')
print(f"총득점 매칭: {ump_hp['총득점'].notna().sum()}/{len(ump_hp)}")

# === 심판 함수 ===
def calc_ump_stats(ump_df, min_periods=5):
    """
    심판별 expanding 평균 총득점 계산 (교차시즌 누적)
    리턴: {gamePk: {'ump_avg_runs': float, 'ump_vs_league': float}}
    """
    ump_sorted = ump_df.sort_values('날짜').reset_index(drop=True)

    # 전체 경기 expanding 리그 평균 (심판 무관)
    all_runs = []

    # 심판별 히스토리
    ump_history = {}  # 심판명 → [총득점, ...]
    result = {}

    for _, row in ump_sorted.iterrows():
        gpk = row['gamePk']
        ump = row['심판명']
        runs = row['총득점']

        if pd.isna(runs):
            continue

        history = ump_history.get(ump, [])

        if len(history) >= min_periods:
            avg_runs = np.mean(history)
            league_avg = np.mean(all_runs) if len(all_runs) > 0 else avg_runs
            result[gpk] = {
                'ump_avg_runs_calc': avg_runs,
                'ump_vs_league_calc': avg_runs - league_avg
            }
        else:
            result[gpk] = {
                'ump_avg_runs_calc': np.nan,
                'ump_vs_league_calc': np.nan
            }

        # 현재 경기 히스토리에 추가
        ump_history.setdefault(ump, []).append(runs)
        all_runs.append(runs)

    return result

ump_result = calc_ump_stats(ump_hp)
print(f"계산 완료: {len(ump_result)}경기")

# === 검증: 마스터테이블과 비교 ===
errors = 0
checks = 0
for _, row in master.iterrows():
    gpk = row['gamePk']
    calc = ump_result.get(gpk, {})
    calc_avg = calc.get('ump_avg_runs_calc', np.nan)
    ref_avg = row['ump_avg_runs']

    if pd.isna(calc_avg) and pd.isna(ref_avg):
        continue
    if pd.isna(calc_avg) or pd.isna(ref_avg):
        checks += 1
        errors += 1
        continue

    checks += 1
    if abs(calc_avg - ref_avg) >= 0.05:
        errors += 1

print(f"\n검증: {checks}건 중 오류 {errors}건 ({errors/checks*100:.1f}%)")
if errors == 0:
    print("✅ 완전 일치!")
else:
    # 오류 샘플
    print(f"\n오류 샘플 (상위 5):")
    err_cnt = 0
    for _, row in master.iterrows():
        if err_cnt >= 5: break
        gpk = row['gamePk']
        calc = ump_result.get(gpk, {})
        calc_avg = calc.get('ump_avg_runs_calc', np.nan)
        ref_avg = row['ump_avg_runs']
        if pd.notna(calc_avg) and pd.notna(ref_avg) and abs(calc_avg - ref_avg) >= 0.05:
            print(f"  gpk={gpk} 날짜={row['날짜']} 심판={row['심판명']} "
                  f"함수={calc_avg:.3f} 마스터={ref_avg:.3f} diff={calc_avg-ref_avg:.3f}")
            err_cnt += 1

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 심판 CSV에서 마스터에 없는 경기(시범경기) 확인
ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate'][['gamePk','날짜','심판명']].copy()

master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
master_gpks = set(master['gamePk'])

# 마스터에 없는 경기
missing = ump_hp[~ump_hp['gamePk'].isin(master_gpks)]
print(f"마스터에 없는 심판경기: {len(missing)}건")
print(f"날짜 범위: {missing['날짜'].min()} ~ {missing['날짜'].max()}")

# 시범경기 총득점 — 경기결과 CSV에서 가져올 수 있는지
for yr in [2023,2024,2025]:
    for prefix in ['시범경기','정규시즌']:
        import os
        files = [f for f in os.listdir(f'{MLB_PATH}/{yr}') if '경기결과' in f]
        for f in files:
            df = pd.read_csv(f'{MLB_PATH}/{yr}/{f}')
            matched = df['gamePk'].isin(missing['gamePk']).sum()
            if matched > 0:
                print(f"  {yr}/{f}: {matched}건 매칭")

In [ ]:
import os, pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate']
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
missing_gpks = set(ump_hp['gamePk']) - set(master['gamePk'])

# 모든 경기결과 CSV에서 missing 경기 찾기
found = 0
for yr in [2023,2024,2025]:
    path = f'{MLB_PATH}/{yr}'
    for f in sorted(os.listdir(path)):
        if '경기결과' in f and f.endswith('.csv'):
            df = pd.read_csv(f'{path}/{f}')
            matched = df[df['gamePk'].isin(missing_gpks)]
            if len(matched) > 0:
                has_runs = '총득점' in matched.columns or '홈팀득점' in matched.columns
                print(f"  {yr}/{f}: {len(matched)}건 매칭, 득점컬럼={has_runs}")
                found += len(matched)

print(f"\n총 {found}/{len(missing_gpks)}건 찾음")

In [ ]:
import os
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

for yr in [2023, 2024, 2025]:
    path = f'{MLB_PATH}/{yr}'
    print(f"\n=== {yr} ({len(os.listdir(path))}파일) ===")
    for f in sorted(os.listdir(path)):
        size = os.path.getsize(f'{path}/{f}')/1024
        print(f"  {f} ({size:.0f}KB)")

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'

# 시범+정규 경기결과 합치기 (총득점만 필요)
all_runs = []
for yr in [2023,2024,2025]:
    for prefix in ['시범경기','정규시즌']:
        df = pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_{prefix}_경기결과.csv')
        if '총득점' not in df.columns and '홈팀득점' in df.columns:
            df['총득점'] = df['홈팀득점'] + df['원정팀득점']
        all_runs.append(df[['gamePk','총득점']].drop_duplicates())
runs_map = pd.concat(all_runs, ignore_index=True).drop_duplicates(subset='gamePk')
print(f"전체 경기결과: {len(runs_map)}건")

# 심판 CSV (Home Plate)
ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate'][['gamePk','날짜','심판명','심판ID']].copy()
ump_hp['날짜'] = pd.to_datetime(ump_hp['날짜'])
ump_hp = ump_hp.merge(runs_map, on='gamePk', how='left')
ump_hp = ump_hp.sort_values('날짜').reset_index(drop=True)
print(f"심판 HP: {len(ump_hp)}건, 총득점 매칭: {ump_hp['총득점'].notna().sum()}")

# 심판 함수 (시범경기 포함)
def calc_ump_stats(ump_df, min_periods=5):
    ump_sorted = ump_df.sort_values('날짜').reset_index(drop=True)
    all_runs = []
    ump_history = {}
    result = {}
    for _, row in ump_sorted.iterrows():
        gpk = row['gamePk']
        ump = row['심판명']
        runs = row['총득점']
        if pd.isna(runs): continue
        history = ump_history.get(ump, [])
        if len(history) >= min_periods:
            avg_runs = np.mean(history)
            league_avg = np.mean(all_runs) if all_runs else avg_runs
            result[gpk] = {'ump_avg_runs_calc': avg_runs, 'ump_vs_league_calc': avg_runs - league_avg}
        else:
            result[gpk] = {'ump_avg_runs_calc': np.nan, 'ump_vs_league_calc': np.nan}
        ump_history.setdefault(ump, []).append(runs)
        all_runs.append(runs)
    return result

ump_result = calc_ump_stats(ump_hp)

# 검증
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
errors = 0
checks = 0
for _, row in master.iterrows():
    calc = ump_result.get(row['gamePk'], {})
    c = calc.get('ump_avg_runs_calc', np.nan)
    r = row['ump_avg_runs']
    if pd.isna(c) and pd.isna(r): continue
    checks += 1
    if pd.isna(c) or pd.isna(r) or abs(c - r) >= 0.05:
        errors += 1

print(f"\n검증: {checks}건 중 오류 {errors}건 ({errors/checks*100:.1f}%)")
if errors == 0:
    print("✅ 완전 일치!")

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 정규시즌만 — 심판 CSV에서 마스터에 있는 gamePk만
ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate'][['gamePk','날짜','심판명']].copy()
ump_hp['날짜'] = pd.to_datetime(ump_hp['날짜'])

# 마스터 gamePk만 필터 (정규시즌만)
master_gpks = set(master['gamePk'])
ump_hp = ump_hp[ump_hp['gamePk'].isin(master_gpks)].copy()

# 총득점 붙이기
game_runs = master[['gamePk','총득점']].drop_duplicates()
ump_hp = ump_hp.merge(game_runs, on='gamePk', how='left')
ump_hp = ump_hp.sort_values(['날짜','gamePk']).reset_index(drop=True)
print(f"정규시즌 HP 심판: {len(ump_hp)}건")

# 오류 146건 원인 분석 — date vs 날짜 문제인지
# 심판 CSV의 날짜 vs 마스터의 날짜/date 비교
ump_dates = ump_hp[['gamePk','날짜']].rename(columns={'날짜':'ump_date'})
master_dates = master[['gamePk','date','날짜']].copy()
master_dates['date'] = pd.to_datetime(master_dates['date'])
master_dates['날짜'] = pd.to_datetime(master_dates['날짜'])

comp = ump_dates.merge(master_dates, on='gamePk')
diff_date = (comp['ump_date'] != comp['date']).sum()
diff_nalzza = (comp['ump_date'] != comp['날짜']).sum()
print(f"\n심판CSV날짜 vs master.date 불일치: {diff_date}건")
print(f"심판CSV날짜 vs master.날짜 불일치: {diff_nalzza}건")

# 심판CSV는 어느 날짜 기준인지 확인
sample = comp[comp['ump_date'] != comp['날짜']].head(5)
if len(sample) > 0:
    print(f"\n심판CSV ≠ 날짜 샘플:")
    for _, r in sample.iterrows():
        print(f"  gpk={r['gamePk']} 심판CSV={r['ump_date'].strftime('%Y-%m-%d')} "
              f"date={r['date'].strftime('%Y-%m-%d')} 날짜={r['날짜'].strftime('%Y-%m-%d')}")

In [ ]:
import pandas as pd
import numpy as np
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')

# 심판 + date(예정일) 기준으로 계산
ump_all = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_hp = ump_all[ump_all['포지션']=='Home Plate'][['gamePk','심판명']].copy()

# master의 date(예정일) + 총득점 붙이기
master_info = master[['gamePk','date','총득점']].drop_duplicates()
ump_hp = ump_hp.merge(master_info, on='gamePk', how='inner')
ump_hp['date'] = pd.to_datetime(ump_hp['date'])
ump_hp = ump_hp.sort_values(['date','gamePk']).reset_index(drop=True)
print(f"date(예정일) 기준 심판: {len(ump_hp)}건")

# date 기준 expanding
all_runs = []
ump_history = {}
result_date = {}
for _, row in ump_hp.iterrows():
    gpk, ump, runs = row['gamePk'], row['심판명'], row['총득점']
    if pd.isna(runs): continue
    history = ump_history.get(ump, [])
    if len(history) >= 5:
        result_date[gpk] = {'avg': np.mean(history), 'vs': np.mean(history) - np.mean(all_runs)}
    else:
        result_date[gpk] = {'avg': np.nan, 'vs': np.nan}
    ump_history.setdefault(ump, []).append(runs)
    all_runs.append(runs)

# 검증
errors_date = 0
checks = 0
for _, row in master.iterrows():
    c = result_date.get(row['gamePk'], {}).get('avg', np.nan)
    r = row['ump_avg_runs']
    if pd.isna(c) and pd.isna(r): continue
    checks += 1
    if pd.isna(c) or pd.isna(r) or abs(c - r) >= 0.05:
        errors_date += 1

print(f"date(예정일) 기준: {checks}건 중 오류 {errors_date}건 ({errors_date/checks*100:.1f}%)")

In [ ]:
# ============================================================
# daily_predictor.py — STEP 1: 데이터 로드 + rolling 피처
# ============================================================
import pandas as pd
import numpy as np
import pickle, json, warnings, os
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
DATA_2026 = f'{MLB_PATH}/2026'

# ============================================================
# 1. today CSV 로드
# ============================================================
# 날짜 설정 (미국 기준)
TARGET_DATE = '2026-04-01'  # ← 매일 바꿔주면 됨

today_path = f'{DATA_2026}/today_{TARGET_DATE}.csv'
today = pd.read_csv(today_path)
print(f"오늘 경기: {len(today)}개")
print(today[['away_team','home_team','home_sp_name','away_sp_name']].to_string(index=False))

# ============================================================
# 2. 히스토리 데이터 로드
# ============================================================
results = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_경기결과.csv')
pit = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_투수개인기록.csv')
print(f"\n히스토리: 경기결과 {len(results)}건, 투수기록 {len(pit)}건")
print(f"경기 날짜 범위: {results['날짜'].min()} ~ {results['날짜'].max()}")

# 심판 CSV 확인
ump_path = f'{DATA_2026}/mlb_2026_심판.csv'
if os.path.exists(ump_path):
    ump_csv = pd.read_csv(ump_path)
    ump_hp = ump_csv[ump_csv['포지션']=='Home Plate']
    print(f"심판: {len(ump_hp)}건 (HP)")
else:
    print("⚠️ 심판 CSV 없음 — ump_avg_runs는 NaN 처리")
    ump_hp = pd.DataFrame()

# starters (선발투수 이름 — 불펜 계산용)
st_path = f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2026.csv'
if os.path.exists(st_path):
    starters = pd.read_csv(st_path)
    print(f"starters: {len(starters)}건")
else:
    print("⚠️ starters CSV 없음")
    starters = pd.DataFrame()

# ============================================================
# 3. 최근폼 계산 (wr5/10, rs5/10, ra5/10)
# ============================================================
def calc_recent_form(results_df, target_date):
    prev = results_df[results_df['날짜'] < target_date].sort_values('날짜')
    form = {}
    teams = set(prev['홈팀'].unique()) | set(prev['원정팀'].unique())
    for team in teams:
        home_mask = prev['홈팀'] == team
        away_mask = prev['원정팀'] == team
        team_games = []
        for _, row in prev[home_mask | away_mask].iterrows():
            if row['홈팀'] == team:
                win = 1 if row['홈팀결과'] == '승' else 0
                rs, ra = row['홈팀득점'], row['원정팀득점']
            else:
                win = 1 if row['원정팀결과'] == '승' else 0
                rs, ra = row['원정팀득점'], row['홈팀득점']
            team_games.append((win, rs, ra))
        form[team] = {}
        for window, suffix in [(5,'5'),(10,'10')]:
            recent = team_games[-window:]
            if len(recent) >= 3:
                form[team][f'wr{suffix}'] = np.mean([g[0] for g in recent])
                form[team][f'rs{suffix}'] = np.mean([g[1] for g in recent])
                form[team][f'ra{suffix}'] = np.mean([g[2] for g in recent])
            else:
                form[team][f'wr{suffix}'] = np.nan
                form[team][f'rs{suffix}'] = np.nan
                form[team][f'ra{suffix}'] = np.nan
    return form

form = calc_recent_form(results, TARGET_DATE)

for _, row in today.iterrows():
    h, a = row['home_team'], row['away_team']
    hf = form.get(h, {})
    af = form.get(a, {})
    for key in ['wr5','wr10','rs5','rs10','ra5','ra10']:
        today.loc[today['gamePk']==row['gamePk'], f'home_{key}'] = hf.get(key, np.nan)
        today.loc[today['gamePk']==row['gamePk'], f'away_{key}'] = af.get(key, np.nan)

print(f"\n최근폼: home_wr5 있음 {today['home_wr5'].notna().sum()}/{len(today)}")

# ============================================================
# 4. 불펜소모도 계산 (bp_fatigue_1d/3d)
# ============================================================
TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

if len(starters) > 0 and len(pit) > 0:
    starter_set = set()
    for _, r in starters.iterrows():
        starter_set.add((r['gamePk'], r.get('home_sp_name','')))
        starter_set.add((r['gamePk'], r.get('away_sp_name','')))

    pit['is_starter'] = pit.apply(lambda r: (r['gamePk'], r['선수명']) in starter_set, axis=1)
    bp = pit[~pit['is_starter']].copy()

    bp_game = bp.groupby(['gamePk','팀']).agg(bp_ip=('이닝','sum')).reset_index()
    bp_game['팀'] = bp_game['팀'].apply(normalize_team)

    date_map = results[['gamePk','날짜']].drop_duplicates()
    bp_game = bp_game.merge(date_map, on='gamePk', how='left')
    bp_game['날짜'] = pd.to_datetime(bp_game['날짜'])

    target_dt = pd.to_datetime(TARGET_DATE)
    d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')

    for _, row in today.iterrows():
        for team_col, prefix in [('home_team','home'),('away_team','away')]:
            team = row[team_col]
            team_bp = bp_game[(bp_game['팀']==team) & (bp_game['날짜']<target_dt)]
            s1d = team_bp[team_bp['날짜'] >= d1]['bp_ip'].sum()
            s3d = team_bp[team_bp['날짜'] >= d3]['bp_ip'].sum()
            today.loc[today['gamePk']==row['gamePk'], f'{prefix}_bp_fatigue_1d'] = s1d
            today.loc[today['gamePk']==row['gamePk'], f'{prefix}_bp_fatigue_3d'] = s3d

    print(f"불펜소모도: 완료")
else:
    today['home_bp_fatigue_1d'] = 0.0
    today['home_bp_fatigue_3d'] = 0.0
    today['away_bp_fatigue_1d'] = 0.0
    today['away_bp_fatigue_3d'] = 0.0
    print("불펜소모도: starters/투수기록 없어서 0으로 설정")

# ============================================================
# 5. 심판 (ump_avg_runs, ump_vs_league)
# ============================================================
if len(ump_hp) > 0:
    # 정규시즌 경기만 (히스토리에 있는 gamePk)
    reg_gpks = set(results['gamePk'])
    ump_reg = ump_hp[ump_hp['gamePk'].isin(reg_gpks)].copy()
    ump_reg = ump_reg.merge(results[['gamePk','총득점','날짜']].drop_duplicates(), on='gamePk', how='left')
    ump_reg = ump_reg.sort_values('날짜')

    # 심판별 expanding
    ump_history = {}
    all_runs = []
    ump_stats = {}  # 심판명 → (avg, vs_league)
    for _, r in ump_reg.iterrows():
        ump_name = r['심판명']
        runs = r['총득점']
        if pd.isna(runs): continue
        ump_history.setdefault(ump_name, []).append(runs)
        all_runs.append(runs)

    league_avg = np.mean(all_runs) if all_runs else 9.0
    for ump_name, hist in ump_history.items():
        if len(hist) >= 5:
            ump_stats[ump_name] = (np.mean(hist), np.mean(hist) - league_avg)

    for _, row in today.iterrows():
        ump_name = row.get('umpire_name', '')
        if ump_name in ump_stats:
            today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = ump_stats[ump_name][0]
            today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = ump_stats[ump_name][1]
        else:
            today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = np.nan
            today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = np.nan

    print(f"심판: ump_avg_runs 있음 {today['ump_avg_runs'].notna().sum()}/{len(today)}")
else:
    today['ump_avg_runs'] = np.nan
    today['ump_vs_league'] = np.nan
    print("심판: CSV 없어서 NaN")

# ============================================================
# 현황 출력
# ============================================================
print(f"\n{'='*50}")
print(f"rolling 피처 현황 ({TARGET_DATE})")
print(f"{'='*50}")
for col in ['home_wr5','home_wr10','home_bp_fatigue_1d','home_bp_fatigue_3d','ump_avg_runs']:
    n = today[col].notna().sum() if col in today.columns else 0
    print(f"  {col}: {n}/{len(today)}")

In [ ]:
# 심판 fix
if len(ump_hp) > 0:
    reg_gpks = set(results['gamePk'])
    ump_reg = ump_hp[ump_hp['gamePk'].isin(reg_gpks)].copy()

    # 컬럼명 충돌 방지
    runs_df = results[['gamePk','총득점','날짜']].drop_duplicates()
    runs_df.columns = ['gamePk','총득점','game_date']
    ump_reg = ump_reg.merge(runs_df, on='gamePk', how='left')
    ump_reg = ump_reg.sort_values('game_date')

    print(f"심판 컬럼: {list(ump_reg.columns)}")

    ump_history = {}
    all_runs = []
    ump_stats = {}
    for _, r in ump_reg.iterrows():
        ump_name = r['심판명']
        runs = r['총득점']
        if pd.isna(runs): continue
        ump_history.setdefault(ump_name, []).append(runs)
        all_runs.append(runs)

    league_avg = np.mean(all_runs) if all_runs else 9.0
    for ump_name, hist in ump_history.items():
        if len(hist) >= 5:
            ump_stats[ump_name] = (np.mean(hist), np.mean(hist) - league_avg)

    for _, row in today.iterrows():
        ump_name = row.get('umpire_name', '')
        if ump_name in ump_stats:
            today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = ump_stats[ump_name][0]
            today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = ump_stats[ump_name][1]
        else:
            today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = np.nan
            today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = np.nan

    print(f"심판: ump_avg_runs 있음 {today['ump_avg_runs'].notna().sum()}/{len(today)}")
    print(f"  히스토리 심판 {len(ump_stats)}명 (5경기+)")
else:
    today['ump_avg_runs'] = np.nan
    today['ump_vs_league'] = np.nan

# 현황 출력
print(f"\n{'='*50}")
print(f"rolling 피처 현황 ({TARGET_DATE})")
print(f"{'='*50}")
for col in ['home_wr5','home_wr10','home_bp_fatigue_1d','home_bp_fatigue_3d','ump_avg_runs']:
    n = today[col].notna().sum() if col in today.columns else 0
    print(f"  {col}: {n}/{len(today)}")

In [ ]:
# 심판 컬럼명 확인
print(f"ump_hp 컬럼: {list(ump_hp.columns)}")
print(f"results 컬럼: {list(results.columns)}")

In [ ]:
ump_reg = ump_hp[ump_hp['gamePk'].isin(set(results['gamePk']))].copy()
ump_reg = ump_reg.merge(results[['gamePk','총득점']], on='gamePk', how='left')
ump_reg = ump_reg.sort_values('날짜')
print(f"ump_reg: {len(ump_reg)}건, 컬럼: {list(ump_reg.columns)}")

In [ ]:
ump_history = {}
all_runs = []
for _, r in ump_reg.iterrows():
    ump_history.setdefault(r['심판명'], []).append(r['총득점'])
    all_runs.append(r['총득점'])

league_avg = np.mean(all_runs)
ump_stats = {}
for name, hist in ump_history.items():
    if len(hist) >= 5:
        ump_stats[name] = (np.mean(hist), np.mean(hist) - league_avg)

print(f"리그평균: {league_avg:.2f}")
print(f"5경기+ 심판: {len(ump_stats)}명")
print(f"오늘 경기 심판 매칭:")
for _, row in today.iterrows():
    ump = row.get('umpire_name','')
    matched = '✅' if ump in ump_stats else '❌ (NaN)'
    print(f"  {ump:20s} {matched}")

In [ ]:
# 2023~2025 + 2026 심판 히스토리 통합
ump_prev = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_prev = ump_prev[ump_prev['포지션']=='Home Plate']

# 2023~2025 정규시즌 총득점
master = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
prev_runs = master[['gamePk','총득점']].drop_duplicates()
ump_prev = ump_prev.merge(prev_runs, on='gamePk', how='inner')

# 2026 합치기
ump_2026 = ump_reg[['gamePk','날짜','심판명','총득점']].copy()
ump_prev_slim = ump_prev[['gamePk','날짜','심판명','총득점']].copy()
ump_all = pd.concat([ump_prev_slim, ump_2026], ignore_index=True)
ump_all['날짜'] = pd.to_datetime(ump_all['날짜'])
ump_all = ump_all.sort_values('날짜')

print(f"심판 히스토리: {len(ump_all)}건 (23~26)")

# 재계산
ump_history = {}
all_runs = []
for _, r in ump_all.iterrows():
    ump_history.setdefault(r['심판명'], []).append(r['총득점'])
    all_runs.append(r['총득점'])

league_avg = np.mean(all_runs)
ump_stats = {}
for name, hist in ump_history.items():
    if len(hist) >= 5:
        ump_stats[name] = (np.mean(hist), np.mean(hist) - league_avg)

print(f"5경기+ 심판: {len(ump_stats)}명")
matched = sum(1 for _, r in today.iterrows() if r.get('umpire_name','') in ump_stats)
print(f"오늘 매칭: {matched}/{len(today)}")

In [ ]:
for _, row in today.iterrows():
    ump = row.get('umpire_name','')
    if ump in ump_stats:
        today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = ump_stats[ump][0]
        today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = ump_stats[ump][1]
    else:
        today.loc[today['gamePk']==row['gamePk'], 'ump_avg_runs'] = np.nan
        today.loc[today['gamePk']==row['gamePk'], 'ump_vs_league'] = np.nan

print(f"ump_avg_runs 있음: {today['ump_avg_runs'].notna().sum()}/{len(today)}")

# 전체 rolling 피처 현황
print(f"\n{'='*50}")
print(f"rolling 피처 현황 ({TARGET_DATE})")
print(f"{'='*50}")
for col in ['home_wr5','home_wr10','home_rs5','home_rs10',
            'home_bp_fatigue_1d','home_bp_fatigue_3d','ump_avg_runs','ump_vs_league']:
    n = today[col].notna().sum() if col in today.columns else 0
    print(f"  {col}: {n}/{len(today)}")

In [ ]:
# 파생 피처 생성
today['era_diff'] = today['home_sp_era'] - today['away_sp_era']
today['k9_diff'] = today['home_sp_k9'] - today['away_sp_k9']
today['bb9_diff'] = today['home_sp_bb9'] - today['away_sp_bb9']
today['ops_diff'] = today['home_lineup_ops'] - today['away_lineup_ops']
today['wr5_diff'] = today['home_wr5'] - today['away_wr5']
today['wr10_diff'] = today['home_wr10'] - today['away_wr10']
today['rs10_diff'] = today['home_rs10'] - today['away_rs10']
today['bp_fatigue_1d_diff'] = today['home_bp_fatigue_1d'] - today['away_bp_fatigue_1d']

# 총득점용
today['era_sum'] = today['home_sp_era'] + today['away_sp_era']
today['whip_sum'] = today['home_sp_whip'] + today['away_sp_whip']
today['k9_sum'] = today['home_sp_k9'] + today['away_sp_k9']
today['bb9_sum'] = today['home_sp_bb9'] + today['away_sp_bb9']
today['hr9_sum'] = today['home_sp_hr9'] + today['away_sp_hr9']
today['ops_sum'] = today['home_lineup_ops'] + today['away_lineup_ops']
today['rs5_sum'] = today['home_rs5'] + today['away_rs5']
today['rs10_sum'] = today['home_rs10'] + today['away_rs10']
today['ra5_sum'] = today['home_ra5'] + today['away_ra5']
today['ra10_sum'] = today['home_ra10'] + today['away_ra10']
today['bp_fatigue_sum'] = today['home_bp_fatigue_3d'] + today['away_bp_fatigue_3d']
today['era_max'] = today[['home_sp_era','away_sp_era']].max(axis=1)
today['whip_max'] = today[['home_sp_whip','away_sp_whip']].max(axis=1)
today['era_vs_run_total'] = today['era_sum'] - today['run_total_baseline']
today['ops_vs_run_total'] = today['ops_sum'] * 10 - today['run_total_baseline']

nan_check = today[['era_diff','ops_diff','era_sum','run_total_baseline','era_vs_run_total']].isna().sum()
print(f"파생 피처 NaN:\n{nan_check[nan_check>0]}")
if nan_check.sum() == 0:
    print("✅ 파생 피처 NaN 없음")

In [ ]:
import pickle

# pkl 로드
win_lr = pickle.load(open(f'{MLB_PATH}/종결_승패_LR.pkl','rb'))
win_lgb = pickle.load(open(f'{MLB_PATH}/종결_승패_LGB.pkl','rb'))
run_total_lr = pickle.load(open(f'{MLB_PATH}/종결_총득점_LR.pkl','rb'))
run_total_lgb = pickle.load(open(f'{MLB_PATH}/종결_총득점_LGB.pkl','rb'))

WIN_FEATS = ['era_diff','k9_diff','bb9_diff','ops_diff','wr5_diff','wr10_diff',
             'rs10_diff','bp_fatigue_1d_diff','run_total_baseline','run_margin','home_WIN_LOSS',
             'park_HR','ump_avg_runs','ump_vs_league','temperature']
RUN_TOTAL_FEATS = ['era_sum','whip_sum','k9_sum','bb9_sum','hr9_sum','ops_sum',
            'rs5_sum','rs10_sum','ra5_sum','ra10_sum','bp_fatigue_sum',
            'park_factor','park_R','park_HR','park_H','temperature',
            'ump_avg_runs','ump_vs_league','era_max','whip_max',
            'run_margin','home_WIN_LOSS','run_total_baseline','era_vs_run_total','ops_vs_run_total']

# 승패 예측
wp_lr = win_lr.predict_proba(today[WIN_FEATS])[:,1]
wp_lgb = win_lgb.predict_proba(today[WIN_FEATS])[:,1]
today['win_prob'] = 0.8 * wp_lr + 0.2 * wp_lgb
today['win_conf'] = np.maximum(today['win_prob'], 1 - today['win_prob'])

# 총득점 예측
op_lr = run_total_lr.predict_proba(today[RUN_TOTAL_FEATS])[:,1]
op_lgb = run_total_lgb.predict_proba(today[RUN_TOTAL_FEATS])[:,1]
today['run_total_prob'] = 0.5 * op_lr + 0.5 * op_lgb
today['run_total_conf'] = np.maximum(today['run_total_prob'], 1 - today['run_total_prob'])

# 결과 출력
print(f"{'='*70}")
print(f"  MLB 예측 — {TARGET_DATE} (승패 0.64 / 총득점 0.60)")
print(f"{'='*70}")

for _, g in today.iterrows():
    win_prediction = g['home_team'] if g['win_prob'] >= 0.5 else g['away_team']
    run_total_prediction = 'UPPER' if g['run_total_prob'] >= 0.5 else 'LOWER'
    win_selection = '선택' if g['win_conf'] >= 0.64 else ''
    run_total_selection = '선택' if g['run_total_conf'] >= 0.60 else ''

    print(f"\n  {g['away_team']} @ {g['home_team']}")
    print(f"    승패: {win_prediction} ({g['win_prob']:.1%}, 확신 {g['win_conf']:.1%}) {win_selection}")
    print(f"    RUN_TOTAL:  {run_total_prediction} {g['run_total_baseline']} ({g['run_total_prob']:.1%}, 확신 {g['run_total_conf']:.1%}) {run_total_selection}")

In [ ]:
# 4/1 실제 결과 확인
results = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_경기결과.csv')
apr1 = results[results['날짜']=='2026-04-01']
print(f"4/1 경기결과: {len(apr1)}건")

# 승패 적중 확인
win_selections = today[today['win_conf'] >= 0.64].copy()
win_correct = 0
for _, selection in win_selections.iterrows():
    actual = apr1[apr1['gamePk']==selection['gamePk']]
    if len(actual)==0: continue
    a = actual.iloc[0]
    home_won = a['홈팀결과'] == '승'
    predicted_home = selection['win_prob'] >= 0.5
    correct = home_won == predicted_home
    win_correct += int(correct)
    mark = '✅' if correct else '❌'
    prediction = selection['home_team'] if predicted_home else selection['away_team']
    winner = a['홈팀'] if home_won else a['원정팀']
    print(f"  {mark} {prediction} (예측) vs {winner} (실제) — {a['원정팀']} {a['원정팀득점']}:{a['홈팀득점']} {a['홈팀']}")

print(f"\n승패 선택: {win_correct}/{len(win_selections)} ({win_correct/len(win_selections)*100:.0f}%)")

In [ ]:
run_total_selections = today[today['run_total_conf'] >= 0.60].copy()
run_total_correct = 0
run_total_total = 0
for _, selection in run_total_selections.iterrows():
    actual = apr1[apr1['gamePk']==selection['gamePk']]
    if len(actual)==0: continue
    a = actual.iloc[0]
    total_runs = a['홈팀득점'] + a['원정팀득점']
    predicted_upper = selection['run_total_prob'] >= 0.5
    actual_upper = total_runs > selection['run_total_baseline']
    if total_runs == selection['run_total_baseline']:
        continue  # push
    correct = predicted_upper == actual_upper
    run_total_correct += int(correct)
    run_total_total += 1
    mark = '✅' if correct else '❌'
    prediction = 'UPPER' if predicted_upper else 'LOWER'
    result = 'UPPER' if actual_upper else 'LOWER'
    print(f"  {mark} {prediction} {selection['run_total_baseline']} (예측) vs {result} (실제 {total_runs}점) — {a['원정팀']} @ {a['홈팀']}")

print(f"\n총득점 선택: {run_total_correct}/{run_total_total} ({run_total_correct/run_total_total*100:.0f}%)" if run_total_total > 0 else "결과 없음")
print(f"\n※ 1일 치 결과로 모델 성능 판단 불가 — 최소 1~2달 누적 필요")

In [ ]:
import pandas as pd
import numpy as np
import pickle, warnings
warnings.filterwarnings('ignore')

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
DATA_2026 = f'{MLB_PATH}/2026'

# 데이터 로드
results = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_경기결과.csv')
pit = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_투수개인기록.csv')
starters = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2026.csv')
today_all = pd.read_csv(f'{DATA_2026}/today_2026-04-01.csv')  # 컬럼 구조 참고용

# 심판 히스토리 (23~25 + 26)
ump_prev = pd.concat([
    pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]
], ignore_index=True)
ump_prev = ump_prev[ump_prev['포지션']=='Home Plate']
master_prev = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
prev_runs = master_prev[['gamePk','총득점']].drop_duplicates()
ump_prev = ump_prev.merge(prev_runs, on='gamePk', how='inner')

ump_2026 = pd.read_csv(f'{DATA_2026}/mlb_2026_심판.csv')
ump_2026 = ump_2026[ump_2026['포지션']=='Home Plate']

# 모델 로드
win_lr = pickle.load(open(f'{MLB_PATH}/종결_승패_LR.pkl','rb'))
win_lgb = pickle.load(open(f'{MLB_PATH}/종결_승패_LGB.pkl','rb'))
run_total_lr = pickle.load(open(f'{MLB_PATH}/종결_총득점_LR.pkl','rb'))
run_total_lgb = pickle.load(open(f'{MLB_PATH}/종결_총득점_LGB.pkl','rb'))

WIN_FEATS = ['era_diff','k9_diff','bb9_diff','ops_diff','wr5_diff','wr10_diff',
             'rs10_diff','bp_fatigue_1d_diff','run_total_baseline','run_margin','home_WIN_LOSS',
             'park_HR','ump_avg_runs','ump_vs_league','temperature']
RUN_TOTAL_FEATS = ['era_sum','whip_sum','k9_sum','bb9_sum','hr9_sum','ops_sum',
            'rs5_sum','rs10_sum','ra5_sum','ra10_sum','bp_fatigue_sum',
            'park_factor','park_R','park_HR','park_H','temperature',
            'ump_avg_runs','ump_vs_league','era_max','whip_max',
            'run_margin','home_WIN_LOSS','run_total_baseline','era_vs_run_total','ops_vs_run_total']

TEAM_NORMALIZE = {'Oakland Athletics': 'Athletics'}
def normalize_team(name):
    return TEAM_NORMALIZE.get(name, name)

print(f"경기결과: {len(results)}건 ({results['날짜'].min()} ~ {results['날짜'].max()})")
print(f"starters에 today 컬럼 있는지 확인 중...")
print(f"starters 컬럼: {list(starters.columns)[:10]}")
print(f"today 컬럼: {list(today_all.columns)[:10]}")

In [ ]:
import os
OL_PATH = f'{MLB_PATH}/MARKET_PRICE,LINEUP'
files_2026 = [f for f in os.listdir(OL_PATH) if '2026' in f]
for f in sorted(files_2026):
    size = os.path.getsize(f'{OL_PATH}/{f}')/1024
    print(f"  {f} ({size:.0f}KB)")

In [ ]:
import pandas as pd
OL_PATH = f'{MLB_PATH}/MARKET_PRICE,LINEUP'

market_price = pd.read_csv(f'{OL_PATH}/mlb_market_price_2026.csv')
lineup = pd.read_csv(f'{OL_PATH}/mlb_lineup_2026.csv')
st = pd.read_csv(f'{OL_PATH}/mlb_starters_2026.csv')

print(f"=== market_price: {market_price.shape} ===")
print(f"  컬럼: {list(market_price.columns)}")
print(market_price.head(2).to_string())

print(f"\n=== lineup: {lineup.shape} ===")
print(f"  컬럼: {list(lineup.columns)}")

print(f"\n=== starters: {st.shape} ===")
print(f"  컬럼: {list(st.columns)}")

In [ ]:
# 기본 테이블: starters + market_price + lineup OPS 합치기
import pandas as pd
import numpy as np

# starters 기본
base = st[['gamePk','date','away_team','home_team',
           'home_sp_name','home_sp_era','home_sp_whip','home_sp_k9','home_sp_bb9','home_sp_hr9',
           'away_sp_name','away_sp_era','away_sp_whip','away_sp_k9','away_sp_bb9','away_sp_hr9']].copy()

# market_price 매칭 (date + home_team)
market_price_slim = market_price[['date','home_team','home_WIN_LOSS','away_WIN_LOSS','run_total_baseline','run_margin','temperature']].copy()
base = base.merge(market_price_slim, on=['date','home_team'], how='left')
print(f"market_price 매칭: {base['home_WIN_LOSS'].notna().sum()}/{len(base)}")

# lineup OPS (gamePk + side별 평균)
lineup_ops = lineup.groupby(['gamePk','side'])['ops'].mean().reset_index()
home_ops = lineup_ops[lineup_ops['side']=='home'][['gamePk','ops']].rename(columns={'ops':'home_lineup_ops'})
away_ops = lineup_ops[lineup_ops['side']=='away'][['gamePk','ops']].rename(columns={'ops':'away_lineup_ops'})
base = base.merge(home_ops, on='gamePk', how='left')
base = base.merge(away_ops, on='gamePk', how='left')
print(f"lineup 매칭: home={base['home_lineup_ops'].notna().sum()}, away={base['away_lineup_ops'].notna().sum()}")

# 파크팩터 (today CSV에서 구장매핑 확인)
base = base.merge(results[['gamePk','홈팀득점','원정팀득점','홈팀결과','총득점']].drop_duplicates(), on='gamePk', how='left')
print(f"결과 매칭: {base['홈팀득점'].notna().sum()}/{len(base)}")
print(f"\nbase: {base.shape}")
print(f"날짜 범위: {base['date'].min()} ~ {base['date'].max()}")

In [ ]:
# 파크팩터 — 2025 기준 사용 (2026 시즌초)
pf = pd.read_csv(f'{MLB_PATH}/park_factors_2025.csv')
print(f"파크팩터: {pf.shape}")
print(f"컬럼: {list(pf.columns)}")
print(pf.head(3).to_string())

In [ ]:
# 파크팩터 팀명 매핑
PF_TEAM_MAP = {
    'Rockies':'Colorado Rockies','Athletics':'Athletics','Tigers':'Detroit Tigers',
    'Reds':'Cincinnati Reds','Orioles':'Baltimore Orioles','Rangers':'Texas Rangers',
    'Red Sox':'Boston Red Sox','Cubs':'Chicago Cubs','Twins':'Minnesota Twins',
    'Brewers':'Milwaukee Brewers','Phillies':'Philadelphia Phillies','Yankees':'New York Yankees',
    'Braves':'Atlanta Braves','Cardinals':'St. Louis Cardinals','Astros':'Houston Astros',
    'Blue Jays':'Toronto Blue Jays','Mets':'New York Mets','Royals':'Kansas City Royals',
    'Diamondbacks':'Arizona Diamondbacks','Nationals':'Washington Nationals',
    'Dodgers':'Los Angeles Dodgers','Pirates':'Pittsburgh Pirates','Padres':'San Diego Padres',
    'Giants':'San Francisco Giants','Marlins':'Miami Marlins','Angels':'Los Angeles Angels',
    'Guardians':'Cleveland Guardians','White Sox':'Chicago White Sox',
    'Rays':'Tampa Bay Rays','Mariners':'Seattle Mariners',
}
pf['home_team'] = pf['Team'].map(PF_TEAM_MAP)
pf_slim = pf[['home_team','ParkFactor','R','HR','H']].rename(
    columns={'ParkFactor':'park_factor','R':'park_R','HR':'park_HR','H':'park_H'})

base = base.merge(pf_slim, on='home_team', how='left')
print(f"파크팩터 매칭: {base['park_factor'].notna().sum()}/{len(base)}")
print(f"park_HR 범위: {base['park_HR'].min()} ~ {base['park_HR'].max()}")

In [ ]:
# 날짜별 백테스트 — 각 경기 시점에서 이전 데이터만 사용
from sklearn.impute import SimpleImputer

base['date'] = pd.to_datetime(base['date'])
base = base.sort_values('date').reset_index(drop=True)
dates = sorted(base['date'].unique())
print(f"날짜: {len(dates)}일, 경기: {len(base)}건")

# 심판 히스토리 (23~25)
ump_prev_slim = ump_prev[['심판명','총득점']].copy()
ump_hist_base = {}
all_runs_base = []
for _, r in ump_prev_slim.iterrows():
    ump_hist_base.setdefault(r['심판명'], []).append(r['총득점'])
    all_runs_base.append(r['총득점'])

# 2026 심판 매핑
ump26 = pd.read_csv(f'{DATA_2026}/mlb_2026_심판.csv')
ump26_hp = ump26[ump26['포지션']=='Home Plate'][['gamePk','심판명']].drop_duplicates()
base = base.merge(ump26_hp, on='gamePk', how='left')
print(f"심판 매칭: {base['심판명'].notna().sum()}/{len(base)}")

# 선발세트 (불펜용)
starter_set = set()
for _, r in st.iterrows():
    starter_set.add((r['gamePk'], r.get('home_sp_name','')))
    starter_set.add((r['gamePk'], r.get('away_sp_name','')))

pit['is_starter'] = pit.apply(lambda r: (r['gamePk'], r['선수명']) in starter_set, axis=1)
bp = pit[~pit['is_starter']].copy()
bp_game = bp.groupby(['gamePk','팀']).agg(bp_ip=('이닝','sum')).reset_index()
bp_game['팀'] = bp_game['팀'].apply(normalize_team)
bp_game = bp_game.merge(results[['gamePk','날짜']].drop_duplicates(), on='gamePk', how='left')
bp_game['날짜'] = pd.to_datetime(bp_game['날짜'])

print(f"불펜 로그: {len(bp_game)}건")
print(f"\n준비 완료 — 다음 셀에서 루프 실행")

In [ ]:
# 날짜별 백테스트 루프
all_preds = []

for target_date in dates:
    day_games = base[base['date']==target_date].copy()
    if len(day_games)==0: continue

    prev_results = results[pd.to_datetime(results['날짜']) < target_date]

    # 최근폼
    form = {}
    if len(prev_results) > 0:
        teams = set(prev_results['홈팀']) | set(prev_results['원정팀'])
        prev_sorted = prev_results.sort_values('날짜')
        for team in teams:
            tg = []
            for _, r in prev_sorted[(prev_sorted['홈팀']==team)|(prev_sorted['원정팀']==team)].iterrows():
                if r['홈팀']==team:
                    tg.append((1 if r['홈팀결과']=='승' else 0, r['홈팀득점'], r['원정팀득점']))
                else:
                    tg.append((1 if r['원정팀결과']=='승' else 0, r['원정팀득점'], r['홈팀득점']))
            form[team] = {}
            for w, s in [(5,'5'),(10,'10')]:
                rec = tg[-w:]
                if len(rec) >= 3:
                    form[team][f'wr{s}'] = np.mean([g[0] for g in rec])
                    form[team][f'rs{s}'] = np.mean([g[1] for g in rec])
                    form[team][f'ra{s}'] = np.mean([g[2] for g in rec])

    for col in ['home_wr5','home_wr10','home_rs5','home_rs10','home_ra5','home_ra10',
                'away_wr5','away_wr10','away_rs5','away_rs10','away_ra5','away_ra10']:
        day_games[col] = np.nan

    for idx, row in day_games.iterrows():
        for team_col, prefix in [('home_team','home'),('away_team','away')]:
            tf = form.get(row[team_col], {})
            for key in ['wr5','wr10','rs5','rs10','ra5','ra10']:
                day_games.loc[idx, f'{prefix}_{key}'] = tf.get(key, np.nan)

    # 불펜소모도
    target_dt = pd.Timestamp(target_date)
    d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
    prev_bp = bp_game[bp_game['날짜'] < target_dt]

    for idx, row in day_games.iterrows():
        for team_col, prefix in [('home_team','home'),('away_team','away')]:
            team = row[team_col]
            tbp = prev_bp[prev_bp['팀']==team]
            day_games.loc[idx, f'{prefix}_bp_fatigue_1d'] = tbp[tbp['날짜']>=d1]['bp_ip'].sum()
            day_games.loc[idx, f'{prefix}_bp_fatigue_3d'] = tbp[tbp['날짜']>=d3]['bp_ip'].sum()

    # 심판
    ump_hist = {k: v[:] for k, v in ump_hist_base.items()}
    all_runs_copy = all_runs_base[:]
    prev_ump = ump26_hp[ump26_hp['gamePk'].isin(set(prev_results['gamePk']))]
    for _, r in prev_ump.merge(results[['gamePk','총득점']].drop_duplicates(), on='gamePk').iterrows():
        ump_hist.setdefault(r['심판명'], []).append(r['총득점'])
        all_runs_copy.append(r['총득점'])

    league_avg = np.mean(all_runs_copy) if all_runs_copy else 9.0
    for idx, row in day_games.iterrows():
        ump = row.get('심판명','')
        hist = ump_hist.get(ump, [])
        if len(hist) >= 5:
            day_games.loc[idx, 'ump_avg_runs'] = np.mean(hist)
            day_games.loc[idx, 'ump_vs_league'] = np.mean(hist) - league_avg
        else:
            day_games.loc[idx, 'ump_avg_runs'] = np.nan
            day_games.loc[idx, 'ump_vs_league'] = np.nan

    # 파생 피처
    day_games['era_diff'] = day_games['home_sp_era'] - day_games['away_sp_era']
    day_games['k9_diff'] = day_games['home_sp_k9'] - day_games['away_sp_k9']
    day_games['bb9_diff'] = day_games['home_sp_bb9'] - day_games['away_sp_bb9']
    day_games['ops_diff'] = day_games['home_lineup_ops'] - day_games['away_lineup_ops']
    day_games['wr5_diff'] = day_games['home_wr5'] - day_games['away_wr5']
    day_games['wr10_diff'] = day_games['home_wr10'] - day_games['away_wr10']
    day_games['rs10_diff'] = day_games['home_rs10'] - day_games['away_rs10']
    day_games['bp_fatigue_1d_diff'] = day_games['home_bp_fatigue_1d'] - day_games['away_bp_fatigue_1d']
    day_games['era_sum'] = day_games['home_sp_era'] + day_games['away_sp_era']
    day_games['whip_sum'] = day_games['home_sp_whip'] + day_games['away_sp_whip']
    day_games['k9_sum'] = day_games['home_sp_k9'] + day_games['away_sp_k9']
    day_games['bb9_sum'] = day_games['home_sp_bb9'] + day_games['away_sp_bb9']
    day_games['hr9_sum'] = day_games['home_sp_hr9'] + day_games['away_sp_hr9']
    day_games['ops_sum'] = day_games['home_lineup_ops'] + day_games['away_lineup_ops']
    day_games['rs5_sum'] = day_games['home_rs5'] + day_games['away_rs5']
    day_games['rs10_sum'] = day_games['home_rs10'] + day_games['away_rs10']
    day_games['ra5_sum'] = day_games['home_ra5'] + day_games['away_ra5']
    day_games['ra10_sum'] = day_games['home_ra10'] + day_games['away_ra10']
    day_games['bp_fatigue_sum'] = day_games['home_bp_fatigue_3d'] + day_games['away_bp_fatigue_3d']
    day_games['era_max'] = day_games[['home_sp_era','away_sp_era']].max(axis=1)
    day_games['whip_max'] = day_games[['home_sp_whip','away_sp_whip']].max(axis=1)
    day_games['era_vs_run_total'] = day_games['era_sum'] - day_games['run_total_baseline']
    day_games['ops_vs_run_total'] = day_games['ops_sum'] * 10 - day_games['run_total_baseline']

    # 예측
    try:
        wp = 0.8*win_lr.predict_proba(day_games[WIN_FEATS])[:,1] + 0.2*win_lgb.predict_proba(day_games[WIN_FEATS])[:,1]
        op = 0.5*run_total_lr.predict_proba(day_games[RUN_TOTAL_FEATS])[:,1] + 0.5*run_total_lgb.predict_proba(day_games[RUN_TOTAL_FEATS])[:,1]
        day_games['win_prob'] = wp
        day_games['run_total_prob'] = op
        day_games['win_conf'] = np.maximum(wp, 1-wp)
        day_games['run_total_conf'] = np.maximum(op, 1-op)
        all_preds.append(day_games)
    except Exception as e:
        print(f"  {target_date.strftime('%Y-%m-%d')}: 예측 실패 — {e}")

pred_df = pd.concat(all_preds, ignore_index=True)
pred_df = pred_df[pred_df['홈팀결과'].notna()]  # 결과 있는 것만

# 적중률
pred_df['home_win'] = (pred_df['홈팀결과']=='승').astype(int)
pred_df['win_pred'] = (pred_df['win_prob']>=0.5).astype(int)
pred_df['win_correct'] = (pred_df['win_pred']==pred_df['home_win']).astype(int)

pred_df['total_runs'] = pred_df['홈팀득점'] + pred_df['원정팀득점']
pred_df['run_total_pred'] = (pred_df['run_total_prob']>=0.5).astype(int)
pred_df['actual_upper'] = (pred_df['total_runs']>pred_df['run_total_baseline']).astype(int)
pred_df['run_total_correct'] = (pred_df['run_total_pred']==pred_df['actual_upper']).astype(int)
push = pred_df['total_runs']==pred_df['run_total_baseline']

print(f"{'='*50}")
print(f"2026 시즌초 백테스트 ({pred_df['date'].min().strftime('%m/%d')}~{pred_df['date'].max().strftime('%m/%d')})")
print(f"{'='*50}")
print(f"\n전체: {len(pred_df)}경기")

for thr in [0.55, 0.60, 0.64]:
    wm = pred_df['win_conf']>=thr
    om = (pred_df['run_total_conf']>=thr) & (~push)
    wa = pred_df.loc[wm,'win_correct'].mean()*100 if wm.sum()>0 else 0
    oa = pred_df.loc[om,'run_total_correct'].mean()*100 if om.sum()>0 else 0
    print(f"  {thr:.2f}: 승패 {wa:.0f}% ({wm.sum()}경기) | 총득점 {oa:.0f}% ({om.sum()}경기)")

In [ ]:
# today_2026-04-01.csv (daily_collector) vs starters+market_price (백테스트)
import pandas as pd
MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
DATA_2026 = f'{MLB_PATH}/2026'

today = pd.read_csv(f'{DATA_2026}/today_2026-04-01.csv')
st = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_starters_2026.csv')
market_price = pd.read_csv(f'{MLB_PATH}/MARKET_PRICE,LINEUP/mlb_market_price_2026.csv')

# 4/1 경기만
st_apr1 = st[st['date']=='2026-04-01']
market_price_apr1 = market_price[market_price['date']=='2026-04-01']

print(f"today: {len(today)}경기, starters: {len(st_apr1)}경기, market_price: {len(market_price_apr1)}경기")

# 같은 경기 ERA 비교
for _, t in today.iterrows():
    home = t['home_team']
    s = st_apr1[st_apr1['home_team']==home]
    if len(s)==0: continue
    s = s.iloc[0]
    print(f"\n  {t['away_team']} @ {home}")
    print(f"    home ERA: today={t['home_sp_era']:.2f}  starters={s['home_sp_era']:.2f}  diff={t['home_sp_era']-s['home_sp_era']:.2f}")
    print(f"    away ERA: today={t['away_sp_era']:.2f}  starters={s['away_sp_era']:.2f}  diff={t['away_sp_era']-s['away_sp_era']:.2f}")

# market_price 비교도
print(f"\n=== 시장가격 비교 ===")
for _, t in today.iterrows():
    home = t['home_team']
    o = market_price_apr1[market_price_apr1['home_team']==home]
    if len(o)==0: continue
    o = o.iloc[0]
    print(f"  {home}: WIN_LOSS today={t['home_WIN_LOSS']:.0f} market_price={o['home_WIN_LOSS']:.0f} | RUN_TOTAL today={t['run_total_baseline']} market_price={o['run_total_baseline']}")

In [ ]:
# 백테스트 루프의 4/1 예측 vs daily_predictor 예측 비교
bt_apr1 = pred_df[pred_df['date']=='2026-04-01']
today = pd.read_csv(f'{DATA_2026}/today_2026-04-01.csv')

print(f"백테스트 4/1: {len(bt_apr1)}경기, today: {len(today)}경기")

for _, bt in bt_apr1.iterrows():
    home = bt['home_team']
    t = today[today['home_team']==home]
    if len(t)==0: continue
    t = t.iloc[0]

    print(f"\n  {home}")
    # rolling 피처 비교
    for col in ['home_wr5','away_wr5','home_bp_fatigue_1d','away_bp_fatigue_1d',
                'ump_avg_runs','ump_vs_league']:
        bv = bt.get(col, np.nan)
        tv = t.get(col, np.nan) if col in t.index else np.nan
        diff = '' if (pd.isna(bv) and pd.isna(tv)) else f'diff={bv-tv:.3f}' if (pd.notna(bv) and pd.notna(tv)) else 'NaN차이'
        print(f"    {col:25s} 백테={bv if pd.notna(bv) else 'NaN':>8} today={tv if pd.notna(tv) else 'NaN':>8}  {diff}")

In [ ]:
import os
DATA_2026 = '<DRIVE_ROOT>/MLB데이터/2026'
files = [f for f in os.listdir(DATA_2026) if f.endswith('.csv')]
for f in sorted(files):
    size = os.path.getsize(f'{DATA_2026}/{f}')/1024
    print(f"  {f} ({size:.0f}KB)")

In [ ]:
# ============================================================
# daily_predictor 통합 테스트 (원클릭)
# ============================================================
import pandas as pd
import numpy as np
import pickle, os, warnings
warnings.filterwarnings('ignore')

MLB_PATH = '<DRIVE_ROOT>/MLB데이터'
DATA_2026 = f'{MLB_PATH}/2026'
OL_PATH = f'{MLB_PATH}/MARKET_PRICE,LINEUP'
TARGET_DATE = '2026-04-01'  # ← 매일 바꾸기

# === 1. 데이터 로드 ===
st = pd.read_csv(f'{OL_PATH}/mlb_starters_2026.csv')
market_price = pd.read_csv(f'{OL_PATH}/mlb_market_price_2026.csv')
lineup = pd.read_csv(f'{OL_PATH}/mlb_lineup_2026.csv')
results = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_경기결과.csv')
pit = pd.read_csv(f'{DATA_2026}/mlb_2026_정규시즌_투수개인기록.csv')
pf = pd.read_csv(f'{MLB_PATH}/park_factors_2025.csv')

# 심판 (23~25 히스토리 + 2026)
ump_prev = pd.concat([pd.read_csv(f'{MLB_PATH}/{yr}/mlb_{yr}_심판.csv') for yr in [2023,2024,2025]], ignore_index=True)
ump_prev = ump_prev[ump_prev['포지션']=='Home Plate']
master_prev = pd.read_csv(f'{MLB_PATH}/B안_마스터테이블.csv')
ump_prev = ump_prev.merge(master_prev[['gamePk','총득점']].drop_duplicates(), on='gamePk', how='inner')
ump_2026 = pd.read_csv(f'{DATA_2026}/mlb_2026_심판.csv')
ump_2026_hp = ump_2026[ump_2026['포지션']=='Home Plate'][['gamePk','심판명']].drop_duplicates()

# 모델
win_lr = pickle.load(open(f'{MLB_PATH}/종결_승패_LR.pkl','rb'))
win_lgb = pickle.load(open(f'{MLB_PATH}/종결_승패_LGB.pkl','rb'))
run_total_lr = pickle.load(open(f'{MLB_PATH}/종결_총득점_LR.pkl','rb'))
run_total_lgb = pickle.load(open(f'{MLB_PATH}/종결_총득점_LGB.pkl','rb'))

WIN_FEATS = ['era_diff','k9_diff','bb9_diff','ops_diff','wr5_diff','wr10_diff',
             'rs10_diff','bp_fatigue_1d_diff','run_total_baseline','run_margin','home_WIN_LOSS',
             'park_HR','ump_avg_runs','ump_vs_league','temperature']
RUN_TOTAL_FEATS = ['era_sum','whip_sum','k9_sum','bb9_sum','hr9_sum','ops_sum',
            'rs5_sum','rs10_sum','ra5_sum','ra10_sum','bp_fatigue_sum',
            'park_factor','park_R','park_HR','park_H','temperature',
            'ump_avg_runs','ump_vs_league','era_max','whip_max',
            'run_margin','home_WIN_LOSS','run_total_baseline','era_vs_run_total','ops_vs_run_total']

TEAM_NORMALIZE = {'Oakland Athletics':'Athletics'}
PF_MAP = {'Rockies':'Colorado Rockies','Athletics':'Athletics','Tigers':'Detroit Tigers',
    'Reds':'Cincinnati Reds','Orioles':'Baltimore Orioles','Rangers':'Texas Rangers',
    'Red Sox':'Boston Red Sox','Cubs':'Chicago Cubs','Twins':'Minnesota Twins',
    'Brewers':'Milwaukee Brewers','Phillies':'Philadelphia Phillies','Yankees':'New York Yankees',
    'Braves':'Atlanta Braves','Cardinals':'St. Louis Cardinals','Astros':'Houston Astros',
    'Blue Jays':'Toronto Blue Jays','Mets':'New York Mets','Royals':'Kansas City Royals',
    'Diamondbacks':'Arizona Diamondbacks','Nationals':'Washington Nationals',
    'Dodgers':'Los Angeles Dodgers','Pirates':'Pittsburgh Pirates','Padres':'San Diego Padres',
    'Giants':'San Francisco Giants','Marlins':'Miami Marlins','Angels':'Los Angeles Angels',
    'Guardians':'Cleveland Guardians','White Sox':'Chicago White Sox',
    'Rays':'Tampa Bay Rays','Mariners':'Seattle Mariners'}

# === 2. base 테이블 (당일 경기) ===
day_st = st[st['date']==TARGET_DATE].copy()
if len(day_st)==0:
    print(f"⚠️ {TARGET_DATE} starters 없음")
else:
    base = day_st[['gamePk','date','away_team','home_team',
        'home_sp_name','home_sp_era','home_sp_whip','home_sp_k9','home_sp_bb9','home_sp_hr9',
        'away_sp_name','away_sp_era','away_sp_whip','away_sp_k9','away_sp_bb9','away_sp_hr9']].copy()

    # market_price
    base = base.merge(market_price[['date','home_team','home_WIN_LOSS','away_WIN_LOSS','run_total_baseline','run_margin','temperature']],
                      on=['date','home_team'], how='left')

    # lineup OPS
    lu_ops = lineup.groupby(['gamePk','side'])['ops'].mean().reset_index()
    base = base.merge(lu_ops[lu_ops['side']=='home'][['gamePk','ops']].rename(columns={'ops':'home_lineup_ops'}), on='gamePk', how='left')
    base = base.merge(lu_ops[lu_ops['side']=='away'][['gamePk','ops']].rename(columns={'ops':'away_lineup_ops'}), on='gamePk', how='left')

    # 파크팩터
    pf['home_team'] = pf['Team'].map(PF_MAP)
    base = base.merge(pf[['home_team','ParkFactor','R','HR','H']].rename(
        columns={'ParkFactor':'park_factor','R':'park_R','HR':'park_HR','H':'park_H'}), on='home_team', how='left')

    # 경기결과 (적중 확인용)
    base = base.merge(results[['gamePk','홈팀득점','원정팀득점','홈팀결과','총득점']].drop_duplicates(), on='gamePk', how='left')

    # 심판
    base = base.merge(ump_2026_hp, on='gamePk', how='left')

    # === 3. rolling 피처 ===
    target_dt = pd.Timestamp(TARGET_DATE)
    prev_results = results[pd.to_datetime(results['날짜']) < target_dt]

    # 최근폼
    form = {}
    if len(prev_results) > 0:
        prev_sorted = prev_results.sort_values('날짜')
        for team in set(prev_sorted['홈팀']) | set(prev_sorted['원정팀']):
            tg = []
            for _, r in prev_sorted[(prev_sorted['홈팀']==team)|(prev_sorted['원정팀']==team)].iterrows():
                if r['홈팀']==team:
                    tg.append((1 if r['홈팀결과']=='승' else 0, r['홈팀득점'], r['원정팀득점']))
                else:
                    tg.append((1 if r['원정팀결과']=='승' else 0, r['원정팀득점'], r['홈팀득점']))
            form[team] = {}
            for w, s in [(5,'5'),(10,'10')]:
                rec = tg[-w:]
                if len(rec) >= 3:
                    form[team][f'wr{s}'] = np.mean([g[0] for g in rec])
                    form[team][f'rs{s}'] = np.mean([g[1] for g in rec])
                    form[team][f'ra{s}'] = np.mean([g[2] for g in rec])

    for idx, row in base.iterrows():
        for tc, pf_ in [('home_team','home'),('away_team','away')]:
            tf = form.get(row[tc], {})
            for k in ['wr5','wr10','rs5','rs10','ra5','ra10']:
                base.loc[idx, f'{pf_}_{k}'] = tf.get(k, np.nan)

    # 불펜소모도
    starter_set = set()
    for _, r in st.iterrows():
        starter_set.add((r['gamePk'], r.get('home_sp_name','')))
        starter_set.add((r['gamePk'], r.get('away_sp_name','')))
    pit_c = pit.copy()
    pit_c['is_starter'] = pit_c.apply(lambda r: (r['gamePk'], r['선수명']) in starter_set, axis=1)
    bp = pit_c[~pit_c['is_starter']]
    bp_game = bp.groupby(['gamePk','팀']).agg(bp_ip=('이닝','sum')).reset_index()
    bp_game['팀'] = bp_game['팀'].apply(lambda x: TEAM_NORMALIZE.get(x,x))
    bp_game = bp_game.merge(results[['gamePk','날짜']].drop_duplicates(), on='gamePk', how='left')
    bp_game['날짜'] = pd.to_datetime(bp_game['날짜'])
    d1 = (target_dt - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    d3 = (target_dt - pd.Timedelta(days=3)).strftime('%Y-%m-%d')
    prev_bp = bp_game[bp_game['날짜'] < target_dt]

    for idx, row in base.iterrows():
        for tc, pf_ in [('home_team','home'),('away_team','away')]:
            tbp = prev_bp[prev_bp['팀']==row[tc]]
            base.loc[idx, f'{pf_}_bp_fatigue_1d'] = tbp[tbp['날짜']>=d1]['bp_ip'].sum()
            base.loc[idx, f'{pf_}_bp_fatigue_3d'] = tbp[tbp['날짜']>=d3]['bp_ip'].sum()

    # 심판 (교차누적)
    ump_hist = {}
    all_runs = []
    for _, r in ump_prev.iterrows():
        ump_hist.setdefault(r['심판명'], []).append(r['총득점'])
        all_runs.append(r['총득점'])
    prev_ump = ump_2026_hp[ump_2026_hp['gamePk'].isin(set(prev_results['gamePk']))]
    for _, r in prev_ump.merge(results[['gamePk','총득점']].drop_duplicates(), on='gamePk').iterrows():
        ump_hist.setdefault(r['심판명'], []).append(r['총득점'])
        all_runs.append(r['총득점'])
    league_avg = np.mean(all_runs)

    for idx, row in base.iterrows():
        ump = row.get('심판명','')
        hist = ump_hist.get(ump, [])
        if len(hist) >= 5:
            base.loc[idx, 'ump_avg_runs'] = np.mean(hist)
            base.loc[idx, 'ump_vs_league'] = np.mean(hist) - league_avg
        else:
            base.loc[idx, 'ump_avg_runs'] = np.nan
            base.loc[idx, 'ump_vs_league'] = np.nan

    # === 4. 파생 피처 ===
    base['era_diff'] = base['home_sp_era'] - base['away_sp_era']
    base['k9_diff'] = base['home_sp_k9'] - base['away_sp_k9']
    base['bb9_diff'] = base['home_sp_bb9'] - base['away_sp_bb9']
    base['ops_diff'] = base['home_lineup_ops'] - base['away_lineup_ops']
    base['wr5_diff'] = base['home_wr5'] - base['away_wr5']
    base['wr10_diff'] = base['home_wr10'] - base['away_wr10']
    base['rs10_diff'] = base['home_rs10'] - base['away_rs10']
    base['bp_fatigue_1d_diff'] = base['home_bp_fatigue_1d'] - base['away_bp_fatigue_1d']
    base['era_sum'] = base['home_sp_era'] + base['away_sp_era']
    base['whip_sum'] = base['home_sp_whip'] + base['away_sp_whip']
    base['k9_sum'] = base['home_sp_k9'] + base['away_sp_k9']
    base['bb9_sum'] = base['home_sp_bb9'] + base['away_sp_bb9']
    base['hr9_sum'] = base['home_sp_hr9'] + base['away_sp_hr9']
    base['ops_sum'] = base['home_lineup_ops'] + base['away_lineup_ops']
    base['rs5_sum'] = base['home_rs5'] + base['away_rs5']
    base['rs10_sum'] = base['home_rs10'] + base['away_rs10']
    base['ra5_sum'] = base['home_ra5'] + base['away_ra5']
    base['ra10_sum'] = base['home_ra10'] + base['away_ra10']
    base['bp_fatigue_sum'] = base['home_bp_fatigue_3d'] + base['away_bp_fatigue_3d']
    base['era_max'] = base[['home_sp_era','away_sp_era']].max(axis=1)
    base['whip_max'] = base[['home_sp_whip','away_sp_whip']].max(axis=1)
    base['era_vs_run_total'] = base['era_sum'] - base['run_total_baseline']
    base['ops_vs_run_total'] = base['ops_sum'] * 10 - base['run_total_baseline']

    # === 5. 예측 ===
    wp = 0.8*win_lr.predict_proba(base[WIN_FEATS])[:,1] + 0.2*win_lgb.predict_proba(base[WIN_FEATS])[:,1]
    op = 0.5*run_total_lr.predict_proba(base[RUN_TOTAL_FEATS])[:,1] + 0.5*run_total_lgb.predict_proba(base[RUN_TOTAL_FEATS])[:,1]
    base['win_prob'] = wp
    base['run_total_prob'] = op
    base['win_conf'] = np.maximum(wp, 1-wp)
    base['run_total_conf'] = np.maximum(op, 1-op)

    # === 6. 출력 ===
    print(f"{'='*70}")
    print(f"  MLB 예측 — {TARGET_DATE}")
    print(f"  승패 선택: 확신 64%+ | 총득점 선택: 확신 60%+")
    print(f"{'='*70}")

    win_selections = 0
    run_total_selections = 0
    for _, g in base.iterrows():
        win_prediction = g['home_team'] if g['win_prob']>=0.5 else g['away_team']
        run_total_prediction = 'UPPER' if g['run_total_prob']>=0.5 else 'LOWER'
        wb = '선택' if g['win_conf']>=0.64 else ''
        ob = '선택' if g['run_total_conf']>=0.60 else ''
        if wb: win_selections += 1
        if ob: run_total_selections += 1

        print(f"\n  {g['away_team']} @ {g['home_team']}")
        print(f"    승패: {win_prediction} ({g['win_prob']:.1%}, 확신 {g['win_conf']:.1%}) {wb}")
        print(f"    RUN_TOTAL:  {run_total_prediction} {g['run_total_baseline']} ({g['run_total_prob']:.1%}, 확신 {g['run_total_conf']:.1%}) {ob}")

    print(f"\n{'='*70}")
    print(f"  선택 요약: 승패 {win_selections}경기, 총득점 {run_total_selections}경기")
    print(f"{'='*70}")

In [ ]:
# Step 1: 경기 목록 + 선발투수 (MLB Stats API)
import requests, pandas as pd
from datetime import date

TODAY = str(date.today())  # 또는 '2026-04-02'
print(f"📅 {TODAY}")

url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={TODAY}&hydrate=probablePitcher,linescore"
data = requests.get(url, timeout=15).json()

rows = []
for d in data.get('dates', []):
    for g in d.get('games', []):
        home = g['teams']['home']
        away = g['teams']['away']

        # 선발투수 ID
        h_sp = home.get('probablePitcher', {})
        a_sp = away.get('probablePitcher', {})

        rows.append({
            'gamePk': g['gamePk'],
            'date': g['officialDate'],
            'status': g['status']['detailedState'],
            'home_team': home['team']['name'],
            'away_team': away['team']['name'],
            'home_sp_name': h_sp.get('fullName', ''),
            'home_sp_id': h_sp.get('id', ''),
            'away_sp_name': a_sp.get('fullName', ''),
            'away_sp_id': a_sp.get('id', ''),
        })

df = pd.DataFrame(rows)
print(f"\n경기 수: {len(df)}")
print(df[['gamePk','home_team','away_team','home_sp_name','away_sp_name','status']].to_string())

# 선발투수 시즌 스탯 수집
def get_pitcher_stats(player_id):
    if not player_id: return {}
    url = f"https://statsapi.mlb.com/api/v1/people/{player_id}?hydrate=stats(group=[pitching],type=[season])"
    data = requests.get(url, timeout=10).json()
    stats = {}
    try:
        s = data['people'][0]['stats'][0]['splits'][-1]['stat']
        stats = {
            'era': float(s.get('era', 0)),
            'whip': float(s.get('whip', 0)),
            'k9': float(s.get('strikeoutsPer9Inn', 0)),
            'bb9': float(s.get('walksPer9Inn', 0)),
            'hr9': float(s.get('homeRunsPer9', 0)),
        }
    except: pass
    return stats

for idx, row in df.iterrows():
    for side in ['home', 'away']:
        sp_id = row[f'{side}_sp_id']
        stats = get_pitcher_stats(sp_id)
        for k, v in stats.items():
            df.at[idx, f'{side}_sp_{k}'] = v

print(f"\n선발투수 스탯 수집 완료:")
print(df[['home_team','home_sp_name','home_sp_era','home_sp_whip','away_sp_name','away_sp_era','away_sp_whip']].to_string())

In [ ]:
# Step 2 수정: 경기 전 라인업 수집 (game feed 방식)
import requests, pandas as pd

TODAY = '2026-04-02'

# Step 1 결과에서 활성 경기 gamePk 가져오기 (또는 직접 지정)
url = f"https://statsapi.mlb.com/api/v1/schedule?sportId=1&date={TODAY}&hydrate=probablePitcher"
sched = requests.get(url, timeout=15).json()
game_pks = []
for d in sched.get('dates', []):
    for g in d.get('games', []):
        if g['status']['detailedState'] not in ['Postponed', 'Cancelled']:
            game_pks.append((g['gamePk'], g['teams']['home']['team']['name'], g['teams']['away']['team']['name']))

print(f"경기: {len(game_pks)}개\n")

# 방법1: game feed에서 라인업 확인
for gpk, home, away in game_pks[:2]:  # 테스트용 2경기만
    print(f"--- {gpk}: {away} @ {home} ---")

    # feed/live 확인
    feed = requests.get(f"https://statsapi.mlb.com/api/v1.1/game/{gpk}/feed/live", timeout=15).json()
    box = feed.get('liveData', {}).get('boxscore', {}).get('teams', {})

    for side in ['home', 'away']:
        batting = box.get(side, {}).get('battingOrder', [])
        players = box.get(side, {}).get('players', {})
        team = box.get(side, {}).get('team', {}).get('name', side)

        if batting:
            ops_list = []
            for pid in batting:
                p = players.get(f'ID{pid}', {})
                name = p.get('person', {}).get('fullName', '?')
                ops = p.get('seasonStats', {}).get('batting', {}).get('ops', '')
                pos = p.get('position', {}).get('abbreviation', '')
                ops_list.append(float(ops) if ops and ops != '.---' else None)
                print(f"  {side} {len(ops_list)}. {name:20s} {pos:3s} OPS={ops}")
            valid = [o for o in ops_list if o is not None]
            avg_ops = sum(valid)/len(valid) if valid else None
            print(f"  → 평균 OPS: {avg_ops}\n")
        else:
            print(f"  {side}: 라인업 미확정\n")

    # 방법2: 혹시 lineups endpoint가 따로 있는지
    try:
        lu = requests.get(f"https://statsapi.mlb.com/api/v1/game/{gpk}/lineups", timeout=10)
        print(f"  lineups endpoint: {lu.status_code}, keys={list(lu.json().keys()) if lu.status_code==200 else 'N/A'}")
    except:
        print(f"  lineups endpoint: 실패")
    print()

In [ ]:
# Step 2: 라인업 OPS (MLB Stats API)
import time

# Postponed 제거
df = df[df['status'] != 'Postponed'].copy().reset_index(drop=True)
print(f"활성 경기: {len(df)}개\n")

def get_lineup_ops(gamePk, side):
    """라인업 확정 시 9명 OPS 평균, 미확정이면 NaN"""
    # 방법1: boxscore (라인업 확정된 경우)
    url = f"https://statsapi.mlb.com/api/v1/game/{gamePk}/boxscore"
    try:
        data = requests.get(url, timeout=10).json()
        team = data['teams'][side]
        batting = team.get('battingOrder', [])
        if not batting:
            return None, '미확정'

        players = team.get('players', {})
        ops_list = []
        for pid in batting:
            p = players.get(f'ID{pid}', {})
            stats = p.get('seasonStats', {}).get('batting', {})
            ops = stats.get('ops', '.000')
            try:
                ops_list.append(float(ops))
            except:
                ops_list.append(0.0)

        if ops_list:
            return round(sum(ops_list) / len(ops_list), 3), f'{len(ops_list)}명'
    except:
        pass
    return None, '미확정'

for idx, row in df.iterrows():
    for side, label in [('home', 'home'), ('away', 'away')]:
        ops, status = get_lineup_ops(row['gamePk'], side)
        df.at[idx, f'{label}_lineup_ops'] = ops
        print(f"  {row['gamePk']} {label}: {row[f'{label}_team'][:20]:20s} → OPS {ops} ({status})")
    time.sleep(0.3)

print(f"\n라인업 OPS 결과:")
print(df[['gamePk','home_team','home_lineup_ops','away_team','away_lineup_ops']].to_string())
print(f"\nOPS 수집 실패: home {df['home_lineup_ops'].isna().sum()}개, away {df['away_lineup_ops'].isna().sum()}개")

In [ ]:
# Step 2b: 라인업 미확정 시 → 해당 팀의 직전 경기 라인업 OPS로 대체
import time

def get_last_lineup_ops(team_name, before_date):
    """직전 5경기 스케줄에서 가장 최근 라인업 OPS 가져오기"""
    url = (f"https://statsapi.mlb.com/api/v1/schedule"
           f"?sportId=1&teamId=&startDate=2026-03-20&endDate={before_date}"
           f"&gameType=R&hydrate=linescore")

    # 팀ID 찾기
    team_url = "https://statsapi.mlb.com/api/v1/teams?sportId=1"
    teams = requests.get(team_url, timeout=10).json()
    team_id = None
    for t in teams['teams']:
        if t['name'] == team_name:
            team_id = t['id']; break
    if not team_id:
        return None

    url = (f"https://statsapi.mlb.com/api/v1/schedule"
           f"?sportId=1&teamId={team_id}&startDate=2026-03-20&endDate={before_date}"
           f"&gameType=R")
    data = requests.get(url, timeout=10).json()

    # Final 경기만, 최신순
    final_games = []
    for d in data.get('dates', []):
        for g in d.get('games', []):
            if g['status']['abstractGameState'] == 'Final':
                final_games.append(g['gamePk'])

    if not final_games:
        return None

    # 가장 최근 경기 boxscore
    last_pk = final_games[-1]
    box_url = f"https://statsapi.mlb.com/api/v1/game/{last_pk}/boxscore"
    try:
        box = requests.get(box_url, timeout=10).json()
        for side in ['home', 'away']:
            team = box['teams'][side]
            if team.get('team', {}).get('name') == team_name:
                batting = team.get('battingOrder', [])
                players = team.get('players', {})
                ops_list = []
                for pid in batting:
                    p = players.get(f'ID{pid}', {})
                    s = p.get('seasonStats', {}).get('batting', {})
                    try: ops_list.append(float(s.get('ops', 0)))
                    except: ops_list.append(0.0)
                if ops_list:
                    return round(sum(ops_list) / len(ops_list), 3)
    except:
        pass
    return None

# 미확정인 팀만 fallback 적용
for idx, row in df.iterrows():
    for side in ['home', 'away']:
        if pd.isna(row[f'{side}_lineup_ops']) or row[f'{side}_lineup_ops'] is None:
            team = row[f'{side}_team']
            ops = get_last_lineup_ops(team, TODAY)
            df.at[idx, f'{side}_lineup_ops'] = ops
            src = 'fallback(직전경기)' if ops else '❌실패'
            print(f"  {row['gamePk']} {side}: {team[:22]:22s} → OPS {ops} [{src}]")
            time.sleep(0.3)

print(f"\n최종 라인업 OPS:")
print(df[['gamePk','home_team','home_lineup_ops','away_team','away_lineup_ops']].to_string())
print(f"\n여전히 NaN: {df[['home_lineup_ops','away_lineup_ops']].isna().sum().sum()}개")

In [ ]:
# Step 4: 파크팩터 로드 (2025 기준 — 2026 시즌 초반에는 전년도 사용)
DRIVE = '<DRIVE_ROOT>/MLB데이터'
pf = pd.read_csv(f'{DRIVE}/park_factors_2025.csv')

# 팀명 매핑 (Savant 약칭 → MLB 풀네임)
PARK_TO_FULL = {
    'Angels':'Los Angeles Angels', 'Astros':'Houston Astros',
    'Athletics':'Athletics', 'Blue Jays':'Toronto Blue Jays',
    'Braves':'Atlanta Braves', 'Brewers':'Milwaukee Brewers',
    'Cardinals':'St. Louis Cardinals', 'Cubs':'Chicago Cubs',
    'D-backs':'Arizona Diamondbacks', 'Dodgers':'Los Angeles Dodgers',
    'Giants':'San Francisco Giants', 'Guardians':'Cleveland Guardians',
    'Mariners':'Seattle Mariners', 'Marlins':'Miami Marlins',
    'Mets':'New York Mets', 'Nationals':'Washington Nationals',
    'Orioles':'Baltimore Orioles', 'Padres':'San Diego Padres',
    'Phillies':'Philadelphia Phillies', 'Pirates':'Pittsburgh Pirates',
    'Rangers':'Texas Rangers', 'Rays':'Tampa Bay Rays',
    'Red Sox':'Boston Red Sox', 'Reds':'Cincinnati Reds',
    'Rockies':'Colorado Rockies', 'Royals':'Kansas City Royals',
    'Tigers':'Detroit Tigers', 'Twins':'Minnesota Twins',
    'White Sox':'Chicago White Sox', 'Yankees':'New York Yankees',
}
pf['team_full'] = pf['Team'].map(PARK_TO_FULL)

# 홈팀 기준 조인
pf_lookup = pf.set_index('team_full')[['ParkFactor','R','HR','H']].to_dict('index')

for idx, row in df.iterrows():
    home = row['home_team']
    # Athletics 특수처리
    lookup_name = home if home != 'Oakland Athletics' else 'Athletics'
    p = pf_lookup.get(lookup_name, {})
    df.at[idx, 'park_factor'] = p.get('ParkFactor', 100)
    df.at[idx, 'park_R'] = p.get('R', 100)
    df.at[idx, 'park_HR'] = p.get('HR', 100)
    df.at[idx, 'park_H'] = p.get('H', 100)
    print(f"  {home[:22]:22s} → PF={p.get('ParkFactor','?')} R={p.get('R','?')} HR={p.get('HR','?')} H={p.get('H','?')}")

print(f"\n파크팩터 결과:")
print(df[['gamePk','home_team','park_factor','park_R','park_HR','park_H']].to_string())

In [ ]:
# Step 5: 롤링 피처 (wr5/10, rs5/10, ra5/10) + 불펜 소모도
import numpy as np

# 히스토리 로드 (2021-2025 마스터 + 2026 경기결과)
master = pd.read_csv(f'{DRIVE}/master_5seasons.csv')
hist = master[['gamePk','date','home_team','away_team','홈팀득점','원정팀득점','홈팀결과']].copy()
hist['date'] = pd.to_datetime(hist['date'])

# 2026 결과 추가
odds26 = pd.read_csv(f'{DRIVE}/MARKET_PRICE,LINEUP/mlb_market_price_2026.csv')
odds26 = odds26[odds26['away_score'].notna()].copy()
if len(odds26) > 0:
    g26 = pd.DataFrame({
        'gamePk': odds26['game_id'],
        'date': pd.to_datetime(odds26['date']),
        'home_team': odds26['home_team'].str.strip(),
        'away_team': odds26['away_team'].str.strip(),
        '홈팀득점': odds26['home_score'],
        '원정팀득점': odds26['away_score'],
        '홈팀결과': np.where(odds26['home_score'] > odds26['away_score'], '승', '패')
    })
    hist = pd.concat([hist, g26], ignore_index=True)
hist = hist.sort_values('date').reset_index(drop=True)
print(f"히스토리: {len(hist):,}경기 (~{hist['date'].max().date()})")

before = pd.to_datetime(TODAY)

def team_rolling(team, before_date):
    h = hist[(hist['home_team']==team) & (hist['date']<before_date)].copy()
    h['win'] = (h['홈팀결과']=='승').astype(int)
    h['rs'] = h['홈팀득점']; h['ra'] = h['원정팀득점']

    a = hist[(hist['away_team']==team) & (hist['date']<before_date)].copy()
    a['win'] = (a['홈팀결과']=='패').astype(int)
    a['rs'] = a['원정팀득점']; a['ra'] = a['홈팀득점']

    games = pd.concat([h[['date','win','rs','ra']], a[['date','win','rs','ra']]]).sort_values('date')
    if len(games) < 5:
        return {k: np.nan for k in ['wr5','wr10','rs5','rs10','ra5','ra10']}

    l5, l10 = games.tail(5), games.tail(10)
    return {
        'wr5': l5['win'].mean(), 'wr10': l10['win'].mean(),
        'rs5': l5['rs'].mean(), 'rs10': l10['rs'].mean(),
        'ra5': l5['ra'].mean(), 'ra10': l10['ra'].mean(),
    }

# 롤링 피처 계산
print("\n--- 롤링 피처 ---")
for idx, row in df.iterrows():
    for side in ['home', 'away']:
        team = row[f'{side}_team']
        roll = team_rolling(team, before)
        for k, v in roll.items():
            df.at[idx, f'{side}_{k}'] = v
        print(f"  {team[:22]:22s} wr5={roll['wr5']:.2f} rs5={roll['rs5']:.1f} ra5={roll['ra5']:.1f}" if roll['wr5'] is not np.nan else f"  {team} → NaN")

# 불펜 소모도 — 2026 일일 파일에서
print("\n--- 불펜 소모도 ---")
bp_date = TODAY.replace('-', '')
bp_file = f'{DRIVE}/2026/mlb_{bp_date}_불펜소모도.csv'

if os.path.exists(bp_file):
    bp = pd.read_csv(bp_file)
    print(f"  불펜파일 로드: {bp_file} ({len(bp)}행)")
    for idx, row in df.iterrows():
        for side in ['home', 'away']:
            team = row[f'{side}_team']
            tbp = bp[bp['팀명'].str.contains(team.split()[-1], case=False, na=False)]
            fatigue_3d = tbp['3일등판수'].sum() if len(tbp) > 0 else 0
            fatigue_1d = len(tbp[tbp['3일등판수'] >= 1]) if len(tbp) > 0 else 0
            df.at[idx, f'{side}_bp_fatigue_1d'] = fatigue_1d
            df.at[idx, f'{side}_bp_fatigue_3d'] = fatigue_3d
            print(f"  {team[:22]:22s} 1d={fatigue_1d} 3d={fatigue_3d}")
else:
    print(f"  ⚠️ {bp_file} 없음 → 0으로 처리")
    for col in ['home_bp_fatigue_1d','home_bp_fatigue_3d','away_bp_fatigue_1d','away_bp_fatigue_3d']:
        df[col] = 0.0

print(f"\n현재 df 컬럼: {len(df.columns)}개")
print(df[['home_team','home_wr5','home_rs5','home_bp_fatigue_3d','away_team','away_wr5','away_rs5','away_bp_fatigue_3d']].to_string())

In [ ]:
# Step 6: diff/sum 피처 생성 → v2 모델 적용 → 분석 후보
import pickle

# 모델 로드
with open(f'{DRIVE}/종결v2_승패_LR.pkl', 'rb') as f: wp_lr = pickle.load(f)
with open(f'{DRIVE}/종결v2_승패_LGB.pkl', 'rb') as f: wp_lgb = pickle.load(f)
with open(f'{DRIVE}/종결v2_총득점_LR.pkl', 'rb') as f: uo_lr = pickle.load(f)
with open(f'{DRIVE}/종결v2_총득점_LGB.pkl', 'rb') as f: uo_lgb = pickle.load(f)

# --- 승패 diff 피처 (12개) ---
df['era_diff'] = df['home_sp_era'] - df['away_sp_era']
df['k9_diff'] = df['home_sp_k9'] - df['away_sp_k9']
df['bb9_diff'] = df['home_sp_bb9'] - df['away_sp_bb9']
df['ops_diff'] = df['home_lineup_ops'] - df['away_lineup_ops']
df['wr5_diff'] = df['home_wr5'] - df['away_wr5']
df['wr10_diff'] = df['home_wr10'] - df['away_wr10']
df['rs10_diff'] = df['home_rs10'] - df['away_rs10']
df['bp_fatigue_1d_diff'] = df['home_bp_fatigue_1d'] - df['away_bp_fatigue_1d']

# --- 총득점 sum 피처 (22개) ---
df['era_sum'] = df['home_sp_era'] + df['away_sp_era']
df['whip_sum'] = df['home_sp_whip'] + df['away_sp_whip']
df['k9_sum'] = df['home_sp_k9'] + df['away_sp_k9']
df['bb9_sum'] = df['home_sp_bb9'] + df['away_sp_bb9']
df['hr9_sum'] = df['home_sp_hr9'] + df['away_sp_hr9']
df['ops_sum'] = df['home_lineup_ops'] + df['away_lineup_ops']
df['rs5_sum'] = df['home_rs5'] + df['away_rs5']
df['rs10_sum'] = df['home_rs10'] + df['away_rs10']
df['ra5_sum'] = df['home_ra5'] + df['away_ra5']
df['ra10_sum'] = df['home_ra10'] + df['away_ra10']
df['bp_fatigue_sum'] = df['home_bp_fatigue_3d'] + df['away_bp_fatigue_3d']
df['era_max'] = df[['home_sp_era','away_sp_era']].max(axis=1)
df['whip_max'] = df[['home_sp_whip','away_sp_whip']].max(axis=1)
df['era_vs_run_total'] = df['era_sum'] - df['run_total_baseline']
df['ops_vs_run_total'] = df['ops_sum'] * 10 - df['run_total_baseline']

# --- 모델 적용 ---
WP_F = list(wp_lr.feature_names_in_)
UO_F = list(uo_lr.feature_names_in_)

wp_prob = 0.8 * wp_lr.predict_proba(df[WP_F])[:,1] + 0.2 * wp_lgb.predict_proba(df[WP_F])[:,1]
uo_prob = 0.5 * uo_lr.predict_proba(df[UO_F])[:,1] + 0.5 * uo_lgb.predict_proba(df[UO_F])[:,1]

df['win_prob'] = wp_prob
df['run_total_prob'] = uo_prob
df['win_conf'] = df['win_prob'].apply(lambda p: max(p, 1-p))
df['run_total_conf'] = df['run_total_prob'].apply(lambda p: max(p, 1-p))

# --- 분석 후보 ---
print(f"{'='*65}")
print(f"  🎯 MLB v2 예측 — {TODAY} ({len(df)}경기)")
print(f"{'='*65}")

print(f"\n📊 승패 (임계 0.64):")
for _, g in df.iterrows():
    side = '홈' if g['win_prob'] >= 0.5 else '원정'
    team = g['home_team'] if side == '홈' else g['away_team']
    win_loss = g['home_WIN_LOSS'] if side == '홈' else g['away_WIN_LOSS']
    flag = '기준 충족' if g['win_conf'] >= 0.64 else '  ---'
    print(f"  {flag} {side} {team:24s} ({win_loss:+.0f}) | 확률 {g['win_prob']:.1%} | 확신 {g['win_conf']:.1%}")

print(f"\n📊 총득점 (임계 0.60):")
for _, g in df.iterrows():
    side = 'UPPER' if g['run_total_prob'] >= 0.5 else 'LOWER'
    flag = '기준 충족' if g['run_total_conf'] >= 0.60 else '  ---'
    print(f"  {flag} {side:5s} {g['home_team']:24s} vs {g['away_team']:20s} | RUN_TOTAL {g['run_total_baseline']:.1f} | 확률 {g['run_total_prob']:.1%} | 확신 {g['run_total_conf']:.1%}")

print(f"\n📋 전체 피처 확인:")
print(f"  승패 피처: {WP_F}")
print(f"  총득점 피처: {UO_F}")
print(f"  df 컬럼 총: {len(df.columns)}개")

In [ ]:
# 타입 수정 후 재실행
df['home_lineup_ops'] = pd.to_numeric(df['home_lineup_ops'], errors='coerce')
df['away_lineup_ops'] = pd.to_numeric(df['away_lineup_ops'], errors='coerce')

# ops_diff 재계산
df['ops_diff'] = df['home_lineup_ops'] - df['away_lineup_ops']
df['ops_sum'] = df['home_lineup_ops'] + df['away_lineup_ops']
df['ops_vs_run_total'] = df['ops_sum'] * 10 - df['run_total_baseline']

# 전체 피처 float 강제변환
for col in WP_F + UO_F:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("타입 확인:")
print(df[WP_F].dtypes)

# 다시 모델 적용
wp_prob = 0.8 * wp_lr.predict_proba(df[WP_F])[:,1] + 0.2 * wp_lgb.predict_proba(df[WP_F])[:,1]
uo_prob = 0.5 * uo_lr.predict_proba(df[UO_F])[:,1] + 0.5 * uo_lgb.predict_proba(df[UO_F])[:,1]

df['win_prob'] = wp_prob
df['run_total_prob'] = uo_prob
df['win_conf'] = df['win_prob'].apply(lambda p: max(p, 1-p))
df['run_total_conf'] = df['run_total_prob'].apply(lambda p: max(p, 1-p))

# 분석 후보
print(f"\n{'='*65}")
print(f"  🎯 MLB v2 예측 — {TODAY} ({len(df)}경기)")
print(f"{'='*65}")

print(f"\n📊 승패 (임계 0.64):")
for _, g in df.iterrows():
    side = '홈' if g['win_prob'] >= 0.5 else '원정'
    team = g['home_team'] if side == '홈' else g['away_team']
    win_loss = g['home_WIN_LOSS'] if side == '홈' else g['away_WIN_LOSS']
    flag = '기준 충족' if g['win_conf'] >= 0.64 else '  ---'
    print(f"  {flag} {side} {team:24s} ({win_loss:+.0f}) | 확률 {g['win_prob']:.1%} | 확신 {g['win_conf']:.1%}")

print(f"\n📊 총득점 (임계 0.60):")
for _, g in df.iterrows():
    side = 'UPPER' if g['run_total_prob'] >= 0.5 else 'LOWER'
    flag = '기준 충족' if g['run_total_conf'] >= 0.60 else '  ---'
    print(f"  {flag} {side:5s} {g['home_team']:24s} vs {g['away_team']:20s} | RUN_TOTAL {g['run_total_baseline']:.1f} | 확률 {g['run_total_prob']:.1%} | 확신 {g['run_total_conf']:.1%}")